In [4]:
import os
import re
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm

from pyproj import CRS
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from minisom import MiniSom

warnings.filterwarnings("ignore")

# =========================================================
# НАСТРОЙКИ
# =========================================================
CELL_SIZE = 500
RANDOM_STATE = 42

# методы сохраняем
SOM_X = 12
SOM_Y = 12
SOM_ITERS = 4000
N_CLUSTERS = 6
USE_SUPERVISED = False

# общий вклад блоков модели
W_GEO = 0.86
W_CLUSTER = 0.08
W_ML = 0.06 if USE_SUPERVISED else 0.0
LOCAL_BONUS_WEIGHT = 0.06
COINCIDENCE_WEIGHT = 0.08

# proximity: магматический фактор делаем чуть более резким,
# чтобы dayki_buf не расползался слишком широко.
Q_FACIES = 0.78
Q_PALEO = 0.76
Q_STRUCT = 0.72
Q_MAGM = 0.42
Q_TECT1 = 0.74
Q_TECT2 = 0.74

# визуализация
N_DISPLAY_CLASSES = 20
TOP_GOLD_Q = 0.08         # лучшие 8% по prognoz кандидаты на золото
TOP_LOCAL_Q = 0.92        # высокий локальный/комбинированный сигнал

# =========================================================
# ПУТИ
# =========================================================
def find_existing_base_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path("/mnt/data/prog_zip"),
        Path("/mnt/data"),
        Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз"),
    ]
    for base in candidates:
        shp_dir = base / "shp_dbf"
        if shp_dir.exists() and (shp_dir / "svita_new.shp").exists():
            return base
    raise FileNotFoundError("Не найден каталог с shp_dbf. Укажи BASE_DIR вручную.")

BASE_DIR = find_existing_base_dir()
SHP_DIR = BASE_DIR / "shp_dbf"
OUT_DIR = BASE_DIR / "same_methods_fixed_v3_result"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAFE_ALIAS_DIR = OUT_DIR / "_safe_shp_aliases"
SAFE_ALIAS_DIR.mkdir(parents=True, exist_ok=True)

OUT_GPKG = OUT_DIR / "gold_forecast_same_methods_fixed_v3.gpkg"
OUT_PNG = OUT_DIR / "gold_forecast_same_methods_fixed_v3.png"
OUT_PROX = OUT_DIR / "prox_magm_same_methods_fixed_v3.png"
OUT_COMPARE = OUT_DIR / "compare_same_methods_fixed_v3.png"
OUT_CSV = OUT_DIR / "grid_attributes_same_methods_fixed_v3.csv"
OUT_JSON = OUT_DIR / "metrics_same_methods_fixed_v3.json"

# =========================================================
# ВСПОМОГАТЕЛЬНЫЕ
# =========================================================
def normalize_01(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    mn = np.nanmin(arr[finite])
    mx = np.nanmax(arr[finite])
    if np.isclose(mx, mn):
        return np.full_like(arr, 0.5, dtype=float)
    out = np.full_like(arr, np.nan, dtype=float)
    out[finite] = (arr[finite] - mn) / (mx - mn)
    return out


def robust_normalize_01(values, q_low=0.03, q_high=0.97):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    lo = np.nanquantile(arr[finite], q_low)
    hi = np.nanquantile(arr[finite], q_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or np.isclose(lo, hi):
        return normalize_01(arr)
    out = (arr - lo) / (hi - lo)
    return np.clip(out, 0, 1)


def read_sidecar_proj4(shp_path: Path):
    sidecar = shp_path.with_name(shp_path.stem + "_shp.pj4")
    if sidecar.exists():
        txt = sidecar.read_text(encoding="utf-8", errors="ignore")
        m = re.search(r"pj4=(.+)", txt)
        if m:
            return m.group(1).strip()
    return None


def prepare_ascii_aliases(shp_dir: Path, alias_dir: Path):
    aliases = {}
    stems = {}
    for name_b in os.listdir(os.fsencode(shp_dir)):
        if not name_b.endswith((b".shp", b".shx", b".dbf", b".prj", b".pj4")):
            continue
        if name_b.endswith(b"_shp.pj4"):
            continue
        base_b, ext_b = os.path.splitext(name_b)
        stems.setdefault(base_b, set()).add(ext_b)

    alias_idx = 0
    for base_b, exts in sorted(stems.items()):
        try:
            base_s = os.fsdecode(base_b)
            safe = all(ord(ch) < 128 and (ch.isalnum() or ch in "_.- ") for ch in base_s)
        except Exception:
            base_s = None
            safe = False

        if safe:
            aliases[base_s] = shp_dir / f"{base_s}.shp"
            continue

        alias_name = f"evidence_{alias_idx:02d}"
        alias_idx += 1
        for ext_b in exts:
            src = os.path.join(os.fsencode(shp_dir), base_b + ext_b)
            dst = alias_dir / f"{alias_name}{os.fsdecode(ext_b)}"
            shutil.copyfile(src, dst)
        pj4_src = os.path.join(os.fsencode(shp_dir), base_b + b"_shp.pj4")
        if os.path.exists(pj4_src):
            shutil.copyfile(pj4_src, alias_dir / f"{alias_name}_shp.pj4")
        aliases[alias_name] = alias_dir / f"{alias_name}.shp"
    return aliases


def load_layer(shp_path: Path):
    gdf = gpd.read_file(shp_path)
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    if gdf.crs is None:
        proj4 = read_sidecar_proj4(shp_path)
        if proj4:
            gdf = gdf.set_crs(CRS.from_proj4(proj4), allow_override=True)
    return gdf


def to_crs_safe(gdf, target_crs):
    if gdf.crs is None and target_crs is not None:
        return gdf.set_crs(target_crs, allow_override=True)
    if target_crs is None or gdf.crs == target_crs:
        return gdf
    return gdf.to_crs(target_crs)


def build_grid(mask, cell_size):
    mask_union = unary_union(mask.geometry)
    prepared_mask = prep(mask_union)
    minx, miny, maxx, maxy = mask.total_bounds
    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)
    rows = []
    cell_id = 0
    for r, y in enumerate(ys):
        for c, x in enumerate(xs):
            geom = box(x, y, x + cell_size, y + cell_size)
            if prepared_mask.intersects(geom):
                rows.append((cell_id, r, c, geom))
                cell_id += 1
    grid = gpd.GeoDataFrame(rows, columns=["cell_id", "row", "col", "geometry"], geometry="geometry", crs=mask.crs)
    return grid, mask_union, (len(ys), len(xs))


def add_distance_feature(grid, source, name):
    source_union = unary_union(source.geometry)
    distances = np.empty(len(grid), dtype=float)
    for i, geom in enumerate(grid.geometry.values):
        distances[i] = 0.0 if geom.intersects(source_union) else geom.distance(source_union)
    grid[name] = distances
    return grid


def distance_to_proximity(distance, transform="sqrt", q=0.75):
    d = np.asarray(distance, dtype=float)
    d = np.clip(d, 0, None)
    if transform == "sqrt":
        dt = np.sqrt(d)
    elif transform == "cbrt":
        dt = np.cbrt(d)
    elif transform == "log1p":
        dt = np.log1p(d)
    else:
        dt = d
    scale = float(np.nanquantile(dt, q))
    if not np.isfinite(scale) or scale <= 0:
        scale = max(float(np.nanmean(dt)), 1.0)
    prox = np.exp(-dt / scale)
    return np.clip(prox, 0, 1)


def smooth_on_regular_grid(grid, value_col, shape, passes=1):
    try:
        from scipy.signal import convolve2d
    except Exception:
        return grid[value_col].to_numpy()
    arr = np.full(shape, np.nan, dtype=float)
    arr[grid["row"].to_numpy(), grid["col"].to_numpy()] = grid[value_col].to_numpy()
    kernel = np.array([[1.0, 1.2, 1.0], [1.2, 3.0, 1.2], [1.0, 1.2, 1.0]], dtype=float)
    smoothed = arr.copy()
    for _ in range(max(1, passes)):
        valid = np.isfinite(smoothed).astype(float)
        filled = np.nan_to_num(smoothed, nan=0.0)
        num = convolve2d(filled, kernel, mode="same", boundary="fill", fillvalue=0)
        den = convolve2d(valid, kernel, mode="same", boundary="fill", fillvalue=0)
        with np.errstate(divide="ignore", invalid="ignore"):
            smoothed = np.where(den > 0, num / den, np.nan)
    return smoothed[grid["row"].to_numpy(), grid["col"].to_numpy()]


def collect_points(mask_crs, aliases):
    point_layers = []
    for name, shp_path in aliases.items():
        if name in {"svita_new", "fasii", "glub_raz_nw", "glub_r_nw", "gr_dol_vp_poly", "kory", "dayki_buf"}:
            continue
        gdf = load_layer(shp_path)
        gdf = to_crs_safe(gdf, mask_crs)
        geom_types = {str(x) for x in gdf.geom_type.unique()}
        if "Point" in geom_types or "MultiPoint" in geom_types:
            point_layers.append(gdf)
    if not point_layers:
        return None
    pts = pd.concat(point_layers, ignore_index=True)
    return gpd.GeoDataFrame(pts, geometry="geometry", crs=mask_crs)


def set_mask_extent(ax, mask):
    minx, miny, maxx, maxy = mask.total_bounds
    padx = (maxx - minx) * 0.02
    pady = (maxy - miny) * 0.02
    ax.set_xlim(minx - padx, maxx + padx)
    ax.set_ylim(miny - pady, maxy + pady)


def local_max_mask(grid, value_col, shape):
    try:
        from scipy.ndimage import maximum_filter
    except Exception:
        vals = grid[value_col].to_numpy()
        thr = np.nanquantile(vals, 0.98)
        return vals >= thr
    arr = np.full(shape, np.nan, dtype=float)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    vals = grid[value_col].to_numpy()
    arr[rows, cols] = vals
    filled = np.nan_to_num(arr, nan=-9999.0)
    locmax = maximum_filter(filled, size=3, mode="nearest")
    mask = np.isfinite(arr) & (filled >= locmax)
    return mask[rows, cols]


def make_display_classes(grid):
    # Для отображения используем prognoz напрямую:
    # низкие значения prognoz = хорошие территории -> синие.
    prog_disp = robust_normalize_01(grid["prognoz"].to_numpy(), 0.02, 0.98)
    grid["display_score"] = prog_disp
    bins = np.linspace(0, 1, N_DISPLAY_CLASSES + 1)
    grid["display_class"] = np.digitize(prog_disp, bins[1:-1], right=False)
    return grid


def mark_gold_zones(grid, shape, mask_union):
    # Лучшие зоны должны быть локальными и подтверждаться комбинацией факторов,
    # а не только большим одиночным tectonic score.
    q_best = float(grid["prognoz"].quantile(TOP_GOLD_Q))
    q_local = float(grid["local_bonus"].quantile(TOP_LOCAL_Q))
    q_coinc = float(grid["coincidence_score"].quantile(TOP_LOCAL_Q))
    q_magm = float(grid["prox_magm"].quantile(0.88))
    q_tmagm = float(grid["tect_magm_intersection"].quantile(0.78))

    mask_boundary = mask_union.boundary
    grid["dist_to_boundary"] = np.array([geom.distance(mask_boundary) for geom in grid.geometry])

    local_peak = local_max_mask(grid, "prospectivity", shape)

    core_gold = (
        (grid["prognoz"] <= q_best) &
        local_peak &
        ((grid["local_bonus"] >= q_local) | (grid["coincidence_score"] >= q_coinc))
    )

    # Дополнительное правило, чтобы не терять узкие золотые полосы на краях маски.
    edge_gold = (
        (grid["dist_to_boundary"] <= CELL_SIZE * 0.85) &
        (grid["prox_magm"] >= q_magm) &
        (grid["tect_magm_intersection"] >= q_tmagm) &
        (grid["prognoz"] <= float(grid["prognoz"].quantile(0.16)))
    )

    gold = (core_gold | edge_gold).astype(int)
    grid["gold_zone"] = gold
    return grid


def plot_prox(grid, mask, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    grid.plot(column="prox_magm", ax=ax, cmap="RdYlBu_r", linewidth=0, legend=True)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    set_mask_extent(ax, mask)
    ax.set_title("prox_magm")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()


def plot_final(grid, mask, points, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    # База: синий-белый-красный по prognoz (синий = лучше, красный = хуже)
    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr.N)
    grid.plot(column="display_class", ax=ax, cmap="bwr", norm=norm, linewidth=0, legend=True)
    # Лучшие локальные зоны накладываем золотым поверх базы.
    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=ax, color="#f2d200", linewidth=0)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    if points is not None and len(points) > 0:
        points.plot(ax=ax, color="yellow", markersize=8, edgecolor="black", linewidth=0.25)
    set_mask_extent(ax, mask)
    ax.set_title("Итоговый прогноз")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()


def plot_compare(grid, mask, points, out_png):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    grid.plot(column="prox_magm", ax=axes[0], cmap="RdYlBu_r", linewidth=0)
    mask.boundary.plot(ax=axes[0], color="black", linewidth=0.5)
    set_mask_extent(axes[0], mask)
    axes[0].set_title("prox_magm")
    axes[0].set_axis_off()

    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr.N)
    grid.plot(column="display_class", ax=axes[1], cmap="bwr", norm=norm, linewidth=0)
    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=axes[1], color="#f2d200", linewidth=0)
    mask.boundary.plot(ax=axes[1], color="black", linewidth=0.5)
    if points is not None and len(points) > 0:
        points.plot(ax=axes[1], color="yellow", markersize=8, edgecolor="black", linewidth=0.25)
    set_mask_extent(axes[1], mask)
    axes[1].set_title("Итоговый прогноз")
    axes[1].set_axis_off()

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()


# =========================================================
# ЗАГРУЗКА
# =========================================================
aliases = prepare_ascii_aliases(SHP_DIR, SAFE_ALIAS_DIR)
mask = load_layer(aliases["svita_new"])
facies = to_crs_safe(load_layer(aliases["fasii"]), mask.crs)
tect1 = to_crs_safe(load_layer(aliases["glub_raz_nw"]), mask.crs)
tect2 = to_crs_safe(load_layer(aliases["glub_r_nw"]), mask.crs)
paleo = to_crs_safe(load_layer(aliases["gr_dol_vp_poly"]), mask.crs)
struct = to_crs_safe(load_layer(aliases["kory"]), mask.crs)
magm = to_crs_safe(load_layer(aliases["dayki_buf"]), mask.crs)
points = collect_points(mask.crs, aliases)

# =========================================================
# СЕТКА
# =========================================================
grid, mask_union, grid_shape = build_grid(mask, CELL_SIZE)

# =========================================================
# ДИСТАНЦИИ
# =========================================================
grid = add_distance_feature(grid, facies, "dist_facies")
grid = add_distance_feature(grid, paleo, "dist_paleo")
grid = add_distance_feature(grid, struct, "dist_struct")
grid = add_distance_feature(grid, magm, "dist_magm")
grid = add_distance_feature(grid, tect1, "dist_tect1")
grid = add_distance_feature(grid, tect2, "dist_tect2")

# =========================================================
# PROXIMITY
# =========================================================
grid["prox_facies"] = distance_to_proximity(grid["dist_facies"], transform="cbrt", q=Q_FACIES)
grid["prox_paleo"] = distance_to_proximity(grid["dist_paleo"], transform="cbrt", q=Q_PALEO)
grid["prox_struct"] = distance_to_proximity(grid["dist_struct"], transform="sqrt", q=Q_STRUCT)
grid["prox_magm"] = distance_to_proximity(grid["dist_magm"], transform="sqrt", q=Q_MAGM)
grid["prox_tect1"] = distance_to_proximity(grid["dist_tect1"], transform="cbrt", q=Q_TECT1)
grid["prox_tect2"] = distance_to_proximity(grid["dist_tect2"], transform="cbrt", q=Q_TECT2)

# =========================================================
# INTERACTIONS
# =========================================================
grid["tect_combo"] = 0.5 * (grid["prox_tect1"] + grid["prox_tect2"])
grid["tect_intersection"] = grid["prox_tect1"] * grid["prox_tect2"]
grid["tect_magm_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_magm"])
grid["tect_struct_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_struct"])
grid["paleo_struct_intersection"] = np.sqrt(grid["prox_paleo"] * grid["prox_struct"])

# дополнительная осмысленная мера совместного проявления факторов
combo_core = (
    np.clip(grid["tect_combo"], 0, 1) *
    np.clip(0.55 * grid["prox_magm"] + 0.45 * grid["prox_struct"], 0, 1) *
    np.clip(0.60 * grid["prox_paleo"] + 0.40 * grid["prox_facies"], 0, 1)
)
grid["coincidence_score"] = robust_normalize_01(np.sqrt(np.clip(combo_core, 0, 1)), 0.02, 0.98)

# штрафуем избыточно tectonic-only области, которые в ответе часто дают ложные зоны
tect_support = 0.45 * grid["prox_magm"] + 0.35 * grid["prox_struct"] + 0.20 * grid["prox_paleo"]
grid["tect_only_penalty"] = robust_normalize_01(np.clip(grid["tect_combo"] - tect_support, 0, 1), 0.02, 0.98)

# =========================================================
# GEO SCORE
# =========================================================
grid["geo_score_raw"] = (
    0.15 * grid["prox_tect1"] +
    0.15 * grid["prox_tect2"] +
    0.14 * grid["prox_paleo"] +
    0.11 * grid["prox_struct"] +
    0.08 * grid["prox_facies"] +
    0.07 * grid["prox_magm"] +
    0.09 * grid["tect_intersection"] +
    0.07 * grid["tect_magm_intersection"] +
    0.05 * grid["tect_struct_intersection"] +
    0.03 * grid["paleo_struct_intersection"] +
    0.06 * grid["coincidence_score"] -
    0.05 * grid["tect_only_penalty"]
)
grid["geo_score"] = robust_normalize_01(grid["geo_score_raw"], 0.02, 0.98)
grid["geo_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "geo_score", grid_shape, passes=1), 0.02, 0.98)

# =========================================================
# LR оставляем, но по умолчанию выключена
# =========================================================
grid["target"] = 0
grid["ml_score"] = grid["geo_score_sm"]
use_supervised = False

feature_cols = [
    "prox_facies", "prox_paleo", "prox_struct", "prox_magm",
    "prox_tect1", "prox_tect2", "tect_combo", "tect_intersection",
    "tect_magm_intersection", "tect_struct_intersection",
    "paleo_struct_intersection", "coincidence_score", "tect_only_penalty",
    "geo_score_sm"
]

if USE_SUPERVISED and points is not None and len(points) > 0:
    try:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", predicate="within")
    except Exception:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", op="within")
    positive_cells = joined["cell_id"].dropna().astype(int).unique().tolist()
    grid.loc[grid["cell_id"].isin(positive_cells), "target"] = 1
    if grid["target"].sum() >= 10 and grid["target"].sum() < len(grid):
        X = grid[feature_cols].fillna(0).to_numpy()
        y = grid["target"].to_numpy()
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        lr = LogisticRegression(random_state=RANDOM_STATE, max_iter=4000, class_weight="balanced")
        lr.fit(X_scaled, y)
        grid["ml_score"] = robust_normalize_01(lr.predict_proba(X_scaled)[:, 1], 0.02, 0.98)
        use_supervised = True

# =========================================================
# SOM + KMEANS
# =========================================================
X = grid[feature_cols].fillna(0).to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

som = MiniSom(x=SOM_X, y=SOM_Y, input_len=X_scaled.shape[1], sigma=1.1, learning_rate=0.38, random_seed=RANDOM_STATE)
som.random_weights_init(X_scaled)
som.train_random(X_scaled, SOM_ITERS)

winners = np.array([som.winner(x) for x in X_scaled])
grid["som_x"] = winners[:, 0]
grid["som_y"] = winners[:, 1]
grid["som_node"] = grid["som_x"].astype(str) + "_" + grid["som_y"].astype(str)

som_weights = som.get_weights().reshape(SOM_X * SOM_Y, X_scaled.shape[1])
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=20)
neuron_cluster = kmeans.fit_predict(som_weights)

node_to_cluster = {}
idx = 0
for i in range(SOM_X):
    for j in range(SOM_Y):
        node_to_cluster[f"{i}_{j}"] = int(neuron_cluster[idx])
        idx += 1
grid["cluster"] = grid["som_node"].map(node_to_cluster).astype(int)

cluster_geo = grid.groupby("cluster")["geo_score_sm"].mean().reset_index(name="cluster_geo_mean")
cluster_ml = grid.groupby("cluster")["ml_score"].mean().reset_index(name="cluster_ml_mean")
cluster_coinc = grid.groupby("cluster")["coincidence_score"].mean().reset_index(name="cluster_coinc_mean")
cluster_stats = cluster_geo.merge(cluster_ml, on="cluster", how="outer").merge(cluster_coinc, on="cluster", how="outer")

if use_supervised:
    cluster_hits = grid.groupby("cluster").agg(cells=("cell_id", "count"), positives=("target", "sum")).reset_index()
    cluster_hits["hit_rate"] = cluster_hits["positives"] / cluster_hits["cells"]
    cluster_stats = cluster_stats.merge(cluster_hits, on="cluster", how="left")
    cluster_stats["hit_rate"] = cluster_stats["hit_rate"].fillna(0)
    cluster_stats["cluster_score"] = robust_normalize_01(
        0.50 * cluster_stats["cluster_geo_mean"] +
        0.20 * cluster_stats["cluster_ml_mean"] +
        0.15 * cluster_stats["cluster_coinc_mean"] +
        0.15 * robust_normalize_01(cluster_stats["hit_rate"], 0.0, 1.0),
        0.02, 0.98,
    )
else:
    cluster_stats["cluster_score"] = robust_normalize_01(
        0.72 * cluster_stats["cluster_geo_mean"] +
        0.12 * cluster_stats["cluster_ml_mean"] +
        0.16 * cluster_stats["cluster_coinc_mean"],
        0.02, 0.98,
    )

grid = grid.merge(cluster_stats[["cluster", "cluster_score"]], on="cluster", how="left")
grid["cluster_score"] = grid["cluster_score"].fillna(grid["geo_score_sm"])

# =========================================================
# ИТОГ
# =========================================================
if use_supervised:
    grid["prospectivity_raw"] = 0.60 * grid["geo_score_sm"] + 0.10 * grid["cluster_score"] + 0.20 * grid["ml_score"] + 0.10 * grid["coincidence_score"]
else:
    grid["prospectivity_raw"] = (
        W_GEO * grid["geo_score_sm"] +
        W_CLUSTER * grid["cluster_score"] +
        COINCIDENCE_WEIGHT * grid["coincidence_score"]
    )

grid["local_bonus"] = robust_normalize_01(
    0.38 * grid["tect_intersection"] +
    0.37 * grid["tect_magm_intersection"] +
    0.25 * grid["tect_struct_intersection"],
    0.02, 0.98,
)

grid["prospectivity_raw"] = grid["prospectivity_raw"] + LOCAL_BONUS_WEIGHT * grid["local_bonus"]
grid["prospectivity"] = robust_normalize_01(grid["prospectivity_raw"], 0.02, 0.98)

# в логике презентации: меньше = лучше
grid["prognoz"] = 1.0 - grid["prospectivity"]

# top зоны для проверки
top_thr = float(grid["prospectivity"].quantile(0.90))
grid["top10"] = (grid["prospectivity"] >= top_thr).astype(int)

try:
    grid["prospect_class"] = pd.qcut(grid["prospectivity"], q=5, labels=["very_low", "low", "medium", "high", "very_high"], duplicates="drop")
except Exception:
    grid["prospect_class"] = "medium"

grid = make_display_classes(grid)
grid = mark_gold_zones(grid, grid_shape, mask_union)

# =========================================================
# СОХРАНЕНИЕ
# =========================================================
if OUT_GPKG.exists():
    OUT_GPKG.unlink()

grid.to_file(OUT_GPKG, layer="forecast_grid", driver="GPKG")
if points is not None and len(points) > 0:
    points.to_file(OUT_GPKG, layer="evidence_points", driver="GPKG")

grid.drop(columns="geometry").to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

plot_prox(grid, mask, OUT_PROX)
plot_final(grid, mask, points, OUT_PNG)
plot_compare(grid, mask, points, OUT_COMPARE)

metrics = {
    "base_dir": str(BASE_DIR),
    "grid_cells": int(len(grid)),
    "cell_size": CELL_SIZE,
    "use_supervised_requested": bool(USE_SUPERVISED),
    "use_supervised_applied": bool(use_supervised),
    "top10_threshold": float(top_thr),
    "prospectivity_min": float(np.nanmin(grid["prospectivity"])),
    "prospectivity_p05": float(np.nanquantile(grid["prospectivity"], 0.05)),
    "prospectivity_p50": float(np.nanquantile(grid["prospectivity"], 0.50)),
    "prospectivity_p95": float(np.nanquantile(grid["prospectivity"], 0.95)),
    "prospectivity_max": float(np.nanmax(grid["prospectivity"])),
    "prognoz_min": float(np.nanmin(grid["prognoz"])),
    "prognoz_max": float(np.nanmax(grid["prognoz"])),
    "display_score_min": float(np.nanmin(grid["display_score"])),
    "display_score_max": float(np.nanmax(grid["display_score"])),
    "gold_zone_count": int(grid["gold_zone"].sum()),
    "point_count": int(len(points)) if points is not None else 0,
}
Path(OUT_JSON).write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

print("Готово.")
print(f"BASE_DIR: {BASE_DIR}")
print(f"PNG: {OUT_PNG}")
print(f"COMPARE: {OUT_COMPARE}")
print(f"GPKG: {OUT_GPKG}")
print(f"CSV: {OUT_CSV}")
print(f"JSON: {OUT_JSON}")
print("Диагностика:")
print(grid[["prospectivity", "prognoz", "display_score", "gold_zone"]].describe())


Готово.
BASE_DIR: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз
PNG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v3_result\gold_forecast_same_methods_fixed_v3.png
COMPARE: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v3_result\compare_same_methods_fixed_v3.png
GPKG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v3_result\gold_forecast_same_methods_fixed_v3.gpkg
CSV: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v3_result\grid_attributes_same_methods_fixed_v3.csv
JSON: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v3_result\metrics_same_methods_fixed_v3.json
Диагностика:
       prospectivity       prognoz  display_score     gold_zone
count   15684.000000  15684.000000   15684.000000  15684.000000
mean        0.414586      0.585414       0.585408      0.024101
std         0.257168      0.257168       0.257176      0.153368
min         0.000000      0.000000       0.000000      0.000000

In [10]:
import os
import re
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm

from pyproj import CRS
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from minisom import MiniSom

warnings.filterwarnings("ignore")

# =========================================================
# НАСТРОЙКИ
# =========================================================
CELL_SIZE = 500
RANDOM_STATE = 42

# методы сохраняем
SOM_X = 12
SOM_Y = 12
SOM_ITERS = 4000
N_CLUSTERS = 6
USE_SUPERVISED = True

# Random Forest как основной ML-блок, но не единственный источник результата
RF_N_ESTIMATORS = 400
RF_MAX_DEPTH = 12
RF_MIN_SAMPLES_LEAF = 4
RF_MIN_SAMPLES_SPLIT = 8

# итоговые веса
W_GEO = 0.50
W_RF = 0.25
W_COINCIDENCE = 0.15
W_CLUSTER = 0.05
W_LOCAL = 0.05

# proximity
Q_FACIES = 0.78
Q_PALEO = 0.76
Q_STRUCT = 0.72
Q_MAGM = 0.42
Q_TECT1 = 0.74
Q_TECT2 = 0.74

# визуализация и gold-зоны
N_DISPLAY_CLASSES = 20
TOP_GOLD_Q = 0.05
TOP_LOCAL_Q = 0.94
SHOW_POINTS = False

# =========================================================
# ПУТИ
# =========================================================
def find_existing_base_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path("/mnt/data/prog_zip"),
        Path("/mnt/data"),
        Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз"),
    ]
    for base in candidates:
        shp_dir = base / "shp_dbf"
        if shp_dir.exists() and (shp_dir / "svita_new.shp").exists():
            return base
    raise FileNotFoundError("Не найден каталог с shp_dbf. Укажи BASE_DIR вручную.")

BASE_DIR = find_existing_base_dir()
SHP_DIR = BASE_DIR / "shp_dbf"
OUT_DIR = BASE_DIR / "same_methods_fixed_v7_random_forest"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAFE_ALIAS_DIR = OUT_DIR / "_safe_shp_aliases"
SAFE_ALIAS_DIR.mkdir(parents=True, exist_ok=True)

OUT_GPKG = OUT_DIR / "gold_forecast_same_methods_fixed_v7_rf.gpkg"
OUT_PNG = OUT_DIR / "gold_forecast_same_methods_fixed_v7_rf.png"
OUT_PROX = OUT_DIR / "prox_magm_same_methods_fixed_v7_rf.png"
OUT_COMPARE = OUT_DIR / "compare_same_methods_fixed_v7_rf.png"
OUT_CSV = OUT_DIR / "grid_attributes_same_methods_fixed_v7_rf.csv"
OUT_JSON = OUT_DIR / "metrics_same_methods_fixed_v7_rf.json"

# =========================================================
# ВСПОМОГАТЕЛЬНЫЕ
# =========================================================
def normalize_01(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    mn = np.nanmin(arr[finite])
    mx = np.nanmax(arr[finite])
    if np.isclose(mx, mn):
        return np.full_like(arr, 0.5, dtype=float)
    out = np.full_like(arr, np.nan, dtype=float)
    out[finite] = (arr[finite] - mn) / (mx - mn)
    return out

def robust_normalize_01(values, q_low=0.03, q_high=0.97):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    lo = np.nanquantile(arr[finite], q_low)
    hi = np.nanquantile(arr[finite], q_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or np.isclose(lo, hi):
        return normalize_01(arr)
    out = (arr - lo) / (hi - lo)
    return np.clip(out, 0, 1)

def read_sidecar_proj4(shp_path: Path):
    sidecar = shp_path.with_name(shp_path.stem + "_shp.pj4")
    if sidecar.exists():
        txt = sidecar.read_text(encoding="utf-8", errors="ignore")
        m = re.search(r"pj4=(.+)", txt)
        if m:
            return m.group(1).strip()
    return None

def prepare_ascii_aliases(shp_dir: Path, alias_dir: Path):
    aliases = {}
    stems = {}
    for name_b in os.listdir(os.fsencode(shp_dir)):
        if not name_b.endswith((b".shp", b".shx", b".dbf", b".prj", b".pj4")):
            continue
        if name_b.endswith(b"_shp.pj4"):
            continue
        base_b, ext_b = os.path.splitext(name_b)
        stems.setdefault(base_b, set()).add(ext_b)

    alias_idx = 0
    for base_b, exts in sorted(stems.items()):
        try:
            base_s = os.fsdecode(base_b)
            safe = all(ord(ch) < 128 and (ch.isalnum() or ch in "_-. ") for ch in base_s)
        except Exception:
            base_s = None
            safe = False

        if safe:
            aliases[base_s] = shp_dir / f"{base_s}.shp"
            continue

        alias_name = f"evidence_{alias_idx:02d}"
        alias_idx += 1
        for ext_b in exts:
            src = os.path.join(os.fsencode(shp_dir), base_b + ext_b)
            dst = alias_dir / f"{alias_name}{os.fsdecode(ext_b)}"
            shutil.copyfile(src, dst)
        pj4_src = os.path.join(os.fsencode(shp_dir), base_b + b"_shp.pj4")
        if os.path.exists(pj4_src):
            shutil.copyfile(pj4_src, alias_dir / f"{alias_name}_shp.pj4")
        aliases[alias_name] = alias_dir / f"{alias_name}.shp"
    return aliases

def load_layer(shp_path: Path):
    gdf = gpd.read_file(shp_path)
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    if gdf.crs is None:
        proj4 = read_sidecar_proj4(shp_path)
        if proj4:
            gdf = gdf.set_crs(CRS.from_proj4(proj4), allow_override=True)
    return gdf

def to_crs_safe(gdf, target_crs):
    if gdf.crs is None and target_crs is not None:
        return gdf.set_crs(target_crs, allow_override=True)
    if target_crs is None or gdf.crs == target_crs:
        return gdf
    return gdf.to_crs(target_crs)

def build_grid(mask, cell_size):
    mask_union = unary_union(mask.geometry)
    prepared_mask = prep(mask_union)
    minx, miny, maxx, maxy = mask.total_bounds
    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)
    rows = []
    cell_id = 0
    for r, y in enumerate(ys):
        for c, x in enumerate(xs):
            geom = box(x, y, x + cell_size, y + cell_size)
            if prepared_mask.intersects(geom):
                rows.append((cell_id, r, c, geom))
                cell_id += 1
    grid = gpd.GeoDataFrame(rows, columns=["cell_id", "row", "col", "geometry"], geometry="geometry", crs=mask.crs)
    return grid, mask_union, (len(ys), len(xs))

def add_distance_feature(grid, source, name):
    source_union = unary_union(source.geometry)
    distances = np.empty(len(grid), dtype=float)
    for i, geom in enumerate(grid.geometry.values):
        distances[i] = 0.0 if geom.intersects(source_union) else geom.distance(source_union)
    grid[name] = distances
    return grid

def distance_to_proximity(distance, transform="sqrt", q=0.75):
    d = np.asarray(distance, dtype=float)
    d = np.clip(d, 0, None)
    if transform == "sqrt":
        dt = np.sqrt(d)
    elif transform == "cbrt":
        dt = np.cbrt(d)
    elif transform == "log1p":
        dt = np.log1p(d)
    else:
        dt = d
    scale = float(np.nanquantile(dt, q))
    if not np.isfinite(scale) or scale <= 0:
        scale = max(float(np.nanmean(dt)), 1.0)
    prox = np.exp(-dt / scale)
    return np.clip(prox, 0, 1)

def smooth_on_regular_grid(grid, value_col, shape, passes=1):
    try:
        from scipy.signal import convolve2d
    except Exception:
        return grid[value_col].to_numpy()
    arr = np.full(shape, np.nan, dtype=float)
    arr[grid["row"].to_numpy(), grid["col"].to_numpy()] = grid[value_col].to_numpy()
    kernel = np.array([[1.0, 1.2, 1.0], [1.2, 3.0, 1.2], [1.0, 1.2, 1.0]], dtype=float)
    smoothed = arr.copy()
    for _ in range(max(1, passes)):
        valid = np.isfinite(smoothed).astype(float)
        filled = np.nan_to_num(smoothed, nan=0.0)
        num = convolve2d(filled, kernel, mode="same", boundary="fill", fillvalue=0)
        den = convolve2d(valid, kernel, mode="same", boundary="fill", fillvalue=0)
        with np.errstate(divide="ignore", invalid="ignore"):
            smoothed = np.where(den > 0, num / den, np.nan)
    return smoothed[grid["row"].to_numpy(), grid["col"].to_numpy()]

def collect_points(mask_crs, aliases):
    point_layers = []
    for name, shp_path in aliases.items():
        if name in {"svita_new", "fasii", "glub_raz_nw", "glub_r_nw", "gr_dol_vp_poly", "kory", "dayki_buf"}:
            continue
        gdf = load_layer(shp_path)
        gdf = to_crs_safe(gdf, mask_crs)
        geom_types = {str(x) for x in gdf.geom_type.unique()}
        if "Point" in geom_types or "MultiPoint" in geom_types:
            point_layers.append(gdf)
    if not point_layers:
        return None
    pts = pd.concat(point_layers, ignore_index=True)
    return gpd.GeoDataFrame(pts, geometry="geometry", crs=mask_crs)

def set_mask_extent(ax, mask):
    minx, miny, maxx, maxy = mask.total_bounds
    padx = (maxx - minx) * 0.02
    pady = (maxy - miny) * 0.02
    ax.set_xlim(minx - padx, maxx + padx)
    ax.set_ylim(miny - pady, maxy + pady)

def local_max_mask(grid, value_col, shape):
    try:
        from scipy.ndimage import maximum_filter
    except Exception:
        vals = grid[value_col].to_numpy()
        thr = np.nanquantile(vals, 0.98)
        return vals >= thr
    arr = np.full(shape, np.nan, dtype=float)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    vals = grid[value_col].to_numpy()
    arr[rows, cols] = vals
    filled = np.nan_to_num(arr, nan=-9999.0)
    locmax = maximum_filter(filled, size=3, mode="nearest")
    mask = np.isfinite(arr) & (filled >= locmax)
    return mask[rows, cols]

def keep_large_components(grid, bool_col, shape, min_cells=4):
    try:
        from scipy import ndimage
    except Exception:
        return grid[bool_col].to_numpy().astype(bool)
    arr = np.zeros(shape, dtype=np.uint8)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    arr[rows, cols] = grid[bool_col].to_numpy().astype(np.uint8)
    structure = np.ones((3, 3), dtype=np.uint8)
    labeled, n = ndimage.label(arr, structure=structure)
    if n == 0:
        return grid[bool_col].to_numpy().astype(bool)
    sizes = np.bincount(labeled.ravel())
    keep = np.isin(labeled, np.where(sizes >= min_cells)[0])
    keep &= labeled > 0
    return keep[rows, cols]

def make_display_classes(grid):
    prog_disp = robust_normalize_01(grid["prognoz"].to_numpy(), 0.02, 0.98)
    grid["display_score"] = prog_disp
    bins = np.linspace(0, 1, N_DISPLAY_CLASSES + 1)
    grid["display_class"] = np.digitize(prog_disp, bins[1:-1], right=False)
    return grid

def mark_gold_zones(grid, shape, mask_union):
    q_best = float(grid["prognoz"].quantile(TOP_GOLD_Q))
    q_local = float(grid["local_bonus"].quantile(TOP_LOCAL_Q))
    q_coinc = float(grid["coincidence_score"].quantile(TOP_LOCAL_Q))
    q_magm = float(grid["prox_magm"].quantile(0.86))
    q_tmagm = float(grid["tect_magm_intersection"].quantile(0.75))
    q_rf = float(grid["rf_score_sm"].quantile(0.88))

    local_peak = local_max_mask(grid, "prospectivity", shape)

    seed_gold = (
        (grid["prognoz"] <= q_best) &
        local_peak &
        (grid["rf_score_sm"] >= q_rf) &
        (
            ((grid["local_bonus"] >= q_local) & (grid["tect_magm_intersection"] >= q_tmagm)) |
            ((grid["coincidence_score"] >= q_coinc) & (grid["prox_magm"] >= q_magm))
        )
    )

    edge_gold = (
        (grid["dist_to_boundary"] <= CELL_SIZE * 0.90) &
        (grid["prox_magm"] >= q_magm) &
        (grid["tect_magm_intersection"] >= q_tmagm) &
        (grid["rf_score_sm"] >= float(grid["rf_score_sm"].quantile(0.78)))
    )

    raw_gold = (seed_gold | edge_gold)
    grid["gold_seed"] = raw_gold.astype(int)
    large = keep_large_components(grid, "gold_seed", shape, min_cells=4)
    grid["gold_zone"] = large.astype(int)
    return grid

def plot_prox(grid, mask, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    grid.plot(column="prox_magm", ax=ax, cmap="RdYlBu_r", linewidth=0, legend=True)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    set_mask_extent(ax, mask)
    ax.set_title("prox_magm")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_final(grid, mask, points, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr.N)
    grid.plot(column="display_class", ax=ax, cmap="bwr", norm=norm, linewidth=0, legend=True)
    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=ax, color="#f2d200", linewidth=0)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=ax, color="yellow", markersize=8, edgecolor="black", linewidth=0.25)
    set_mask_extent(ax, mask)
    ax.set_title("Итоговый прогноз")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_compare(grid, mask, points, out_png):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    grid.plot(column="prox_magm", ax=axes[0], cmap="RdYlBu_r", linewidth=0)
    mask.boundary.plot(ax=axes[0], color="black", linewidth=0.5)
    set_mask_extent(axes[0], mask)
    axes[0].set_title("prox_magm")
    axes[0].set_axis_off()

    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr.N)
    grid.plot(column="display_class", ax=axes[1], cmap="bwr", norm=norm, linewidth=0)
    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=axes[1], color="#f2d200", linewidth=0)
    mask.boundary.plot(ax=axes[1], color="black", linewidth=0.5)
    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=axes[1], color="yellow", markersize=8, edgecolor="black", linewidth=0.25)
    set_mask_extent(axes[1], mask)
    axes[1].set_title("Итоговый прогноз")
    axes[1].set_axis_off()

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

# =========================================================
# ЗАГРУЗКА
# =========================================================
aliases = prepare_ascii_aliases(SHP_DIR, SAFE_ALIAS_DIR)
mask = load_layer(aliases["svita_new"])
facies = to_crs_safe(load_layer(aliases["fasii"]), mask.crs)
tect1 = to_crs_safe(load_layer(aliases["glub_raz_nw"]), mask.crs)
tect2 = to_crs_safe(load_layer(aliases["glub_r_nw"]), mask.crs)
paleo = to_crs_safe(load_layer(aliases["gr_dol_vp_poly"]), mask.crs)
struct = to_crs_safe(load_layer(aliases["kory"]), mask.crs)
magm = to_crs_safe(load_layer(aliases["dayki_buf"]), mask.crs)
points = collect_points(mask.crs, aliases)

# =========================================================
# СЕТКА
# =========================================================
grid, mask_union, grid_shape = build_grid(mask, CELL_SIZE)

# =========================================================
# ДИСТАНЦИИ
# =========================================================
grid = add_distance_feature(grid, facies, "dist_facies")
grid = add_distance_feature(grid, paleo, "dist_paleo")
grid = add_distance_feature(grid, struct, "dist_struct")
grid = add_distance_feature(grid, magm, "dist_magm")
grid = add_distance_feature(grid, tect1, "dist_tect1")
grid = add_distance_feature(grid, tect2, "dist_tect2")

# =========================================================
# PROXIMITY
# =========================================================
grid["prox_facies"] = distance_to_proximity(grid["dist_facies"], transform="cbrt", q=Q_FACIES)
grid["prox_paleo"] = distance_to_proximity(grid["dist_paleo"], transform="cbrt", q=Q_PALEO)
grid["prox_struct"] = distance_to_proximity(grid["dist_struct"], transform="sqrt", q=Q_STRUCT)
grid["prox_magm"] = distance_to_proximity(grid["dist_magm"], transform="sqrt", q=Q_MAGM)
grid["prox_tect1"] = distance_to_proximity(grid["dist_tect1"], transform="cbrt", q=Q_TECT1)
grid["prox_tect2"] = distance_to_proximity(grid["dist_tect2"], transform="cbrt", q=Q_TECT2)

# =========================================================
# INTERACTIONS
# =========================================================
grid["tect_combo"] = 0.5 * (grid["prox_tect1"] + grid["prox_tect2"])
grid["tect_intersection"] = grid["prox_tect1"] * grid["prox_tect2"]
grid["tect_magm_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_magm"])
grid["tect_struct_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_struct"])
grid["paleo_struct_intersection"] = np.sqrt(grid["prox_paleo"] * grid["prox_struct"])

combo_core = (
    np.clip(grid["tect_combo"], 0, 1) *
    np.clip(0.55 * grid["prox_magm"] + 0.45 * grid["prox_struct"], 0, 1) *
    np.clip(0.60 * grid["prox_paleo"] + 0.40 * grid["prox_facies"], 0, 1)
)
grid["coincidence_score"] = robust_normalize_01(np.sqrt(np.clip(combo_core, 0, 1)), 0.02, 0.98)

tect_support = 0.40 * grid["prox_magm"] + 0.35 * grid["prox_struct"] + 0.25 * grid["prox_paleo"]
grid["tect_only_penalty"] = robust_normalize_01(np.clip(grid["tect_combo"] - tect_support, 0, 1), 0.02, 0.98)

# =========================================================
# GEO SCORE - базовая геологическая поверхность
# =========================================================
grid["geo_score_raw"] = (
    0.14 * grid["prox_tect1"] +
    0.14 * grid["prox_tect2"] +
    0.14 * grid["prox_paleo"] +
    0.11 * grid["prox_struct"] +
    0.08 * grid["prox_facies"] +
    0.08 * grid["prox_magm"] +
    0.08 * grid["tect_intersection"] +
    0.08 * grid["tect_magm_intersection"] +
    0.05 * grid["tect_struct_intersection"] +
    0.04 * grid["paleo_struct_intersection"] +
    0.10 * grid["coincidence_score"] -
    0.08 * grid["tect_only_penalty"]
)
grid["geo_score"] = robust_normalize_01(grid["geo_score_raw"], 0.02, 0.98)
grid["geo_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "geo_score", grid_shape, passes=2), 0.02, 0.98)

# =========================================================
# RANDOM FOREST AS ML BLOCK
# =========================================================
grid["target"] = 0
grid["rf_score"] = grid["geo_score_sm"]
use_supervised = False
rf_test_proxy = None
feature_importance = {}

feature_cols = [
    "prox_facies", "prox_paleo", "prox_struct", "prox_magm",
    "prox_tect1", "prox_tect2", "tect_combo", "tect_intersection",
    "tect_magm_intersection", "tect_struct_intersection",
    "paleo_struct_intersection", "coincidence_score", "tect_only_penalty",
    "geo_score_sm"
]

if USE_SUPERVISED and points is not None and len(points) > 0:
    try:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", predicate="within")
    except Exception:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", op="within")

    positive_cells = joined["cell_id"].dropna().astype(int).unique().tolist()
    grid.loc[grid["cell_id"].isin(positive_cells), "target"] = 1

    pos = int(grid["target"].sum())
    neg = int((grid["target"] == 0).sum())

    if pos >= 20 and neg > pos:
        X = grid[feature_cols].fillna(0).to_numpy()
        y = grid["target"].to_numpy()

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
        )

        rf_eval = RandomForestClassifier(
            n_estimators=RF_N_ESTIMATORS,
            max_depth=RF_MAX_DEPTH,
            min_samples_leaf=RF_MIN_SAMPLES_LEAF,
            min_samples_split=RF_MIN_SAMPLES_SPLIT,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        rf_eval.fit(X_train, y_train)
        test_prob = rf_eval.predict_proba(X_test)[:, 1]
        y_test = np.asarray(y_test)
        pos_mean = float(np.mean(test_prob[y_test == 1])) if np.any(y_test == 1) else np.nan
        neg_mean = float(np.mean(test_prob[y_test == 0])) if np.any(y_test == 0) else np.nan
        rf_test_proxy = pos_mean - neg_mean

        rf = RandomForestClassifier(
            n_estimators=RF_N_ESTIMATORS,
            max_depth=RF_MAX_DEPTH,
            min_samples_leaf=RF_MIN_SAMPLES_LEAF,
            min_samples_split=RF_MIN_SAMPLES_SPLIT,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        rf.fit(X, y)
        grid["rf_score"] = robust_normalize_01(rf.predict_proba(X)[:, 1], 0.02, 0.98)
        feature_importance = dict(zip(feature_cols, rf.feature_importances_.tolist()))
        use_supervised = True

grid["rf_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "rf_score", grid_shape, passes=2), 0.02, 0.98)
grid["ml_score"] = grid["rf_score_sm"]

# =========================================================
# SOM + KMEANS
# =========================================================
X = grid[feature_cols].fillna(0).to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

som = MiniSom(
    x=SOM_X, y=SOM_Y, input_len=X_scaled.shape[1],
    sigma=1.1, learning_rate=0.38, random_seed=RANDOM_STATE
)
som.random_weights_init(X_scaled)
som.train_random(X_scaled, SOM_ITERS)

winners = np.array([som.winner(x) for x in X_scaled])
grid["som_x"] = winners[:, 0]
grid["som_y"] = winners[:, 1]
grid["som_node"] = grid["som_x"].astype(str) + "_" + grid["som_y"].astype(str)

som_weights = som.get_weights().reshape(SOM_X * SOM_Y, X_scaled.shape[1])
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=20)
neuron_cluster = kmeans.fit_predict(som_weights)

node_to_cluster = {}
idx = 0
for i in range(SOM_X):
    for j in range(SOM_Y):
        node_to_cluster[f"{i}_{j}"] = int(neuron_cluster[idx])
        idx += 1
grid["cluster"] = grid["som_node"].map(node_to_cluster).astype(int)

cluster_geo = grid.groupby("cluster")["geo_score_sm"].mean().reset_index(name="cluster_geo_mean")
cluster_rf = grid.groupby("cluster")["rf_score_sm"].mean().reset_index(name="cluster_rf_mean")
cluster_coinc = grid.groupby("cluster")["coincidence_score"].mean().reset_index(name="cluster_coinc_mean")
cluster_stats = cluster_geo.merge(cluster_rf, on="cluster", how="outer").merge(cluster_coinc, on="cluster", how="outer")

if use_supervised:
    cluster_hits = grid.groupby("cluster").agg(cells=("cell_id", "count"), positives=("target", "sum")).reset_index()
    cluster_hits["hit_rate"] = cluster_hits["positives"] / cluster_hits["cells"]
    cluster_stats = cluster_stats.merge(cluster_hits, on="cluster", how="left")
    cluster_stats["hit_rate"] = cluster_stats["hit_rate"].fillna(0)
    cluster_stats["cluster_score"] = robust_normalize_01(
        0.35 * cluster_stats["cluster_geo_mean"] +
        0.35 * cluster_stats["cluster_rf_mean"] +
        0.15 * cluster_stats["cluster_coinc_mean"] +
        0.15 * robust_normalize_01(cluster_stats["hit_rate"], 0.0, 1.0),
        0.02, 0.98,
    )
else:
    cluster_stats["cluster_score"] = robust_normalize_01(
        0.45 * cluster_stats["cluster_geo_mean"] +
        0.35 * cluster_stats["cluster_rf_mean"] +
        0.20 * cluster_stats["cluster_coinc_mean"],
        0.02, 0.98,
    )

grid = grid.merge(cluster_stats[["cluster", "cluster_score"]], on="cluster", how="left")
grid["cluster_score"] = grid["cluster_score"].fillna(grid["geo_score_sm"])

# =========================================================
# ИТОГ
# =========================================================
grid["local_bonus"] = robust_normalize_01(
    0.38 * grid["tect_intersection"] +
    0.37 * grid["tect_magm_intersection"] +
    0.25 * grid["tect_struct_intersection"],
    0.02, 0.98,
)

mask_boundary = mask_union.boundary
grid["dist_to_boundary"] = np.array([geom.distance(mask_boundary) for geom in grid.geometry])
edge_near = np.exp(-grid["dist_to_boundary"].to_numpy() / (CELL_SIZE * 2.2))
grid["edge_false_penalty"] = robust_normalize_01(
    edge_near * np.clip(grid["tect_only_penalty"], 0, 1),
    0.02, 0.98
)

grid["prospectivity_raw"] = (
    W_GEO * grid["geo_score_sm"] +
    W_RF * grid["rf_score_sm"] +
    W_COINCIDENCE * grid["coincidence_score"] +
    W_CLUSTER * grid["cluster_score"] +
    W_LOCAL * grid["local_bonus"]
)
grid["prospectivity_raw"] = (
    grid["prospectivity_raw"]
    - 0.05 * grid["tect_only_penalty"]
    - 0.04 * grid["edge_false_penalty"]
)
grid["prospectivity_raw_sm"] = smooth_on_regular_grid(grid, "prospectivity_raw", grid_shape, passes=2)
grid["prospectivity"] = robust_normalize_01(grid["prospectivity_raw_sm"], 0.02, 0.98)

# в логике презентации: меньше = лучше
grid["prognoz"] = 1.0 - grid["prospectivity"]

top_thr = float(grid["prospectivity"].quantile(0.90))
grid["top10"] = (grid["prospectivity"] >= top_thr).astype(int)

try:
    grid["prospect_class"] = pd.qcut(
        grid["prospectivity"],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop"
    )
except Exception:
    grid["prospect_class"] = "medium"

grid = make_display_classes(grid)
grid = mark_gold_zones(grid, grid_shape, mask_union)

# =========================================================
# СОХРАНЕНИЕ
# =========================================================
if OUT_GPKG.exists():
    OUT_GPKG.unlink()

grid.to_file(OUT_GPKG, layer="forecast_grid", driver="GPKG")
if points is not None and len(points) > 0:
    points.to_file(OUT_GPKG, layer="evidence_points", driver="GPKG")

grid.drop(columns="geometry").to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

plot_prox(grid, mask, OUT_PROX)
plot_final(grid, mask, points, OUT_PNG)
plot_compare(grid, mask, points, OUT_COMPARE)

metrics = {
    "base_dir": str(BASE_DIR),
    "grid_cells": int(len(grid)),
    "cell_size": CELL_SIZE,
    "use_supervised_requested": bool(USE_SUPERVISED),
    "use_supervised_applied": bool(use_supervised),
    "positive_cells": int(grid["target"].sum()),
    "top10_threshold": float(top_thr),
    "prospectivity_min": float(np.nanmin(grid["prospectivity"])),
    "prospectivity_p05": float(np.nanquantile(grid["prospectivity"], 0.05)),
    "prospectivity_p50": float(np.nanquantile(grid["prospectivity"], 0.50)),
    "prospectivity_p95": float(np.nanquantile(grid["prospectivity"], 0.95)),
    "prospectivity_max": float(np.nanmax(grid["prospectivity"])),
    "prognoz_min": float(np.nanmin(grid["prognoz"])),
    "prognoz_max": float(np.nanmax(grid["prognoz"])),
    "display_score_min": float(np.nanmin(grid["display_score"])),
    "display_score_max": float(np.nanmax(grid["display_score"])),
    "rf_score_min": float(np.nanmin(grid["rf_score"])),
    "rf_score_max": float(np.nanmax(grid["rf_score"])),
    "rf_score_sm_min": float(np.nanmin(grid["rf_score_sm"])),
    "rf_score_sm_max": float(np.nanmax(grid["rf_score_sm"])),
    "rf_test_proxy": None if rf_test_proxy is None else float(rf_test_proxy),
    "edge_false_penalty_min": float(np.nanmin(grid["edge_false_penalty"])),
    "edge_false_penalty_max": float(np.nanmax(grid["edge_false_penalty"])),
    "gold_zone_count": int(grid["gold_zone"].sum()),
    "point_count": int(len(points)) if points is not None else 0,
    "rf_feature_importance": feature_importance,
}
Path(OUT_JSON).write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

print("Готово.")
print(f"BASE_DIR: {BASE_DIR}")
print(f"PNG: {OUT_PNG}")
print(f"COMPARE: {OUT_COMPARE}")
print(f"GPKG: {OUT_GPKG}")
print(f"CSV: {OUT_CSV}")
print(f"JSON: {OUT_JSON}")
print("Диагностика:")
print(grid[["prospectivity", "prognoz", "display_score", "rf_score_sm", "gold_zone"]].describe())

Готово.
BASE_DIR: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз
PNG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v7_random_forest\gold_forecast_same_methods_fixed_v7_rf.png
COMPARE: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v7_random_forest\compare_same_methods_fixed_v7_rf.png
GPKG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v7_random_forest\gold_forecast_same_methods_fixed_v7_rf.gpkg
CSV: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v7_random_forest\grid_attributes_same_methods_fixed_v7_rf.csv
JSON: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v7_random_forest\metrics_same_methods_fixed_v7_rf.json
Диагностика:
       prospectivity       prognoz  display_score   rf_score_sm     gold_zone
count   15684.000000  15684.000000   15684.000000  15684.000000  15684.000000
mean        0.418994      0.581006       0.580971      0.190389      0.005547
std         0.254734      0.254734 

In [5]:
import os
import re
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm

from pyproj import CRS
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from minisom import MiniSom

warnings.filterwarnings("ignore")

# =========================================================
# НАСТРОЙКИ
# =========================================================
CELL_SIZE = 500
RANDOM_STATE = 42

# Основной режим: регрессия в приоритете
USE_SUPERVISED = True
TEST_SIZE = 0.30

# SOM / KMeans оставляем, но их вклад уменьшаем
SOM_X = 12
SOM_Y = 12
SOM_ITERS = 4000
N_CLUSTERS = 6

# Вклады в итоговую перспективность
W_ML = 0.60
W_GEO = 0.18
W_CLUSTER = 0.07
W_COINCIDENCE = 0.10
W_LOCAL_BONUS = 0.05

# proximity: magm делаем резче, потому что слой already buffered
Q_FACIES = 0.78
Q_PALEO = 0.76
Q_STRUCT = 0.72
Q_MAGM = 0.42
Q_TECT1 = 0.74
Q_TECT2 = 0.74

# визуализация
N_DISPLAY_CLASSES = 20
TOP_GOLD_Q = 0.06
TOP_LOCAL_Q = 0.94

# =========================================================
# ПУТИ
# =========================================================
def find_existing_base_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path("/mnt/data/prog_zip"),
        Path("/mnt/data"),
        Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз"),
    ]
    for base in candidates:
        shp_dir = base / "shp_dbf"
        if shp_dir.exists() and (shp_dir / "svita_new.shp").exists():
            return base
    raise FileNotFoundError("Не найден каталог с shp_dbf. Укажи BASE_DIR вручную.")

BASE_DIR = find_existing_base_dir()
SHP_DIR = BASE_DIR / "shp_dbf"

OUT_DIR = BASE_DIR / "same_methods_regression_priority_v5"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAFE_ALIAS_DIR = OUT_DIR / "_safe_shp_aliases"
SAFE_ALIAS_DIR.mkdir(parents=True, exist_ok=True)

OUT_GPKG = OUT_DIR / "gold_forecast_regression_priority_v5.gpkg"
OUT_PNG = OUT_DIR / "gold_forecast_regression_priority_v5.png"
OUT_COMPARE = OUT_DIR / "compare_regression_priority_v5.png"
OUT_PROX = OUT_DIR / "prox_magm_regression_priority_v5.png"
OUT_CSV = OUT_DIR / "grid_attributes_regression_priority_v5.csv"
OUT_JSON = OUT_DIR / "metrics_regression_priority_v5.json"

# =========================================================
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# =========================================================
def normalize_01(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    mn = np.nanmin(arr[finite])
    mx = np.nanmax(arr[finite])
    if np.isclose(mx, mn):
        out = np.full_like(arr, 0.5, dtype=float)
        out[~finite] = np.nan
        return out
    out = np.full_like(arr, np.nan, dtype=float)
    out[finite] = (arr[finite] - mn) / (mx - mn)
    return out

def robust_normalize_01(values, q_low=0.02, q_high=0.98):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    lo = np.nanquantile(arr[finite], q_low)
    hi = np.nanquantile(arr[finite], q_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or np.isclose(lo, hi):
        return normalize_01(arr)
    out = (arr - lo) / (hi - lo)
    out = np.clip(out, 0, 1)
    out[~finite] = np.nan
    return out

def read_sidecar_proj4(shp_path: Path):
    sidecar = shp_path.with_name(shp_path.stem + "_shp.pj4")
    if sidecar.exists():
        txt = sidecar.read_text(encoding="utf-8", errors="ignore")
        m = re.search(r"pj4=(.+)", txt)
        if m:
            return m.group(1).strip()
    return None

def prepare_ascii_aliases(shp_dir: Path, alias_dir: Path):
    aliases = {}
    stems = {}
    for name_b in os.listdir(os.fsencode(shp_dir)):
        if not name_b.endswith((b".shp", b".shx", b".dbf", b".prj", b".pj4")):
            continue
        if name_b.endswith(b"_shp.pj4"):
            continue
        base_b, ext_b = os.path.splitext(name_b)
        stems.setdefault(base_b, set()).add(ext_b)

    alias_idx = 0
    for base_b, exts in sorted(stems.items()):
        try:
            base_s = os.fsdecode(base_b)
            safe = all(ord(ch) < 128 and (ch.isalnum() or ch in "_.- ") for ch in base_s)
        except Exception:
            base_s = None
            safe = False

        if safe:
            aliases[base_s] = shp_dir / f"{base_s}.shp"
            continue

        alias_name = f"evidence_{alias_idx:02d}"
        alias_idx += 1
        for ext_b in exts:
            src = os.path.join(os.fsencode(shp_dir), base_b + ext_b)
            dst = alias_dir / f"{alias_name}{os.fsdecode(ext_b)}"
            shutil.copyfile(src, dst)
        pj4_src = os.path.join(os.fsencode(shp_dir), base_b + b"_shp.pj4")
        if os.path.exists(pj4_src):
            shutil.copyfile(pj4_src, alias_dir / f"{alias_name}_shp.pj4")
        aliases[alias_name] = alias_dir / f"{alias_name}.shp"
    return aliases

def load_layer(shp_path: Path):
    gdf = gpd.read_file(shp_path)
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    if gdf.crs is None:
        proj4 = read_sidecar_proj4(shp_path)
        if proj4:
            gdf = gdf.set_crs(CRS.from_proj4(proj4), allow_override=True)
    return gdf

def to_crs_safe(gdf, target_crs):
    if gdf.crs is None and target_crs is not None:
        return gdf.set_crs(target_crs, allow_override=True)
    if target_crs is None or gdf.crs == target_crs:
        return gdf
    return gdf.to_crs(target_crs)

def build_grid(mask, cell_size):
    mask_union = unary_union(mask.geometry)
    prepared_mask = prep(mask_union)

    minx, miny, maxx, maxy = mask.total_bounds
    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)

    rows = []
    cell_id = 0
    for r, y in enumerate(ys):
        for c, x in enumerate(xs):
            geom = box(x, y, x + cell_size, y + cell_size)
            if prepared_mask.intersects(geom):
                rows.append((cell_id, r, c, geom))
                cell_id += 1

    grid = gpd.GeoDataFrame(
        rows,
        columns=["cell_id", "row", "col", "geometry"],
        geometry="geometry",
        crs=mask.crs,
    )
    return grid, mask_union, (len(ys), len(xs))

def add_distance_feature(grid, source, name):
    source_union = unary_union(source.geometry)
    distances = np.empty(len(grid), dtype=float)
    for i, geom in enumerate(grid.geometry.values):
        distances[i] = 0.0 if geom.intersects(source_union) else geom.distance(source_union)
    grid[name] = distances
    return grid

def distance_to_proximity(distance, transform="sqrt", q=0.75):
    d = np.asarray(distance, dtype=float)
    d = np.clip(d, 0, None)

    if transform == "sqrt":
        dt = np.sqrt(d)
    elif transform == "cbrt":
        dt = np.cbrt(d)
    elif transform == "log1p":
        dt = np.log1p(d)
    else:
        dt = d

    scale = float(np.nanquantile(dt, q))
    if not np.isfinite(scale) or scale <= 0:
        scale = max(float(np.nanmean(dt)), 1.0)

    prox = np.exp(-dt / scale)
    return np.clip(prox, 0, 1)

def smooth_on_regular_grid(grid, value_col, shape, passes=1):
    try:
        from scipy.signal import convolve2d
    except Exception:
        return grid[value_col].to_numpy()

    arr = np.full(shape, np.nan, dtype=float)
    arr[grid["row"].to_numpy(), grid["col"].to_numpy()] = grid[value_col].to_numpy()

    kernel = np.array(
        [[1.0, 1.2, 1.0],
         [1.2, 3.0, 1.2],
         [1.0, 1.2, 1.0]],
        dtype=float,
    )

    smoothed = arr.copy()
    for _ in range(max(1, passes)):
        valid = np.isfinite(smoothed).astype(float)
        filled = np.nan_to_num(smoothed, nan=0.0)
        num = convolve2d(filled, kernel, mode="same", boundary="fill", fillvalue=0)
        den = convolve2d(valid, kernel, mode="same", boundary="fill", fillvalue=0)
        with np.errstate(divide="ignore", invalid="ignore"):
            smoothed = np.where(den > 0, num / den, np.nan)

    return smoothed[grid["row"].to_numpy(), grid["col"].to_numpy()]

def collect_points(mask_crs, aliases):
    point_layers = []
    for name, shp_path in aliases.items():
        if name in {"svita_new", "fasii", "glub_raz_nw", "glub_r_nw", "gr_dol_vp_poly", "kory", "dayki_buf"}:
            continue
        gdf = load_layer(shp_path)
        gdf = to_crs_safe(gdf, mask_crs)
        geom_types = {str(x) for x in gdf.geom_type.unique()}
        if "Point" in geom_types or "MultiPoint" in geom_types:
            point_layers.append(gdf)

    if not point_layers:
        return None

    pts = pd.concat(point_layers, ignore_index=True)
    pts = gpd.GeoDataFrame(pts, geometry="geometry", crs=mask_crs)
    return pts

def set_mask_extent(ax, mask):
    minx, miny, maxx, maxy = mask.total_bounds
    padx = (maxx - minx) * 0.02
    pady = (maxy - miny) * 0.02
    ax.set_xlim(minx - padx, maxx + padx)
    ax.set_ylim(miny - pady, maxy + pady)

def local_max_mask(grid, value_col, shape):
    try:
        from scipy.ndimage import maximum_filter
    except Exception:
        vals = grid[value_col].to_numpy()
        thr = np.nanquantile(vals, 0.98)
        return vals >= thr
    arr = np.full(shape, np.nan, dtype=float)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    vals = grid[value_col].to_numpy()
    arr[rows, cols] = vals
    filled = np.nan_to_num(arr, nan=-9999.0)
    locmax = maximum_filter(filled, size=3, mode="nearest")
    mask = np.isfinite(arr) & (filled >= locmax)
    return mask[rows, cols]

def make_display_classes(grid):
    prog_disp = robust_normalize_01(grid["prognoz"].to_numpy(), 0.02, 0.98)
    grid["display_score"] = prog_disp
    bins = np.linspace(0, 1, N_DISPLAY_CLASSES + 1)
    grid["display_class"] = np.digitize(prog_disp, bins[1:-1], right=False)
    return grid

def mark_gold_zones(grid, shape, mask_union):
    q_best = float(grid["prognoz"].quantile(TOP_GOLD_Q))
    q_local = float(grid["local_bonus"].quantile(TOP_LOCAL_Q))
    q_coinc = float(grid["coincidence_score"].quantile(TOP_LOCAL_Q))
    q_tmagm = float(grid["tect_magm_intersection"].quantile(0.70))

    mask_boundary = mask_union.boundary
    grid["dist_to_boundary"] = np.array([geom.distance(mask_boundary) for geom in grid.geometry])

    local_peak = local_max_mask(grid, "prospectivity", shape)

    core_gold = (
        (grid["prognoz"] <= q_best) &
        local_peak &
        (grid["tect_magm_intersection"] >= q_tmagm) &
        ((grid["local_bonus"] >= q_local) | (grid["coincidence_score"] >= q_coinc))
    )

    edge_gold = (
        (grid["dist_to_boundary"] <= CELL_SIZE * 1.10) &
        (grid["prox_magm"] >= float(grid["prox_magm"].quantile(0.84))) &
        (grid["tect_magm_intersection"] >= float(grid["tect_magm_intersection"].quantile(0.72))) &
        (grid["coincidence_score"] >= float(grid["coincidence_score"].quantile(0.72)))
    )

    grid["gold_zone"] = (core_gold | edge_gold).astype(int)
    return grid

def plot_prox(grid, mask, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    grid.plot(column="prox_magm", ax=ax, cmap="RdYlBu_r", linewidth=0, legend=True)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    set_mask_extent(ax, mask)
    ax.set_title("prox_magm")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_final(grid, mask, points, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr.N)
    grid.plot(column="display_class", ax=ax, cmap="bwr", norm=norm, linewidth=0, legend=True)
    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=ax, color="#f2d200", linewidth=0)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    if points is not None and len(points) > 0:
        points.plot(ax=ax, color="yellow", markersize=8, edgecolor="black", linewidth=0.25)
    set_mask_extent(ax, mask)
    ax.set_title("Итоговый прогноз")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_compare(grid, mask, points, out_png):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    grid.plot(column="prox_magm", ax=axes[0], cmap="RdYlBu_r", linewidth=0)
    mask.boundary.plot(ax=axes[0], color="black", linewidth=0.5)
    set_mask_extent(axes[0], mask)
    axes[0].set_title("prox_magm")
    axes[0].set_axis_off()

    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr.N)
    grid.plot(column="display_class", ax=axes[1], cmap="bwr", norm=norm, linewidth=0)
    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=axes[1], color="#f2d200", linewidth=0)
    mask.boundary.plot(ax=axes[1], color="black", linewidth=0.5)
    if points is not None and len(points) > 0:
        points.plot(ax=axes[1], color="yellow", markersize=8, edgecolor="black", linewidth=0.25)
    set_mask_extent(axes[1], mask)
    axes[1].set_title("Итоговый прогноз")
    axes[1].set_axis_off()

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

# =========================================================
# ЗАГРУЗКА
# =========================================================
aliases = prepare_ascii_aliases(SHP_DIR, SAFE_ALIAS_DIR)

mask = load_layer(aliases["svita_new"])
facies = to_crs_safe(load_layer(aliases["fasii"]), mask.crs)
tect1 = to_crs_safe(load_layer(aliases["glub_raz_nw"]), mask.crs)
tect2 = to_crs_safe(load_layer(aliases["glub_r_nw"]), mask.crs)
paleo = to_crs_safe(load_layer(aliases["gr_dol_vp_poly"]), mask.crs)
struct = to_crs_safe(load_layer(aliases["kory"]), mask.crs)
magm = to_crs_safe(load_layer(aliases["dayki_buf"]), mask.crs)
points = collect_points(mask.crs, aliases)

# =========================================================
# СЕТКА
# =========================================================
grid, mask_union, grid_shape = build_grid(mask, CELL_SIZE)

# =========================================================
# ДИСТАНЦИИ
# =========================================================
grid = add_distance_feature(grid, facies, "dist_facies")
grid = add_distance_feature(grid, paleo, "dist_paleo")
grid = add_distance_feature(grid, struct, "dist_struct")
grid = add_distance_feature(grid, magm, "dist_magm")
grid = add_distance_feature(grid, tect1, "dist_tect1")
grid = add_distance_feature(grid, tect2, "dist_tect2")

# =========================================================
# PROXIMITY
# =========================================================
grid["prox_facies"] = distance_to_proximity(grid["dist_facies"], transform="cbrt", q=Q_FACIES)
grid["prox_paleo"] = distance_to_proximity(grid["dist_paleo"], transform="cbrt", q=Q_PALEO)
grid["prox_struct"] = distance_to_proximity(grid["dist_struct"], transform="sqrt", q=Q_STRUCT)
grid["prox_magm"] = distance_to_proximity(grid["dist_magm"], transform="sqrt", q=Q_MAGM)
grid["prox_tect1"] = distance_to_proximity(grid["dist_tect1"], transform="cbrt", q=Q_TECT1)
grid["prox_tect2"] = distance_to_proximity(grid["dist_tect2"], transform="cbrt", q=Q_TECT2)

# =========================================================
# INTERACTIONS
# =========================================================
grid["tect_combo"] = 0.5 * (grid["prox_tect1"] + grid["prox_tect2"])
grid["tect_intersection"] = grid["prox_tect1"] * grid["prox_tect2"]
grid["tect_magm_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_magm"])
grid["tect_struct_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_struct"])
grid["paleo_struct_intersection"] = np.sqrt(grid["prox_paleo"] * grid["prox_struct"])

combo_core = (
    np.clip(grid["tect_combo"], 0, 1) *
    np.clip(0.55 * grid["prox_magm"] + 0.45 * grid["prox_struct"], 0, 1) *
    np.clip(0.60 * grid["prox_paleo"] + 0.40 * grid["prox_facies"], 0, 1)
)
grid["coincidence_score"] = robust_normalize_01(np.sqrt(np.clip(combo_core, 0, 1)), 0.02, 0.98)

tect_support = 0.45 * grid["prox_magm"] + 0.35 * grid["prox_struct"] + 0.20 * grid["prox_paleo"]
grid["tect_only_penalty"] = robust_normalize_01(np.clip(grid["tect_combo"] - tect_support, 0, 1), 0.02, 0.98)

# =========================================================
# GEO PRIOR
# =========================================================
grid["geo_score_raw"] = (
    0.12 * grid["prox_tect1"] +
    0.12 * grid["prox_tect2"] +
    0.14 * grid["prox_paleo"] +
    0.11 * grid["prox_struct"] +
    0.08 * grid["prox_facies"] +
    0.08 * grid["prox_magm"] +
    0.09 * grid["tect_intersection"] +
    0.09 * grid["tect_magm_intersection"] +
    0.05 * grid["tect_struct_intersection"] +
    0.04 * grid["paleo_struct_intersection"] +
    0.08 * grid["coincidence_score"] -
    0.09 * grid["tect_only_penalty"]
)

grid["geo_score"] = robust_normalize_01(grid["geo_score_raw"], 0.02, 0.98)
grid["geo_score_sm"] = robust_normalize_01(
    smooth_on_regular_grid(grid, "geo_score", grid_shape, passes=1),
    0.02, 0.98
)

# =========================================================
# TARGET + LOGISTIC REGRESSION
# =========================================================
grid["target"] = 0
grid["ml_score"] = grid["geo_score_sm"]
use_supervised = False
model_metrics = {}

feature_cols = [
    "prox_facies", "prox_paleo", "prox_struct", "prox_magm",
    "prox_tect1", "prox_tect2", "tect_combo", "tect_intersection",
    "tect_magm_intersection", "tect_struct_intersection",
    "paleo_struct_intersection", "coincidence_score",
    "tect_only_penalty", "geo_score_sm"
]

if USE_SUPERVISED and points is not None and len(points) > 0:
    try:
        joined = gpd.sjoin(
            points[["geometry"]],
            grid[["cell_id", "geometry"]],
            how="left",
            predicate="within"
        )
    except Exception:
        joined = gpd.sjoin(
            points[["geometry"]],
            grid[["cell_id", "geometry"]],
            how="left",
            op="within"
        )

    positive_cells = joined["cell_id"].dropna().astype(int).unique().tolist()
    grid.loc[grid["cell_id"].isin(positive_cells), "target"] = 1

    positives = int(grid["target"].sum())
    negatives = int((grid["target"] == 0).sum())

    if positives >= 10 and negatives > positives:
        X = grid[feature_cols].fillna(0).to_numpy()
        y = grid["target"].to_numpy()

        idx = np.arange(len(grid))
        try:
            train_idx, test_idx = train_test_split(
                idx,
                test_size=TEST_SIZE,
                random_state=RANDOM_STATE,
                stratify=y
            )
        except Exception:
            train_idx, test_idx = train_test_split(
                idx,
                test_size=TEST_SIZE,
                random_state=RANDOM_STATE
            )

        scaler_lr = StandardScaler()
        X_train = scaler_lr.fit_transform(X[train_idx])
        X_all = scaler_lr.transform(X)

        lr = LogisticRegression(
            random_state=RANDOM_STATE,
            max_iter=5000,
            class_weight="balanced"
        )
        lr.fit(X_train, y[train_idx])

        grid["ml_score"] = robust_normalize_01(lr.predict_proba(X_all)[:, 1], 0.02, 0.98)
        use_supervised = True

        if len(test_idx) > 0 and len(np.unique(y[test_idx])) > 1:
            from sklearn.metrics import roc_auc_score
            test_proba = lr.predict_proba(X_all[test_idx])[:, 1]
            model_metrics["test_auc"] = float(roc_auc_score(y[test_idx], test_proba))

        coef_df = pd.DataFrame({
            "feature": feature_cols,
            "coef": lr.coef_[0]
        }).sort_values("coef", ascending=False)
        coef_df.to_csv(OUT_DIR / "lr_feature_weights_v5.csv", index=False, encoding="utf-8-sig")
        model_metrics["positives"] = positives
        model_metrics["negatives"] = negatives

# =========================================================
# SOM + KMEANS
# =========================================================
X = grid[feature_cols].fillna(0).to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

som = MiniSom(
    x=SOM_X,
    y=SOM_Y,
    input_len=X_scaled.shape[1],
    sigma=1.1,
    learning_rate=0.38,
    random_seed=RANDOM_STATE
)
som.random_weights_init(X_scaled)
som.train_random(X_scaled, SOM_ITERS)

winners = np.array([som.winner(x) for x in X_scaled])
grid["som_x"] = winners[:, 0]
grid["som_y"] = winners[:, 1]
grid["som_node"] = grid["som_x"].astype(str) + "_" + grid["som_y"].astype(str)

som_weights = som.get_weights().reshape(SOM_X * SOM_Y, X_scaled.shape[1])
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=20)
neuron_cluster = kmeans.fit_predict(som_weights)

node_to_cluster = {}
idx = 0
for i in range(SOM_X):
    for j in range(SOM_Y):
        node_to_cluster[f"{i}_{j}"] = int(neuron_cluster[idx])
        idx += 1

grid["cluster"] = grid["som_node"].map(node_to_cluster).astype(int)

cluster_geo = grid.groupby("cluster")["geo_score_sm"].mean().reset_index(name="cluster_geo_mean")
cluster_ml = grid.groupby("cluster")["ml_score"].mean().reset_index(name="cluster_ml_mean")
cluster_coinc = grid.groupby("cluster")["coincidence_score"].mean().reset_index(name="cluster_coinc_mean")
cluster_stats = cluster_geo.merge(cluster_ml, on="cluster", how="outer").merge(cluster_coinc, on="cluster", how="outer")

if use_supervised:
    cluster_stats["cluster_score"] = robust_normalize_01(
        0.80 * cluster_stats["cluster_geo_mean"] +
        0.10 * cluster_stats["cluster_ml_mean"] +
        0.10 * cluster_stats["cluster_coinc_mean"],
        0.02, 0.98
    )
else:
    cluster_stats["cluster_score"] = robust_normalize_01(
        0.82 * cluster_stats["cluster_geo_mean"] +
        0.18 * cluster_stats["cluster_coinc_mean"],
        0.02, 0.98
    )

grid = grid.merge(cluster_stats[["cluster", "cluster_score"]], on="cluster", how="left")
grid["cluster_score"] = grid["cluster_score"].fillna(grid["geo_score_sm"])

# =========================================================
# ИТОГ: REGRESSION-PRIORITY
# =========================================================
grid["local_bonus"] = robust_normalize_01(
    0.38 * grid["tect_intersection"] +
    0.37 * grid["tect_magm_intersection"] +
    0.25 * grid["tect_struct_intersection"],
    0.02, 0.98
)

if use_supervised:
    grid["prospectivity_raw"] = (
        W_ML * grid["ml_score"] +
        W_GEO * grid["geo_score_sm"] +
        W_CLUSTER * grid["cluster_score"] +
        W_COINCIDENCE * grid["coincidence_score"] +
        W_LOCAL_BONUS * grid["local_bonus"]
    )
else:
    grid["prospectivity_raw"] = (
        0.78 * grid["geo_score_sm"] +
        0.08 * grid["cluster_score"] +
        0.09 * grid["coincidence_score"] +
        0.05 * grid["local_bonus"]
    )

grid["prospectivity"] = robust_normalize_01(grid["prospectivity_raw"], 0.02, 0.98)

# логика презентации: чем меньше prognoz, тем лучше
grid["prognoz"] = 1.0 - grid["prospectivity"]

top_thr = float(grid["prospectivity"].quantile(0.90))
grid["top10"] = (grid["prospectivity"] >= top_thr).astype(int)

try:
    grid["prospect_class"] = pd.qcut(
        grid["prospectivity"],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop"
    )
except Exception:
    grid["prospect_class"] = "medium"

grid = make_display_classes(grid)
grid = mark_gold_zones(grid, grid_shape, mask_union)

# =========================================================
# СОХРАНЕНИЕ
# =========================================================
if OUT_GPKG.exists():
    OUT_GPKG.unlink()

grid.to_file(OUT_GPKG, layer="forecast_grid", driver="GPKG")
if points is not None and len(points) > 0:
    points.to_file(OUT_GPKG, layer="evidence_points", driver="GPKG")

grid.drop(columns="geometry").to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

plot_prox(grid, mask, OUT_PROX)
plot_final(grid, mask, points, OUT_PNG)
plot_compare(grid, mask, points, OUT_COMPARE)

metrics = {
    "base_dir": str(BASE_DIR),
    "grid_cells": int(len(grid)),
    "cell_size": CELL_SIZE,
    "use_supervised_requested": bool(USE_SUPERVISED),
    "use_supervised_applied": bool(use_supervised),
    "top10_threshold": float(top_thr),
    "point_count": int(len(points)) if points is not None else 0,
    "gold_zone_count": int(grid["gold_zone"].sum()),
    "prospectivity_min": float(np.nanmin(grid["prospectivity"])),
    "prospectivity_p05": float(np.nanquantile(grid["prospectivity"], 0.05)),
    "prospectivity_p50": float(np.nanquantile(grid["prospectivity"], 0.50)),
    "prospectivity_p95": float(np.nanquantile(grid["prospectivity"], 0.95)),
    "prospectivity_max": float(np.nanmax(grid["prospectivity"])),
    "prognoz_min": float(np.nanmin(grid["prognoz"])),
    "prognoz_max": float(np.nanmax(grid["prognoz"])),
    "display_score_min": float(np.nanmin(grid["display_score"])),
    "display_score_max": float(np.nanmax(grid["display_score"])),
    **model_metrics
}

Path(OUT_JSON).write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

print("Готово.")
print(f"BASE_DIR: {BASE_DIR}")
print(f"PNG: {OUT_PNG}")
print(f"COMPARE: {OUT_COMPARE}")
print(f"GPKG: {OUT_GPKG}")
print(f"CSV: {OUT_CSV}")
print(f"JSON: {OUT_JSON}")
print("Диагностика:")
print(grid[["prospectivity", "prognoz", "display_score", "gold_zone"]].describe())

Готово.
BASE_DIR: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз
PNG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_regression_priority_v5\gold_forecast_regression_priority_v5.png
COMPARE: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_regression_priority_v5\compare_regression_priority_v5.png
GPKG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_regression_priority_v5\gold_forecast_regression_priority_v5.gpkg
CSV: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_regression_priority_v5\grid_attributes_regression_priority_v5.csv
JSON: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_regression_priority_v5\metrics_regression_priority_v5.json
Диагностика:
       prospectivity       prognoz  display_score     gold_zone
count   15684.000000  15684.000000   15684.000000  15684.000000
mean        0.526561      0.473439       0.473458      0.014027
std         0.253682      0.253682       0.253704      0.117606
min         0.000000   

In [6]:
import os
import re
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap

from pyproj import CRS
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from minisom import MiniSom

warnings.filterwarnings("ignore")

# =========================================================
# НАСТРОЙКИ
# =========================================================
CELL_SIZE = 500
RANDOM_STATE = 42

# Регрессия в приоритете, но уже без чрезмерной агрессии
USE_SUPERVISED = True
TEST_SIZE = 0.30

# SOM / KMeans сохраняем как слабую структурную поправку
SOM_X = 12
SOM_Y = 12
SOM_ITERS = 4000
N_CLUSTERS = 6

# Итоговые веса
W_ML = 0.48
W_GEO = 0.24
W_CLUSTER = 0.06
W_COINCIDENCE = 0.12
W_LOCAL_BONUS = 0.10

# proximity
Q_FACIES = 0.78
Q_PALEO = 0.76
Q_STRUCT = 0.72
Q_MAGM = 0.40
Q_TECT1 = 0.74
Q_TECT2 = 0.74

# визуализация
N_DISPLAY_CLASSES = 20
TOP_GOLD_Q = 0.055
TOP_LOCAL_Q = 0.935

# =========================================================
# ПУТИ
# =========================================================
def find_existing_base_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path("/mnt/data/prog_zip"),
        Path("/mnt/data"),
        Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз"),
    ]
    for base in candidates:
        shp_dir = base / "shp_dbf"
        if shp_dir.exists() and (shp_dir / "svita_new.shp").exists():
            return base
    raise FileNotFoundError("Не найден каталог с shp_dbf. Укажи BASE_DIR вручную.")

BASE_DIR = find_existing_base_dir()
SHP_DIR = BASE_DIR / "shp_dbf"

OUT_DIR = BASE_DIR / "same_methods_regression_priority_v6"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAFE_ALIAS_DIR = OUT_DIR / "_safe_shp_aliases"
SAFE_ALIAS_DIR.mkdir(parents=True, exist_ok=True)

OUT_GPKG = OUT_DIR / "gold_forecast_regression_priority_v6.gpkg"
OUT_PNG = OUT_DIR / "gold_forecast_regression_priority_v6.png"
OUT_COMPARE = OUT_DIR / "compare_regression_priority_v6.png"
OUT_PROX = OUT_DIR / "prox_magm_regression_priority_v6.png"
OUT_CSV = OUT_DIR / "grid_attributes_regression_priority_v6.csv"
OUT_JSON = OUT_DIR / "metrics_regression_priority_v6.json"

# =========================================================
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# =========================================================
def normalize_01(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    mn = np.nanmin(arr[finite])
    mx = np.nanmax(arr[finite])
    out = np.full_like(arr, np.nan, dtype=float)
    if np.isclose(mx, mn):
        out[finite] = 0.5
        return out
    out[finite] = (arr[finite] - mn) / (mx - mn)
    return out

def robust_normalize_01(values, q_low=0.04, q_high=0.96):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    lo = np.nanquantile(arr[finite], q_low)
    hi = np.nanquantile(arr[finite], q_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or np.isclose(lo, hi):
        return normalize_01(arr)
    out = (arr - lo) / (hi - lo)
    out = np.clip(out, 0, 1)
    out[~finite] = np.nan
    return out

def read_sidecar_proj4(shp_path: Path):
    sidecar = shp_path.with_name(shp_path.stem + "_shp.pj4")
    if sidecar.exists():
        txt = sidecar.read_text(encoding="utf-8", errors="ignore")
        m = re.search(r"pj4=(.+)", txt)
        if m:
            return m.group(1).strip()
    return None

def prepare_ascii_aliases(shp_dir: Path, alias_dir: Path):
    aliases = {}
    stems = {}
    for name_b in os.listdir(os.fsencode(shp_dir)):
        if not name_b.endswith((b".shp", b".shx", b".dbf", b".prj", b".pj4")):
            continue
        if name_b.endswith(b"_shp.pj4"):
            continue
        base_b, ext_b = os.path.splitext(name_b)
        stems.setdefault(base_b, set()).add(ext_b)

    alias_idx = 0
    for base_b, exts in sorted(stems.items()):
        try:
            base_s = os.fsdecode(base_b)
            safe = all(ord(ch) < 128 and (ch.isalnum() or ch in "_.- ") for ch in base_s)
        except Exception:
            safe = False
            base_s = None

        if safe:
            aliases[base_s] = shp_dir / f"{base_s}.shp"
            continue

        alias_name = f"evidence_{alias_idx:02d}"
        alias_idx += 1
        for ext_b in exts:
            src = os.path.join(os.fsencode(shp_dir), base_b + ext_b)
            dst = alias_dir / f"{alias_name}{os.fsdecode(ext_b)}"
            shutil.copyfile(src, dst)
        pj4_src = os.path.join(os.fsencode(shp_dir), base_b + b"_shp.pj4")
        if os.path.exists(pj4_src):
            shutil.copyfile(pj4_src, alias_dir / f"{alias_name}_shp.pj4")
        aliases[alias_name] = alias_dir / f"{alias_name}.shp"
    return aliases

def load_layer(shp_path: Path):
    gdf = gpd.read_file(shp_path)
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    if gdf.crs is None:
        proj4 = read_sidecar_proj4(shp_path)
        if proj4:
            gdf = gdf.set_crs(CRS.from_proj4(proj4), allow_override=True)
    return gdf

def to_crs_safe(gdf, target_crs):
    if gdf.crs is None and target_crs is not None:
        return gdf.set_crs(target_crs, allow_override=True)
    if target_crs is None or gdf.crs == target_crs:
        return gdf
    return gdf.to_crs(target_crs)

def build_grid(mask, cell_size):
    mask_union = unary_union(mask.geometry)
    prepared_mask = prep(mask_union)

    minx, miny, maxx, maxy = mask.total_bounds
    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)

    rows = []
    cell_id = 0
    for r, y in enumerate(ys):
        for c, x in enumerate(xs):
            geom = box(x, y, x + cell_size, y + cell_size)
            if prepared_mask.intersects(geom):
                rows.append((cell_id, r, c, geom))
                cell_id += 1

    grid = gpd.GeoDataFrame(rows, columns=["cell_id", "row", "col", "geometry"], geometry="geometry", crs=mask.crs)
    return grid, mask_union, (len(ys), len(xs))

def add_distance_feature(grid, source, name):
    source_union = unary_union(source.geometry)
    distances = np.empty(len(grid), dtype=float)
    for i, geom in enumerate(grid.geometry.values):
        distances[i] = 0.0 if geom.intersects(source_union) else geom.distance(source_union)
    grid[name] = distances
    return grid

def distance_to_proximity(distance, transform="sqrt", q=0.75):
    d = np.asarray(distance, dtype=float)
    d = np.clip(d, 0, None)
    if transform == "sqrt":
        dt = np.sqrt(d)
    elif transform == "cbrt":
        dt = np.cbrt(d)
    elif transform == "log1p":
        dt = np.log1p(d)
    else:
        dt = d

    scale = float(np.nanquantile(dt, q))
    if not np.isfinite(scale) or scale <= 0:
        scale = max(float(np.nanmean(dt)), 1.0)
    prox = np.exp(-dt / scale)
    return np.clip(prox, 0, 1)

def smooth_on_regular_grid(grid, value_col, shape, passes=1):
    try:
        from scipy.signal import convolve2d
    except Exception:
        return grid[value_col].to_numpy()

    arr = np.full(shape, np.nan, dtype=float)
    arr[grid["row"].to_numpy(), grid["col"].to_numpy()] = grid[value_col].to_numpy()

    kernel = np.array(
        [[1.0, 1.25, 1.0],
         [1.25, 3.2, 1.25],
         [1.0, 1.25, 1.0]],
        dtype=float
    )

    smoothed = arr.copy()
    for _ in range(max(1, passes)):
        valid = np.isfinite(smoothed).astype(float)
        filled = np.nan_to_num(smoothed, nan=0.0)
        num = convolve2d(filled, kernel, mode="same", boundary="fill", fillvalue=0)
        den = convolve2d(valid, kernel, mode="same", boundary="fill", fillvalue=0)
        with np.errstate(divide="ignore", invalid="ignore"):
            smoothed = np.where(den > 0, num / den, np.nan)

    return smoothed[grid["row"].to_numpy(), grid["col"].to_numpy()]

def collect_points(mask_crs, aliases):
    point_layers = []
    for name, shp_path in aliases.items():
        if name in {"svita_new", "fasii", "glub_raz_nw", "glub_r_nw", "gr_dol_vp_poly", "kory", "dayki_buf"}:
            continue
        gdf = load_layer(shp_path)
        gdf = to_crs_safe(gdf, mask_crs)
        geom_types = {str(x) for x in gdf.geom_type.unique()}
        if "Point" in geom_types or "MultiPoint" in geom_types:
            point_layers.append(gdf)
    if not point_layers:
        return None
    pts = pd.concat(point_layers, ignore_index=True)
    return gpd.GeoDataFrame(pts, geometry="geometry", crs=mask_crs)

def set_mask_extent(ax, mask):
    minx, miny, maxx, maxy = mask.total_bounds
    padx = (maxx - minx) * 0.02
    pady = (maxy - miny) * 0.02
    ax.set_xlim(minx - padx, maxx + padx)
    ax.set_ylim(miny - pady, maxy + pady)

def local_max_mask(grid, value_col, shape):
    try:
        from scipy.ndimage import maximum_filter
    except Exception:
        vals = grid[value_col].to_numpy()
        thr = np.nanquantile(vals, 0.985)
        return vals >= thr
    arr = np.full(shape, np.nan, dtype=float)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    vals = grid[value_col].to_numpy()
    arr[rows, cols] = vals
    filled = np.nan_to_num(arr, nan=-9999.0)
    locmax = maximum_filter(filled, size=3, mode="nearest")
    mask = np.isfinite(arr) & (filled >= locmax)
    return mask[rows, cols]

def make_display_classes(grid):
    # Компрессия крайних значений, чтобы красный был мягче и меньше резал глаза.
    prog_disp = robust_normalize_01(grid["prognoz"].to_numpy(), 0.06, 0.94)
    prog_disp = 0.5 + 0.72 * (prog_disp - 0.5)
    prog_disp = np.clip(prog_disp, 0, 1)
    grid["display_score"] = prog_disp
    bins = np.linspace(0, 1, N_DISPLAY_CLASSES + 1)
    grid["display_class"] = np.digitize(prog_disp, bins[1:-1], right=False)
    return grid

def mark_gold_zones(grid, shape, mask_union):
    q_best = float(grid["prognoz"].quantile(TOP_GOLD_Q))
    q_local = float(grid["local_bonus"].quantile(TOP_LOCAL_Q))
    q_coinc = float(grid["coincidence_score"].quantile(TOP_LOCAL_Q))
    q_tmagm = float(grid["tect_magm_intersection"].quantile(0.68))
    q_magm = float(grid["prox_magm"].quantile(0.83))

    mask_boundary = mask_union.boundary
    grid["dist_to_boundary"] = np.array([geom.distance(mask_boundary) for geom in grid.geometry])

    local_peak = local_max_mask(grid, "prospectivity_sm", shape)

    core_gold = (
        (grid["prognoz"] <= q_best) &
        local_peak &
        (grid["tect_magm_intersection"] >= q_tmagm) &
        ((grid["local_bonus"] >= q_local) | (grid["coincidence_score"] >= q_coinc))
    )

    # Отдельно спасаем нижние и краевые узкие магмато-тектонические зоны
    edge_gold = (
        (grid["dist_to_boundary"] <= CELL_SIZE * 1.25) &
        (grid["prox_magm"] >= q_magm) &
        (grid["tect_magm_intersection"] >= float(grid["tect_magm_intersection"].quantile(0.70))) &
        (grid["coincidence_score"] >= float(grid["coincidence_score"].quantile(0.70)))
    )

    grid["gold_zone"] = (core_gold | edge_gold).astype(int)
    return grid

def custom_bwr_soft():
    # Более мягкая палитра, чем стандартная bwr
    colors = [
        "#1f1fff", "#3333f0", "#5050ea", "#6f6fe8", "#8c8ce6",
        "#a6a6e6", "#c2c2ea", "#dcdcf0", "#efefef", "#f4eded",
        "#f0d6d6", "#edc0c0", "#eca4a4", "#ec8c8c", "#ee7474",
        "#f25f5f", "#f84d4d", "#ff3d3d", "#ff2b2b", "#ff1e1e"
    ]
    return ListedColormap(colors)

def plot_prox(grid, mask, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    grid.plot(column="prox_magm", ax=ax, cmap="RdYlBu_r", linewidth=0, legend=True)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    set_mask_extent(ax, mask)
    ax.set_title("prox_magm")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_final(grid, mask, points, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, custom_bwr_soft().N)
    grid.plot(column="display_class", ax=ax, cmap=custom_bwr_soft(), norm=norm, linewidth=0, legend=True)
    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=ax, color="#f2d200", linewidth=0)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    if points is not None and len(points) > 0:
        points.plot(ax=ax, color="yellow", markersize=7, edgecolor="black", linewidth=0.25)
    set_mask_extent(ax, mask)
    ax.set_title("Итоговый прогноз")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_compare(grid, mask, points, out_png):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    grid.plot(column="prox_magm", ax=axes[0], cmap="RdYlBu_r", linewidth=0)
    mask.boundary.plot(ax=axes[0], color="black", linewidth=0.5)
    set_mask_extent(axes[0], mask)
    axes[0].set_title("prox_magm")
    axes[0].set_axis_off()

    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, custom_bwr_soft().N)
    grid.plot(column="display_class", ax=axes[1], cmap=custom_bwr_soft(), norm=norm, linewidth=0)
    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=axes[1], color="#f2d200", linewidth=0)
    mask.boundary.plot(ax=axes[1], color="black", linewidth=0.5)
    if points is not None and len(points) > 0:
        points.plot(ax=axes[1], color="yellow", markersize=7, edgecolor="black", linewidth=0.25)
    set_mask_extent(axes[1], mask)
    axes[1].set_title("Итоговый прогноз")
    axes[1].set_axis_off()

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

# =========================================================
# ЗАГРУЗКА
# =========================================================
aliases = prepare_ascii_aliases(SHP_DIR, SAFE_ALIAS_DIR)

mask = load_layer(aliases["svita_new"])
facies = to_crs_safe(load_layer(aliases["fasii"]), mask.crs)
tect1 = to_crs_safe(load_layer(aliases["glub_raz_nw"]), mask.crs)
tect2 = to_crs_safe(load_layer(aliases["glub_r_nw"]), mask.crs)
paleo = to_crs_safe(load_layer(aliases["gr_dol_vp_poly"]), mask.crs)
struct = to_crs_safe(load_layer(aliases["kory"]), mask.crs)
magm = to_crs_safe(load_layer(aliases["dayki_buf"]), mask.crs)
points = collect_points(mask.crs, aliases)

# =========================================================
# СЕТКА
# =========================================================
grid, mask_union, grid_shape = build_grid(mask, CELL_SIZE)

# =========================================================
# ДИСТАНЦИИ
# =========================================================
grid = add_distance_feature(grid, facies, "dist_facies")
grid = add_distance_feature(grid, paleo, "dist_paleo")
grid = add_distance_feature(grid, struct, "dist_struct")
grid = add_distance_feature(grid, magm, "dist_magm")
grid = add_distance_feature(grid, tect1, "dist_tect1")
grid = add_distance_feature(grid, tect2, "dist_tect2")

# =========================================================
# PROXIMITY
# =========================================================
grid["prox_facies"] = distance_to_proximity(grid["dist_facies"], transform="cbrt", q=Q_FACIES)
grid["prox_paleo"] = distance_to_proximity(grid["dist_paleo"], transform="cbrt", q=Q_PALEO)
grid["prox_struct"] = distance_to_proximity(grid["dist_struct"], transform="sqrt", q=Q_STRUCT)
grid["prox_magm"] = distance_to_proximity(grid["dist_magm"], transform="sqrt", q=Q_MAGM)
grid["prox_tect1"] = distance_to_proximity(grid["dist_tect1"], transform="cbrt", q=Q_TECT1)
grid["prox_tect2"] = distance_to_proximity(grid["dist_tect2"], transform="cbrt", q=Q_TECT2)

# =========================================================
# INTERACTIONS
# =========================================================
grid["tect_combo"] = 0.5 * (grid["prox_tect1"] + grid["prox_tect2"])
grid["tect_intersection"] = grid["prox_tect1"] * grid["prox_tect2"]
grid["tect_magm_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_magm"])
grid["tect_struct_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_struct"])
grid["paleo_struct_intersection"] = np.sqrt(grid["prox_paleo"] * grid["prox_struct"])

combo_core = (
    np.clip(grid["tect_combo"], 0, 1) *
    np.clip(0.58 * grid["prox_magm"] + 0.42 * grid["prox_struct"], 0, 1) *
    np.clip(0.60 * grid["prox_paleo"] + 0.40 * grid["prox_facies"], 0, 1)
)
grid["coincidence_score"] = robust_normalize_01(np.sqrt(np.clip(combo_core, 0, 1)), 0.04, 0.96)

# Штраф для ложных tectonic-only зон усиливаем
tect_support = 0.48 * grid["prox_magm"] + 0.32 * grid["prox_struct"] + 0.20 * grid["prox_paleo"]
grid["tect_only_penalty"] = robust_normalize_01(np.clip(grid["tect_combo"] - tect_support, 0, 1), 0.04, 0.96)

# =========================================================
# GEO PRIOR
# =========================================================
grid["geo_score_raw"] = (
    0.10 * grid["prox_tect1"] +
    0.10 * grid["prox_tect2"] +
    0.14 * grid["prox_paleo"] +
    0.11 * grid["prox_struct"] +
    0.08 * grid["prox_facies"] +
    0.09 * grid["prox_magm"] +
    0.07 * grid["tect_intersection"] +
    0.10 * grid["tect_magm_intersection"] +
    0.05 * grid["tect_struct_intersection"] +
    0.04 * grid["paleo_struct_intersection"] +
    0.11 * grid["coincidence_score"] -
    0.12 * grid["tect_only_penalty"]
)
grid["geo_score"] = robust_normalize_01(grid["geo_score_raw"], 0.04, 0.96)
grid["geo_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "geo_score", grid_shape, passes=1), 0.04, 0.96)

# =========================================================
# TARGET + LOGISTIC REGRESSION
# =========================================================
grid["target"] = 0
grid["ml_score"] = grid["geo_score_sm"]
use_supervised = False
model_metrics = {}

feature_cols = [
    "prox_facies", "prox_paleo", "prox_struct", "prox_magm",
    "prox_tect1", "prox_tect2", "tect_combo", "tect_intersection",
    "tect_magm_intersection", "tect_struct_intersection",
    "paleo_struct_intersection", "coincidence_score",
    "tect_only_penalty", "geo_score_sm"
]

if USE_SUPERVISED and points is not None and len(points) > 0:
    try:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", predicate="within")
    except Exception:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", op="within")

    positive_cells = joined["cell_id"].dropna().astype(int).unique().tolist()
    grid.loc[grid["cell_id"].isin(positive_cells), "target"] = 1

    positives = int(grid["target"].sum())
    negatives = int((grid["target"] == 0).sum())

    if positives >= 10 and negatives > positives:
        X = grid[feature_cols].fillna(0).to_numpy()
        y = grid["target"].to_numpy()
        idx = np.arange(len(grid))

        try:
            train_idx, test_idx = train_test_split(idx, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
        except Exception:
            train_idx, test_idx = train_test_split(idx, test_size=TEST_SIZE, random_state=RANDOM_STATE)

        scaler_lr = StandardScaler()
        X_train = scaler_lr.fit_transform(X[train_idx])
        X_all = scaler_lr.transform(X)

        lr = LogisticRegression(random_state=RANDOM_STATE, max_iter=5000, class_weight="balanced")
        lr.fit(X_train, y[train_idx])

        raw_ml = lr.predict_proba(X_all)[:, 1]
        grid["ml_score_raw"] = raw_ml
        grid["ml_score"] = robust_normalize_01(raw_ml, 0.04, 0.96)

        # Легкое пространственное сглаживание именно regression-score,
        # чтобы убрать слишком резкие красные/синие пятна.
        grid["ml_score_sm"] = robust_normalize_01(
            smooth_on_regular_grid(grid, "ml_score", grid_shape, passes=2),
            0.04, 0.96
        )
        use_supervised = True

        if len(test_idx) > 0 and len(np.unique(y[test_idx])) > 1:
            from sklearn.metrics import roc_auc_score
            model_metrics["test_auc"] = float(roc_auc_score(y[test_idx], raw_ml[test_idx]))

        coef_df = pd.DataFrame({"feature": feature_cols, "coef": lr.coef_[0]}).sort_values("coef", ascending=False)
        coef_df.to_csv(OUT_DIR / "lr_feature_weights_v6.csv", index=False, encoding="utf-8-sig")
        model_metrics["positives"] = positives
        model_metrics["negatives"] = negatives
    else:
        grid["ml_score_sm"] = grid["geo_score_sm"]
else:
    grid["ml_score_sm"] = grid["geo_score_sm"]

# =========================================================
# SOM + KMEANS
# =========================================================
X = grid[feature_cols].fillna(0).to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

som = MiniSom(x=SOM_X, y=SOM_Y, input_len=X_scaled.shape[1], sigma=1.1, learning_rate=0.38, random_seed=RANDOM_STATE)
som.random_weights_init(X_scaled)
som.train_random(X_scaled, SOM_ITERS)

winners = np.array([som.winner(x) for x in X_scaled])
grid["som_x"] = winners[:, 0]
grid["som_y"] = winners[:, 1]
grid["som_node"] = grid["som_x"].astype(str) + "_" + grid["som_y"].astype(str)

som_weights = som.get_weights().reshape(SOM_X * SOM_Y, X_scaled.shape[1])
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=20)
neuron_cluster = kmeans.fit_predict(som_weights)

node_to_cluster = {}
idx = 0
for i in range(SOM_X):
    for j in range(SOM_Y):
        node_to_cluster[f"{i}_{j}"] = int(neuron_cluster[idx])
        idx += 1

grid["cluster"] = grid["som_node"].map(node_to_cluster).astype(int)

cluster_geo = grid.groupby("cluster")["geo_score_sm"].mean().reset_index(name="cluster_geo_mean")
cluster_ml = grid.groupby("cluster")["ml_score_sm"].mean().reset_index(name="cluster_ml_mean")
cluster_coinc = grid.groupby("cluster")["coincidence_score"].mean().reset_index(name="cluster_coinc_mean")
cluster_stats = cluster_geo.merge(cluster_ml, on="cluster", how="outer").merge(cluster_coinc, on="cluster", how="outer")

if use_supervised:
    cluster_stats["cluster_score"] = robust_normalize_01(
        0.78 * cluster_stats["cluster_geo_mean"] +
        0.12 * cluster_stats["cluster_ml_mean"] +
        0.10 * cluster_stats["cluster_coinc_mean"],
        0.04, 0.96
    )
else:
    cluster_stats["cluster_score"] = robust_normalize_01(
        0.82 * cluster_stats["cluster_geo_mean"] +
        0.18 * cluster_stats["cluster_coinc_mean"],
        0.04, 0.96
    )

grid = grid.merge(cluster_stats[["cluster", "cluster_score"]], on="cluster", how="left")
grid["cluster_score"] = grid["cluster_score"].fillna(grid["geo_score_sm"])

# =========================================================
# ИТОГ
# =========================================================
grid["local_bonus"] = robust_normalize_01(
    0.30 * grid["tect_intersection"] +
    0.45 * grid["tect_magm_intersection"] +
    0.25 * grid["tect_struct_intersection"],
    0.04, 0.96
)

if use_supervised:
    grid["prospectivity_raw"] = (
        W_ML * grid["ml_score_sm"] +
        W_GEO * grid["geo_score_sm"] +
        W_CLUSTER * grid["cluster_score"] +
        W_COINCIDENCE * grid["coincidence_score"] +
        W_LOCAL_BONUS * grid["local_bonus"]
    )
else:
    grid["prospectivity_raw"] = (
        0.72 * grid["geo_score_sm"] +
        0.06 * grid["cluster_score"] +
        0.12 * grid["coincidence_score"] +
        0.10 * grid["local_bonus"]
    )

grid["prospectivity"] = robust_normalize_01(grid["prospectivity_raw"], 0.04, 0.96)
grid["prospectivity_sm"] = robust_normalize_01(
    smooth_on_regular_grid(grid, "prospectivity", grid_shape, passes=1),
    0.04, 0.96
)

# В логике презентации: меньше prognoz = лучше
grid["prognoz"] = 1.0 - grid["prospectivity_sm"]

top_thr = float(grid["prospectivity_sm"].quantile(0.90))
grid["top10"] = (grid["prospectivity_sm"] >= top_thr).astype(int)

try:
    grid["prospect_class"] = pd.qcut(
        grid["prospectivity_sm"],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop"
    )
except Exception:
    grid["prospect_class"] = "medium"

grid = make_display_classes(grid)
grid = mark_gold_zones(grid, grid_shape, mask_union)

# =========================================================
# СОХРАНЕНИЕ
# =========================================================
if OUT_GPKG.exists():
    OUT_GPKG.unlink()

grid.to_file(OUT_GPKG, layer="forecast_grid", driver="GPKG")
if points is not None and len(points) > 0:
    points.to_file(OUT_GPKG, layer="evidence_points", driver="GPKG")

grid.drop(columns="geometry").to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

plot_prox(grid, mask, OUT_PROX)
plot_final(grid, mask, points, OUT_PNG)
plot_compare(grid, mask, points, OUT_COMPARE)

metrics = {
    "base_dir": str(BASE_DIR),
    "grid_cells": int(len(grid)),
    "cell_size": CELL_SIZE,
    "use_supervised_requested": bool(USE_SUPERVISED),
    "use_supervised_applied": bool(use_supervised),
    "top10_threshold": float(top_thr),
    "point_count": int(len(points)) if points is not None else 0,
    "gold_zone_count": int(grid["gold_zone"].sum()),
    "gold_zone_share": float(grid["gold_zone"].mean()),
    "prospectivity_min": float(np.nanmin(grid["prospectivity_sm"])),
    "prospectivity_p05": float(np.nanquantile(grid["prospectivity_sm"], 0.05)),
    "prospectivity_p50": float(np.nanquantile(grid["prospectivity_sm"], 0.50)),
    "prospectivity_p95": float(np.nanquantile(grid["prospectivity_sm"], 0.95)),
    "prospectivity_max": float(np.nanmax(grid["prospectivity_sm"])),
    "prognoz_min": float(np.nanmin(grid["prognoz"])),
    "prognoz_max": float(np.nanmax(grid["prognoz"])),
    "display_score_min": float(np.nanmin(grid["display_score"])),
    "display_score_max": float(np.nanmax(grid["display_score"])),
    **model_metrics
}
Path(OUT_JSON).write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

print("Готово.")
print(f"BASE_DIR: {BASE_DIR}")
print(f"PNG: {OUT_PNG}")
print(f"COMPARE: {OUT_COMPARE}")
print(f"GPKG: {OUT_GPKG}")
print(f"CSV: {OUT_CSV}")
print(f"JSON: {OUT_JSON}")
print("Диагностика:")
print(grid[["prospectivity_sm", "prognoz", "display_score", "gold_zone"]].describe())

Готово.
BASE_DIR: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз
PNG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_regression_priority_v6\gold_forecast_regression_priority_v6.png
COMPARE: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_regression_priority_v6\compare_regression_priority_v6.png
GPKG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_regression_priority_v6\gold_forecast_regression_priority_v6.gpkg
CSV: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_regression_priority_v6\grid_attributes_regression_priority_v6.csv
JSON: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_regression_priority_v6\metrics_regression_priority_v6.json
Диагностика:
       prospectivity_sm       prognoz  display_score     gold_zone
count      15684.000000  15684.000000   15684.000000  15684.000000
mean           0.550620      0.449380       0.467169      0.028947
std            0.271351      0.271351       0.206783      0.167662
min        

In [7]:
import os
import re
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm

from pyproj import CRS
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from minisom import MiniSom

warnings.filterwarnings("ignore")

# =========================================================
# НАСТРОЙКИ
# =========================================================
CELL_SIZE = 500
RANDOM_STATE = 42

# методы сохраняем
SOM_X = 12
SOM_Y = 12
SOM_ITERS = 4000
N_CLUSTERS = 6
USE_SUPERVISED = True

# регрессия становится основой,
# но геологический блок и кластеры остаются как стабилизаторы
W_REG = 0.58
W_GEO = 0.24
W_COINCIDENCE = 0.10
W_CLUSTER = 0.04
W_LOCAL = 0.04

# proximity
Q_FACIES = 0.78
Q_PALEO = 0.76
Q_STRUCT = 0.72
Q_MAGM = 0.42
Q_TECT1 = 0.74
Q_TECT2 = 0.74

# визуализация и gold-зоны
N_DISPLAY_CLASSES = 20
TOP_GOLD_Q = 0.06
TOP_LOCAL_Q = 0.94

# =========================================================
# ПУТИ
# =========================================================
def find_existing_base_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path("/mnt/data/prog_zip"),
        Path("/mnt/data"),
        Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз"),
    ]
    for base in candidates:
        shp_dir = base / "shp_dbf"
        if shp_dir.exists() and (shp_dir / "svita_new.shp").exists():
            return base
    raise FileNotFoundError("Не найден каталог с shp_dbf. Укажи BASE_DIR вручную.")

BASE_DIR = find_existing_base_dir()
SHP_DIR = BASE_DIR / "shp_dbf"
OUT_DIR = BASE_DIR / "same_methods_fixed_v4_regression_base"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAFE_ALIAS_DIR = OUT_DIR / "_safe_shp_aliases"
SAFE_ALIAS_DIR.mkdir(parents=True, exist_ok=True)

OUT_GPKG = OUT_DIR / "gold_forecast_same_methods_fixed_v4.gpkg"
OUT_PNG = OUT_DIR / "gold_forecast_same_methods_fixed_v4.png"
OUT_PROX = OUT_DIR / "prox_magm_same_methods_fixed_v4.png"
OUT_COMPARE = OUT_DIR / "compare_same_methods_fixed_v4.png"
OUT_CSV = OUT_DIR / "grid_attributes_same_methods_fixed_v4.csv"
OUT_JSON = OUT_DIR / "metrics_same_methods_fixed_v4.json"

# =========================================================
# ВСПОМОГАТЕЛЬНЫЕ
# =========================================================
def normalize_01(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    mn = np.nanmin(arr[finite])
    mx = np.nanmax(arr[finite])
    if np.isclose(mx, mn):
        return np.full_like(arr, 0.5, dtype=float)
    out = np.full_like(arr, np.nan, dtype=float)
    out[finite] = (arr[finite] - mn) / (mx - mn)
    return out

def robust_normalize_01(values, q_low=0.03, q_high=0.97):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    lo = np.nanquantile(arr[finite], q_low)
    hi = np.nanquantile(arr[finite], q_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or np.isclose(lo, hi):
        return normalize_01(arr)
    out = (arr - lo) / (hi - lo)
    return np.clip(out, 0, 1)

def read_sidecar_proj4(shp_path: Path):
    sidecar = shp_path.with_name(shp_path.stem + "_shp.pj4")
    if sidecar.exists():
        txt = sidecar.read_text(encoding="utf-8", errors="ignore")
        m = re.search(r"pj4=(.+)", txt)
        if m:
            return m.group(1).strip()
    return None

def prepare_ascii_aliases(shp_dir: Path, alias_dir: Path):
    aliases = {}
    stems = {}
    for name_b in os.listdir(os.fsencode(shp_dir)):
        if not name_b.endswith((b".shp", b".shx", b".dbf", b".prj", b".pj4")):
            continue
        if name_b.endswith(b"_shp.pj4"):
            continue
        base_b, ext_b = os.path.splitext(name_b)
        stems.setdefault(base_b, set()).add(ext_b)

    alias_idx = 0
    for base_b, exts in sorted(stems.items()):
        try:
            base_s = os.fsdecode(base_b)
            safe = all(ord(ch) < 128 and (ch.isalnum() or ch in "_-. ") for ch in base_s)
        except Exception:
            base_s = None
            safe = False

        if safe:
            aliases[base_s] = shp_dir / f"{base_s}.shp"
            continue

        alias_name = f"evidence_{alias_idx:02d}"
        alias_idx += 1
        for ext_b in exts:
            src = os.path.join(os.fsencode(shp_dir), base_b + ext_b)
            dst = alias_dir / f"{alias_name}{os.fsdecode(ext_b)}"
            shutil.copyfile(src, dst)
        pj4_src = os.path.join(os.fsencode(shp_dir), base_b + b"_shp.pj4")
        if os.path.exists(pj4_src):
            shutil.copyfile(pj4_src, alias_dir / f"{alias_name}_shp.pj4")
        aliases[alias_name] = alias_dir / f"{alias_name}.shp"
    return aliases

def load_layer(shp_path: Path):
    gdf = gpd.read_file(shp_path)
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    if gdf.crs is None:
        proj4 = read_sidecar_proj4(shp_path)
        if proj4:
            gdf = gdf.set_crs(CRS.from_proj4(proj4), allow_override=True)
    return gdf

def to_crs_safe(gdf, target_crs):
    if gdf.crs is None and target_crs is not None:
        return gdf.set_crs(target_crs, allow_override=True)
    if target_crs is None or gdf.crs == target_crs:
        return gdf
    return gdf.to_crs(target_crs)

def build_grid(mask, cell_size):
    mask_union = unary_union(mask.geometry)
    prepared_mask = prep(mask_union)
    minx, miny, maxx, maxy = mask.total_bounds
    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)
    rows = []
    cell_id = 0
    for r, y in enumerate(ys):
        for c, x in enumerate(xs):
            geom = box(x, y, x + cell_size, y + cell_size)
            if prepared_mask.intersects(geom):
                rows.append((cell_id, r, c, geom))
                cell_id += 1
    grid = gpd.GeoDataFrame(rows, columns=["cell_id", "row", "col", "geometry"], geometry="geometry", crs=mask.crs)
    return grid, mask_union, (len(ys), len(xs))

def add_distance_feature(grid, source, name):
    source_union = unary_union(source.geometry)
    distances = np.empty(len(grid), dtype=float)
    for i, geom in enumerate(grid.geometry.values):
        distances[i] = 0.0 if geom.intersects(source_union) else geom.distance(source_union)
    grid[name] = distances
    return grid

def distance_to_proximity(distance, transform="sqrt", q=0.75):
    d = np.asarray(distance, dtype=float)
    d = np.clip(d, 0, None)
    if transform == "sqrt":
        dt = np.sqrt(d)
    elif transform == "cbrt":
        dt = np.cbrt(d)
    elif transform == "log1p":
        dt = np.log1p(d)
    else:
        dt = d
    scale = float(np.nanquantile(dt, q))
    if not np.isfinite(scale) or scale <= 0:
        scale = max(float(np.nanmean(dt)), 1.0)
    prox = np.exp(-dt / scale)
    return np.clip(prox, 0, 1)

def smooth_on_regular_grid(grid, value_col, shape, passes=1):
    try:
        from scipy.signal import convolve2d
    except Exception:
        return grid[value_col].to_numpy()
    arr = np.full(shape, np.nan, dtype=float)
    arr[grid["row"].to_numpy(), grid["col"].to_numpy()] = grid[value_col].to_numpy()
    kernel = np.array([[1.0, 1.2, 1.0], [1.2, 3.0, 1.2], [1.0, 1.2, 1.0]], dtype=float)
    smoothed = arr.copy()
    for _ in range(max(1, passes)):
        valid = np.isfinite(smoothed).astype(float)
        filled = np.nan_to_num(smoothed, nan=0.0)
        num = convolve2d(filled, kernel, mode="same", boundary="fill", fillvalue=0)
        den = convolve2d(valid, kernel, mode="same", boundary="fill", fillvalue=0)
        with np.errstate(divide="ignore", invalid="ignore"):
            smoothed = np.where(den > 0, num / den, np.nan)
    return smoothed[grid["row"].to_numpy(), grid["col"].to_numpy()]

def collect_points(mask_crs, aliases):
    point_layers = []
    for name, shp_path in aliases.items():
        if name in {"svita_new", "fasii", "glub_raz_nw", "glub_r_nw", "gr_dol_vp_poly", "kory", "dayki_buf"}:
            continue
        gdf = load_layer(shp_path)
        gdf = to_crs_safe(gdf, mask_crs)
        geom_types = {str(x) for x in gdf.geom_type.unique()}
        if "Point" in geom_types or "MultiPoint" in geom_types:
            point_layers.append(gdf)
    if not point_layers:
        return None
    pts = pd.concat(point_layers, ignore_index=True)
    return gpd.GeoDataFrame(pts, geometry="geometry", crs=mask_crs)

def set_mask_extent(ax, mask):
    minx, miny, maxx, maxy = mask.total_bounds
    padx = (maxx - minx) * 0.02
    pady = (maxy - miny) * 0.02
    ax.set_xlim(minx - padx, maxx + padx)
    ax.set_ylim(miny - pady, maxy + pady)

def local_max_mask(grid, value_col, shape):
    try:
        from scipy.ndimage import maximum_filter
    except Exception:
        vals = grid[value_col].to_numpy()
        thr = np.nanquantile(vals, 0.98)
        return vals >= thr
    arr = np.full(shape, np.nan, dtype=float)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    vals = grid[value_col].to_numpy()
    arr[rows, cols] = vals
    filled = np.nan_to_num(arr, nan=-9999.0)
    locmax = maximum_filter(filled, size=3, mode="nearest")
    mask = np.isfinite(arr) & (filled >= locmax)
    return mask[rows, cols]

def make_display_classes(grid):
    prog_disp = robust_normalize_01(grid["prognoz"].to_numpy(), 0.02, 0.98)
    grid["display_score"] = prog_disp
    bins = np.linspace(0, 1, N_DISPLAY_CLASSES + 1)
    grid["display_class"] = np.digitize(prog_disp, bins[1:-1], right=False)
    return grid

def mark_gold_zones(grid, shape, mask_union):
    q_best = float(grid["prognoz"].quantile(TOP_GOLD_Q))
    q_local = float(grid["local_bonus"].quantile(TOP_LOCAL_Q))
    q_coinc = float(grid["coincidence_score"].quantile(TOP_LOCAL_Q))
    q_magm = float(grid["prox_magm"].quantile(0.84))
    q_tmagm = float(grid["tect_magm_intersection"].quantile(0.72))
    q_reg = float(grid["regression_score"].quantile(0.90))

    mask_boundary = mask_union.boundary
    grid["dist_to_boundary"] = np.array([geom.distance(mask_boundary) for geom in grid.geometry])

    local_peak = local_max_mask(grid, "prospectivity", shape)

    core_gold = (
        (grid["prognoz"] <= q_best) &
        local_peak &
        (grid["regression_score"] >= q_reg) &
        (grid["tect_magm_intersection"] >= float(grid["tect_magm_intersection"].quantile(0.70))) &
        ((grid["local_bonus"] >= q_local) | (grid["coincidence_score"] >= q_coinc))
    )

    edge_gold = (
        (grid["dist_to_boundary"] <= CELL_SIZE * 1.10) &
        (grid["prox_magm"] >= q_magm) &
        (grid["tect_magm_intersection"] >= q_tmagm) &
        (grid["coincidence_score"] >= float(grid["coincidence_score"].quantile(0.72))) &
        (grid["regression_score"] >= float(grid["regression_score"].quantile(0.78)))
    )

    grid["gold_zone"] = (core_gold | edge_gold).astype(int)
    return grid

def plot_prox(grid, mask, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    grid.plot(column="prox_magm", ax=ax, cmap="RdYlBu_r", linewidth=0, legend=True)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    set_mask_extent(ax, mask)
    ax.set_title("prox_magm")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_final(grid, mask, points, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr.N)
    grid.plot(column="display_class", ax=ax, cmap="bwr", norm=norm, linewidth=0, legend=True)
    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=ax, color="#f2d200", linewidth=0)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    if points is not None and len(points) > 0:
        points.plot(ax=ax, color="yellow", markersize=8, edgecolor="black", linewidth=0.25)
    set_mask_extent(ax, mask)
    ax.set_title("Итоговый прогноз")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_compare(grid, mask, points, out_png):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    grid.plot(column="prox_magm", ax=axes[0], cmap="RdYlBu_r", linewidth=0)
    mask.boundary.plot(ax=axes[0], color="black", linewidth=0.5)
    set_mask_extent(axes[0], mask)
    axes[0].set_title("prox_magm")
    axes[0].set_axis_off()

    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr.N)
    grid.plot(column="display_class", ax=axes[1], cmap="bwr", norm=norm, linewidth=0)
    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=axes[1], color="#f2d200", linewidth=0)
    mask.boundary.plot(ax=axes[1], color="black", linewidth=0.5)
    if points is not None and len(points) > 0:
        points.plot(ax=axes[1], color="yellow", markersize=8, edgecolor="black", linewidth=0.25)
    set_mask_extent(axes[1], mask)
    axes[1].set_title("Итоговый прогноз")
    axes[1].set_axis_off()

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

# =========================================================
# ЗАГРУЗКА
# =========================================================
aliases = prepare_ascii_aliases(SHP_DIR, SAFE_ALIAS_DIR)
mask = load_layer(aliases["svita_new"])
facies = to_crs_safe(load_layer(aliases["fasii"]), mask.crs)
tect1 = to_crs_safe(load_layer(aliases["glub_raz_nw"]), mask.crs)
tect2 = to_crs_safe(load_layer(aliases["glub_r_nw"]), mask.crs)
paleo = to_crs_safe(load_layer(aliases["gr_dol_vp_poly"]), mask.crs)
struct = to_crs_safe(load_layer(aliases["kory"]), mask.crs)
magm = to_crs_safe(load_layer(aliases["dayki_buf"]), mask.crs)
points = collect_points(mask.crs, aliases)

# =========================================================
# СЕТКА
# =========================================================
grid, mask_union, grid_shape = build_grid(mask, CELL_SIZE)

# =========================================================
# ДИСТАНЦИИ
# =========================================================
grid = add_distance_feature(grid, facies, "dist_facies")
grid = add_distance_feature(grid, paleo, "dist_paleo")
grid = add_distance_feature(grid, struct, "dist_struct")
grid = add_distance_feature(grid, magm, "dist_magm")
grid = add_distance_feature(grid, tect1, "dist_tect1")
grid = add_distance_feature(grid, tect2, "dist_tect2")

# =========================================================
# PROXIMITY
# =========================================================
grid["prox_facies"] = distance_to_proximity(grid["dist_facies"], transform="cbrt", q=Q_FACIES)
grid["prox_paleo"] = distance_to_proximity(grid["dist_paleo"], transform="cbrt", q=Q_PALEO)
grid["prox_struct"] = distance_to_proximity(grid["dist_struct"], transform="sqrt", q=Q_STRUCT)
grid["prox_magm"] = distance_to_proximity(grid["dist_magm"], transform="sqrt", q=Q_MAGM)
grid["prox_tect1"] = distance_to_proximity(grid["dist_tect1"], transform="cbrt", q=Q_TECT1)
grid["prox_tect2"] = distance_to_proximity(grid["dist_tect2"], transform="cbrt", q=Q_TECT2)

# =========================================================
# INTERACTIONS
# =========================================================
grid["tect_combo"] = 0.5 * (grid["prox_tect1"] + grid["prox_tect2"])
grid["tect_intersection"] = grid["prox_tect1"] * grid["prox_tect2"]
grid["tect_magm_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_magm"])
grid["tect_struct_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_struct"])
grid["paleo_struct_intersection"] = np.sqrt(grid["prox_paleo"] * grid["prox_struct"])

combo_core = (
    np.clip(grid["tect_combo"], 0, 1) *
    np.clip(0.55 * grid["prox_magm"] + 0.45 * grid["prox_struct"], 0, 1) *
    np.clip(0.60 * grid["prox_paleo"] + 0.40 * grid["prox_facies"], 0, 1)
)
grid["coincidence_score"] = robust_normalize_01(np.sqrt(np.clip(combo_core, 0, 1)), 0.02, 0.98)

tect_support = 0.45 * grid["prox_magm"] + 0.35 * grid["prox_struct"] + 0.20 * grid["prox_paleo"]
grid["tect_only_penalty"] = robust_normalize_01(np.clip(grid["tect_combo"] - tect_support, 0, 1), 0.02, 0.98)

# =========================================================
# GEO SCORE - ослабленный, как стабилизатор
# =========================================================
grid["geo_score_raw"] = (
    0.12 * grid["prox_tect1"] +
    0.12 * grid["prox_tect2"] +
    0.14 * grid["prox_paleo"] +
    0.11 * grid["prox_struct"] +
    0.08 * grid["prox_facies"] +
    0.08 * grid["prox_magm"] +
    0.09 * grid["tect_intersection"] +
    0.09 * grid["tect_magm_intersection"] +
    0.05 * grid["tect_struct_intersection"] +
    0.04 * grid["paleo_struct_intersection"] +
    0.08 * grid["coincidence_score"] -
    0.09 * grid["tect_only_penalty"]
)
grid["geo_score"] = robust_normalize_01(grid["geo_score_raw"], 0.02, 0.98)
grid["geo_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "geo_score", grid_shape, passes=1), 0.02, 0.98)

# =========================================================
# REGRESSION AS BASE
# =========================================================
grid["target"] = 0
grid["regression_score"] = grid["geo_score_sm"]
use_supervised = False
reg_test_auc_proxy = None

feature_cols = [
    "prox_facies", "prox_paleo", "prox_struct", "prox_magm",
    "prox_tect1", "prox_tect2", "tect_combo", "tect_intersection",
    "tect_magm_intersection", "tect_struct_intersection",
    "paleo_struct_intersection", "coincidence_score", "tect_only_penalty",
    "geo_score_sm"
]

if USE_SUPERVISED and points is not None and len(points) > 0:
    try:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", predicate="within")
    except Exception:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", op="within")

    positive_cells = joined["cell_id"].dropna().astype(int).unique().tolist()
    grid.loc[grid["cell_id"].isin(positive_cells), "target"] = 1

    pos = int(grid["target"].sum())
    neg = int((grid["target"] == 0).sum())

    if pos >= 20 and neg > pos:
        X = grid[feature_cols].fillna(0).to_numpy()
        y = grid["target"].to_numpy()
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # Небольшая проверка на holdout, потом учим на всей выборке
        X_train, X_test, y_train, y_test = train_test_split(
            X_scaled, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
        )
        lr_eval = LogisticRegression(
            random_state=RANDOM_STATE,
            max_iter=4000,
            class_weight="balanced",
            C=0.8
        )
        lr_eval.fit(X_train, y_train)
        test_prob = lr_eval.predict_proba(X_test)[:, 1]

        # proxy-метрика без дополнительных библиотек
        y_test = np.asarray(y_test)
        pos_mean = float(np.mean(test_prob[y_test == 1])) if np.any(y_test == 1) else np.nan
        neg_mean = float(np.mean(test_prob[y_test == 0])) if np.any(y_test == 0) else np.nan
        reg_test_auc_proxy = pos_mean - neg_mean

        lr = LogisticRegression(
            random_state=RANDOM_STATE,
            max_iter=4000,
            class_weight="balanced",
            C=0.8
        )
        lr.fit(X_scaled, y)
        grid["regression_score"] = robust_normalize_01(lr.predict_proba(X_scaled)[:, 1], 0.02, 0.98)
        use_supervised = True

grid["ml_score"] = grid["regression_score"]

# =========================================================
# SOM + KMEANS
# =========================================================
X = grid[feature_cols].fillna(0).to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

som = MiniSom(
    x=SOM_X, y=SOM_Y, input_len=X_scaled.shape[1],
    sigma=1.1, learning_rate=0.38, random_seed=RANDOM_STATE
)
som.random_weights_init(X_scaled)
som.train_random(X_scaled, SOM_ITERS)

winners = np.array([som.winner(x) for x in X_scaled])
grid["som_x"] = winners[:, 0]
grid["som_y"] = winners[:, 1]
grid["som_node"] = grid["som_x"].astype(str) + "_" + grid["som_y"].astype(str)

som_weights = som.get_weights().reshape(SOM_X * SOM_Y, X_scaled.shape[1])
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=20)
neuron_cluster = kmeans.fit_predict(som_weights)

node_to_cluster = {}
idx = 0
for i in range(SOM_X):
    for j in range(SOM_Y):
        node_to_cluster[f"{i}_{j}"] = int(neuron_cluster[idx])
        idx += 1
grid["cluster"] = grid["som_node"].map(node_to_cluster).astype(int)

cluster_geo = grid.groupby("cluster")["geo_score_sm"].mean().reset_index(name="cluster_geo_mean")
cluster_reg = grid.groupby("cluster")["regression_score"].mean().reset_index(name="cluster_reg_mean")
cluster_coinc = grid.groupby("cluster")["coincidence_score"].mean().reset_index(name="cluster_coinc_mean")
cluster_stats = cluster_geo.merge(cluster_reg, on="cluster", how="outer").merge(cluster_coinc, on="cluster", how="outer")

if use_supervised:
    cluster_hits = grid.groupby("cluster").agg(cells=("cell_id", "count"), positives=("target", "sum")).reset_index()
    cluster_hits["hit_rate"] = cluster_hits["positives"] / cluster_hits["cells"]
    cluster_stats = cluster_stats.merge(cluster_hits, on="cluster", how="left")
    cluster_stats["hit_rate"] = cluster_stats["hit_rate"].fillna(0)
    cluster_stats["cluster_score"] = robust_normalize_01(
        0.30 * cluster_stats["cluster_geo_mean"] +
        0.45 * cluster_stats["cluster_reg_mean"] +
        0.10 * cluster_stats["cluster_coinc_mean"] +
        0.15 * robust_normalize_01(cluster_stats["hit_rate"], 0.0, 1.0),
        0.02, 0.98,
    )
else:
    cluster_stats["cluster_score"] = robust_normalize_01(
        0.45 * cluster_stats["cluster_geo_mean"] +
        0.40 * cluster_stats["cluster_reg_mean"] +
        0.15 * cluster_stats["cluster_coinc_mean"],
        0.02, 0.98,
    )

grid = grid.merge(cluster_stats[["cluster", "cluster_score"]], on="cluster", how="left")
grid["cluster_score"] = grid["cluster_score"].fillna(grid["geo_score_sm"])

# =========================================================
# ИТОГ
# =========================================================
grid["local_bonus"] = robust_normalize_01(
    0.38 * grid["tect_intersection"] +
    0.37 * grid["tect_magm_intersection"] +
    0.25 * grid["tect_struct_intersection"],
    0.02, 0.98,
)

grid["prospectivity_raw"] = (
    W_REG * grid["regression_score"] +
    W_GEO * grid["geo_score_sm"] +
    W_COINCIDENCE * grid["coincidence_score"] +
    W_CLUSTER * grid["cluster_score"] +
    W_LOCAL * grid["local_bonus"]
)

# Дополнительное слабое ослабление tectonic-only после сборки
grid["prospectivity_raw"] = grid["prospectivity_raw"] - 0.04 * grid["tect_only_penalty"]

grid["prospectivity"] = robust_normalize_01(grid["prospectivity_raw"], 0.02, 0.98)

# в логике презентации: меньше = лучше
grid["prognoz"] = 1.0 - grid["prospectivity"]

top_thr = float(grid["prospectivity"].quantile(0.90))
grid["top10"] = (grid["prospectivity"] >= top_thr).astype(int)

try:
    grid["prospect_class"] = pd.qcut(
        grid["prospectivity"],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop"
    )
except Exception:
    grid["prospect_class"] = "medium"

grid = make_display_classes(grid)
grid = mark_gold_zones(grid, grid_shape, mask_union)

# =========================================================
# СОХРАНЕНИЕ
# =========================================================
if OUT_GPKG.exists():
    OUT_GPKG.unlink()

grid.to_file(OUT_GPKG, layer="forecast_grid", driver="GPKG")
if points is not None and len(points) > 0:
    points.to_file(OUT_GPKG, layer="evidence_points", driver="GPKG")

grid.drop(columns="geometry").to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

plot_prox(grid, mask, OUT_PROX)
plot_final(grid, mask, points, OUT_PNG)
plot_compare(grid, mask, points, OUT_COMPARE)

metrics = {
    "base_dir": str(BASE_DIR),
    "grid_cells": int(len(grid)),
    "cell_size": CELL_SIZE,
    "use_supervised_requested": bool(USE_SUPERVISED),
    "use_supervised_applied": bool(use_supervised),
    "positive_cells": int(grid["target"].sum()),
    "top10_threshold": float(top_thr),
    "prospectivity_min": float(np.nanmin(grid["prospectivity"])),
    "prospectivity_p05": float(np.nanquantile(grid["prospectivity"], 0.05)),
    "prospectivity_p50": float(np.nanquantile(grid["prospectivity"], 0.50)),
    "prospectivity_p95": float(np.nanquantile(grid["prospectivity"], 0.95)),
    "prospectivity_max": float(np.nanmax(grid["prospectivity"])),
    "prognoz_min": float(np.nanmin(grid["prognoz"])),
    "prognoz_max": float(np.nanmax(grid["prognoz"])),
    "display_score_min": float(np.nanmin(grid["display_score"])),
    "display_score_max": float(np.nanmax(grid["display_score"])),
    "regression_score_min": float(np.nanmin(grid["regression_score"])),
    "regression_score_max": float(np.nanmax(grid["regression_score"])),
    "reg_test_auc_proxy": None if reg_test_auc_proxy is None else float(reg_test_auc_proxy),
    "gold_zone_count": int(grid["gold_zone"].sum()),
    "point_count": int(len(points)) if points is not None else 0,
}
Path(OUT_JSON).write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

print("Готово.")
print(f"BASE_DIR: {BASE_DIR}")
print(f"PNG: {OUT_PNG}")
print(f"COMPARE: {OUT_COMPARE}")
print(f"GPKG: {OUT_GPKG}")
print(f"CSV: {OUT_CSV}")
print(f"JSON: {OUT_JSON}")
print("Диагностика:")
print(grid[["prospectivity", "prognoz", "display_score", "regression_score", "gold_zone"]].describe())

Готово.
BASE_DIR: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз
PNG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v4_regression_base\gold_forecast_same_methods_fixed_v4.png
COMPARE: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v4_regression_base\compare_same_methods_fixed_v4.png
GPKG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v4_regression_base\gold_forecast_same_methods_fixed_v4.gpkg
CSV: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v4_regression_base\grid_attributes_same_methods_fixed_v4.csv
JSON: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v4_regression_base\metrics_same_methods_fixed_v4.json
Диагностика:
       prospectivity       prognoz  display_score  regression_score  \
count   15684.000000  15684.000000   15684.000000      15684.000000   
mean        0.577727      0.422273       0.422280          0.533993   
std         0.249733      0.249733       0.249738          0.

In [8]:

import os
import re
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm

from pyproj import CRS
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from minisom import MiniSom

warnings.filterwarnings("ignore")

# =========================================================
# НАСТРОЙКИ
# =========================================================
CELL_SIZE = 500
RANDOM_STATE = 42

# методы сохраняем
SOM_X = 12
SOM_Y = 12
SOM_ITERS = 4000
N_CLUSTERS = 6
USE_SUPERVISED = True

# регрессия становится основой,
# но геологический блок и кластеры остаются как стабилизаторы
W_REG = 0.42
W_GEO = 0.34
W_COINCIDENCE = 0.12
W_CLUSTER = 0.05
W_LOCAL = 0.07
SHOW_POINTS = False

# proximity
Q_FACIES = 0.78
Q_PALEO = 0.76
Q_STRUCT = 0.72
Q_MAGM = 0.42
Q_TECT1 = 0.74
Q_TECT2 = 0.74

# визуализация и gold-зоны
N_DISPLAY_CLASSES = 20
TOP_GOLD_Q = 0.05
TOP_LOCAL_Q = 0.95

# =========================================================
# ПУТИ
# =========================================================
def find_existing_base_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path("/mnt/data/prog_zip"),
        Path("/mnt/data"),
        Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз"),
    ]
    for base in candidates:
        shp_dir = base / "shp_dbf"
        if shp_dir.exists() and (shp_dir / "svita_new.shp").exists():
            return base
    raise FileNotFoundError("Не найден каталог с shp_dbf. Укажи BASE_DIR вручную.")

BASE_DIR = find_existing_base_dir()
SHP_DIR = BASE_DIR / "shp_dbf"
OUT_DIR = BASE_DIR / "same_methods_fixed_v5_regression_base"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAFE_ALIAS_DIR = OUT_DIR / "_safe_shp_aliases"
SAFE_ALIAS_DIR.mkdir(parents=True, exist_ok=True)

OUT_GPKG = OUT_DIR / "gold_forecast_same_methods_fixed_v5.gpkg"
OUT_PNG = OUT_DIR / "gold_forecast_same_methods_fixed_v5.png"
OUT_PROX = OUT_DIR / "prox_magm_same_methods_fixed_v5.png"
OUT_COMPARE = OUT_DIR / "compare_same_methods_fixed_v5.png"
OUT_CSV = OUT_DIR / "grid_attributes_same_methods_fixed_v5.csv"
OUT_JSON = OUT_DIR / "metrics_same_methods_fixed_v5.json"

# =========================================================
# ВСПОМОГАТЕЛЬНЫЕ
# =========================================================
def normalize_01(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    mn = np.nanmin(arr[finite])
    mx = np.nanmax(arr[finite])
    if np.isclose(mx, mn):
        return np.full_like(arr, 0.5, dtype=float)
    out = np.full_like(arr, np.nan, dtype=float)
    out[finite] = (arr[finite] - mn) / (mx - mn)
    return out

def robust_normalize_01(values, q_low=0.03, q_high=0.97):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    lo = np.nanquantile(arr[finite], q_low)
    hi = np.nanquantile(arr[finite], q_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or np.isclose(lo, hi):
        return normalize_01(arr)
    out = (arr - lo) / (hi - lo)
    return np.clip(out, 0, 1)

def read_sidecar_proj4(shp_path: Path):
    sidecar = shp_path.with_name(shp_path.stem + "_shp.pj4")
    if sidecar.exists():
        txt = sidecar.read_text(encoding="utf-8", errors="ignore")
        m = re.search(r"pj4=(.+)", txt)
        if m:
            return m.group(1).strip()
    return None

def prepare_ascii_aliases(shp_dir: Path, alias_dir: Path):
    aliases = {}
    stems = {}
    for name_b in os.listdir(os.fsencode(shp_dir)):
        if not name_b.endswith((b".shp", b".shx", b".dbf", b".prj", b".pj4")):
            continue
        if name_b.endswith(b"_shp.pj4"):
            continue
        base_b, ext_b = os.path.splitext(name_b)
        stems.setdefault(base_b, set()).add(ext_b)

    alias_idx = 0
    for base_b, exts in sorted(stems.items()):
        try:
            base_s = os.fsdecode(base_b)
            safe = all(ord(ch) < 128 and (ch.isalnum() or ch in "_-. ") for ch in base_s)
        except Exception:
            base_s = None
            safe = False

        if safe:
            aliases[base_s] = shp_dir / f"{base_s}.shp"
            continue

        alias_name = f"evidence_{alias_idx:02d}"
        alias_idx += 1
        for ext_b in exts:
            src = os.path.join(os.fsencode(shp_dir), base_b + ext_b)
            dst = alias_dir / f"{alias_name}{os.fsdecode(ext_b)}"
            shutil.copyfile(src, dst)
        pj4_src = os.path.join(os.fsencode(shp_dir), base_b + b"_shp.pj4")
        if os.path.exists(pj4_src):
            shutil.copyfile(pj4_src, alias_dir / f"{alias_name}_shp.pj4")
        aliases[alias_name] = alias_dir / f"{alias_name}.shp"
    return aliases

def load_layer(shp_path: Path):
    gdf = gpd.read_file(shp_path)
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    if gdf.crs is None:
        proj4 = read_sidecar_proj4(shp_path)
        if proj4:
            gdf = gdf.set_crs(CRS.from_proj4(proj4), allow_override=True)
    return gdf

def to_crs_safe(gdf, target_crs):
    if gdf.crs is None and target_crs is not None:
        return gdf.set_crs(target_crs, allow_override=True)
    if target_crs is None or gdf.crs == target_crs:
        return gdf
    return gdf.to_crs(target_crs)

def build_grid(mask, cell_size):
    mask_union = unary_union(mask.geometry)
    prepared_mask = prep(mask_union)
    minx, miny, maxx, maxy = mask.total_bounds
    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)
    rows = []
    cell_id = 0
    for r, y in enumerate(ys):
        for c, x in enumerate(xs):
            geom = box(x, y, x + cell_size, y + cell_size)
            if prepared_mask.intersects(geom):
                rows.append((cell_id, r, c, geom))
                cell_id += 1
    grid = gpd.GeoDataFrame(rows, columns=["cell_id", "row", "col", "geometry"], geometry="geometry", crs=mask.crs)
    return grid, mask_union, (len(ys), len(xs))

def add_distance_feature(grid, source, name):
    source_union = unary_union(source.geometry)
    distances = np.empty(len(grid), dtype=float)
    for i, geom in enumerate(grid.geometry.values):
        distances[i] = 0.0 if geom.intersects(source_union) else geom.distance(source_union)
    grid[name] = distances
    return grid

def distance_to_proximity(distance, transform="sqrt", q=0.75):
    d = np.asarray(distance, dtype=float)
    d = np.clip(d, 0, None)
    if transform == "sqrt":
        dt = np.sqrt(d)
    elif transform == "cbrt":
        dt = np.cbrt(d)
    elif transform == "log1p":
        dt = np.log1p(d)
    else:
        dt = d
    scale = float(np.nanquantile(dt, q))
    if not np.isfinite(scale) or scale <= 0:
        scale = max(float(np.nanmean(dt)), 1.0)
    prox = np.exp(-dt / scale)
    return np.clip(prox, 0, 1)

def smooth_on_regular_grid(grid, value_col, shape, passes=1):
    try:
        from scipy.signal import convolve2d
    except Exception:
        return grid[value_col].to_numpy()
    arr = np.full(shape, np.nan, dtype=float)
    arr[grid["row"].to_numpy(), grid["col"].to_numpy()] = grid[value_col].to_numpy()
    kernel = np.array([[1.0, 1.2, 1.0], [1.2, 3.0, 1.2], [1.0, 1.2, 1.0]], dtype=float)
    smoothed = arr.copy()
    for _ in range(max(1, passes)):
        valid = np.isfinite(smoothed).astype(float)
        filled = np.nan_to_num(smoothed, nan=0.0)
        num = convolve2d(filled, kernel, mode="same", boundary="fill", fillvalue=0)
        den = convolve2d(valid, kernel, mode="same", boundary="fill", fillvalue=0)
        with np.errstate(divide="ignore", invalid="ignore"):
            smoothed = np.where(den > 0, num / den, np.nan)
    return smoothed[grid["row"].to_numpy(), grid["col"].to_numpy()]

def collect_points(mask_crs, aliases):
    point_layers = []
    for name, shp_path in aliases.items():
        if name in {"svita_new", "fasii", "glub_raz_nw", "glub_r_nw", "gr_dol_vp_poly", "kory", "dayki_buf"}:
            continue
        gdf = load_layer(shp_path)
        gdf = to_crs_safe(gdf, mask_crs)
        geom_types = {str(x) for x in gdf.geom_type.unique()}
        if "Point" in geom_types or "MultiPoint" in geom_types:
            point_layers.append(gdf)
    if not point_layers:
        return None
    pts = pd.concat(point_layers, ignore_index=True)
    return gpd.GeoDataFrame(pts, geometry="geometry", crs=mask_crs)

def set_mask_extent(ax, mask):
    minx, miny, maxx, maxy = mask.total_bounds
    padx = (maxx - minx) * 0.02
    pady = (maxy - miny) * 0.02
    ax.set_xlim(minx - padx, maxx + padx)
    ax.set_ylim(miny - pady, maxy + pady)


def keep_large_components(grid, bool_col, shape, min_cells=4):
    try:
        from scipy import ndimage
    except Exception:
        return grid[bool_col].to_numpy().astype(bool)
    arr = np.zeros(shape, dtype=np.uint8)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    arr[rows, cols] = grid[bool_col].to_numpy().astype(np.uint8)
    structure = np.ones((3, 3), dtype=np.uint8)
    labeled, n = ndimage.label(arr, structure=structure)
    if n == 0:
        return grid[bool_col].to_numpy().astype(bool)
    sizes = np.bincount(labeled.ravel())
    keep = np.isin(labeled, np.where(sizes >= min_cells)[0])
    keep &= labeled > 0
    return keep[rows, cols]

def local_max_mask(grid, value_col, shape):
    try:
        from scipy.ndimage import maximum_filter
    except Exception:
        vals = grid[value_col].to_numpy()
        thr = np.nanquantile(vals, 0.98)
        return vals >= thr
    arr = np.full(shape, np.nan, dtype=float)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    vals = grid[value_col].to_numpy()
    arr[rows, cols] = vals
    filled = np.nan_to_num(arr, nan=-9999.0)
    locmax = maximum_filter(filled, size=3, mode="nearest")
    mask = np.isfinite(arr) & (filled >= locmax)
    return mask[rows, cols]

def make_display_classes(grid):
    prog_disp = robust_normalize_01(grid["prognoz"].to_numpy(), 0.02, 0.98)
    grid["display_score"] = prog_disp
    bins = np.linspace(0, 1, N_DISPLAY_CLASSES + 1)
    grid["display_class"] = np.digitize(prog_disp, bins[1:-1], right=False)
    return grid


def mark_gold_zones(grid, shape, mask_union):
    q_best = float(grid["prognoz"].quantile(TOP_GOLD_Q))
    q_local = float(grid["local_bonus"].quantile(TOP_LOCAL_Q))
    q_coinc = float(grid["coincidence_score"].quantile(TOP_LOCAL_Q))
    q_magm = float(grid["prox_magm"].quantile(0.84))
    q_tmagm = float(grid["tect_magm_intersection"].quantile(0.72))
    q_reg = float(grid["regression_score_sm"].quantile(0.88))

    mask_boundary = mask_union.boundary
    grid["dist_to_boundary"] = np.array([geom.distance(mask_boundary) for geom in grid.geometry])

    local_peak = local_max_mask(grid, "prospectivity", shape)

    seed_gold = (
        (grid["prognoz"] <= q_best) &
        (grid["regression_score_sm"] >= q_reg) &
        (
            (grid["local_bonus"] >= q_local) |
            (grid["coincidence_score"] >= q_coinc) |
            (
                (grid["prox_magm"] >= q_magm) &
                (grid["tect_magm_intersection"] >= q_tmagm)
            )
        )
    )

    edge_gold = (
        (grid["dist_to_boundary"] <= CELL_SIZE * 1.10) &
        (grid["prox_magm"] >= q_magm) &
        (grid["tect_magm_intersection"] >= q_tmagm) &
        (grid["regression_score_sm"] >= float(grid["regression_score_sm"].quantile(0.75)))
    )

    # мягкий контур лучших зон: seeds + локальные максимумы + краевые линейные зоны
    raw_gold = (
        (seed_gold & (local_peak | (grid["coincidence_score"] >= float(grid["coincidence_score"].quantile(0.88))))) |
        edge_gold
    )

    grid["gold_seed"] = raw_gold.astype(int)
    large = keep_large_components(grid, "gold_seed", shape, min_cells=4)
    grid["gold_zone"] = large.astype(int)
    return grid

def plot_prox(grid, mask, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    grid.plot(column="prox_magm", ax=ax, cmap="RdYlBu_r", linewidth=0, legend=True)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    set_mask_extent(ax, mask)
    ax.set_title("prox_magm")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_final(grid, mask, points, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr.N)
    grid.plot(column="display_class", ax=ax, cmap="bwr", norm=norm, linewidth=0, legend=True)
    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=ax, color="#f2d200", linewidth=0)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=ax, color="yellow", markersize=8, edgecolor="black", linewidth=0.25)
    set_mask_extent(ax, mask)
    ax.set_title("Итоговый прогноз")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_compare(grid, mask, points, out_png):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    grid.plot(column="prox_magm", ax=axes[0], cmap="RdYlBu_r", linewidth=0)
    mask.boundary.plot(ax=axes[0], color="black", linewidth=0.5)
    set_mask_extent(axes[0], mask)
    axes[0].set_title("prox_magm")
    axes[0].set_axis_off()

    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr.N)
    grid.plot(column="display_class", ax=axes[1], cmap="bwr", norm=norm, linewidth=0)
    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=axes[1], color="#f2d200", linewidth=0)
    mask.boundary.plot(ax=axes[1], color="black", linewidth=0.5)
    if points is not None and len(points) > 0:
        points.plot(ax=axes[1], color="yellow", markersize=8, edgecolor="black", linewidth=0.25)
    set_mask_extent(axes[1], mask)
    axes[1].set_title("Итоговый прогноз")
    axes[1].set_axis_off()

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

# =========================================================
# ЗАГРУЗКА
# =========================================================
aliases = prepare_ascii_aliases(SHP_DIR, SAFE_ALIAS_DIR)
mask = load_layer(aliases["svita_new"])
facies = to_crs_safe(load_layer(aliases["fasii"]), mask.crs)
tect1 = to_crs_safe(load_layer(aliases["glub_raz_nw"]), mask.crs)
tect2 = to_crs_safe(load_layer(aliases["glub_r_nw"]), mask.crs)
paleo = to_crs_safe(load_layer(aliases["gr_dol_vp_poly"]), mask.crs)
struct = to_crs_safe(load_layer(aliases["kory"]), mask.crs)
magm = to_crs_safe(load_layer(aliases["dayki_buf"]), mask.crs)
points = collect_points(mask.crs, aliases)

# =========================================================
# СЕТКА
# =========================================================
grid, mask_union, grid_shape = build_grid(mask, CELL_SIZE)

# =========================================================
# ДИСТАНЦИИ
# =========================================================
grid = add_distance_feature(grid, facies, "dist_facies")
grid = add_distance_feature(grid, paleo, "dist_paleo")
grid = add_distance_feature(grid, struct, "dist_struct")
grid = add_distance_feature(grid, magm, "dist_magm")
grid = add_distance_feature(grid, tect1, "dist_tect1")
grid = add_distance_feature(grid, tect2, "dist_tect2")

# =========================================================
# PROXIMITY
# =========================================================
grid["prox_facies"] = distance_to_proximity(grid["dist_facies"], transform="cbrt", q=Q_FACIES)
grid["prox_paleo"] = distance_to_proximity(grid["dist_paleo"], transform="cbrt", q=Q_PALEO)
grid["prox_struct"] = distance_to_proximity(grid["dist_struct"], transform="sqrt", q=Q_STRUCT)
grid["prox_magm"] = distance_to_proximity(grid["dist_magm"], transform="sqrt", q=Q_MAGM)
grid["prox_tect1"] = distance_to_proximity(grid["dist_tect1"], transform="cbrt", q=Q_TECT1)
grid["prox_tect2"] = distance_to_proximity(grid["dist_tect2"], transform="cbrt", q=Q_TECT2)

# =========================================================
# INTERACTIONS
# =========================================================
grid["tect_combo"] = 0.5 * (grid["prox_tect1"] + grid["prox_tect2"])
grid["tect_intersection"] = grid["prox_tect1"] * grid["prox_tect2"]
grid["tect_magm_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_magm"])
grid["tect_struct_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_struct"])
grid["paleo_struct_intersection"] = np.sqrt(grid["prox_paleo"] * grid["prox_struct"])

combo_core = (
    np.clip(grid["tect_combo"], 0, 1) *
    np.clip(0.55 * grid["prox_magm"] + 0.45 * grid["prox_struct"], 0, 1) *
    np.clip(0.60 * grid["prox_paleo"] + 0.40 * grid["prox_facies"], 0, 1)
)
grid["coincidence_score"] = robust_normalize_01(np.sqrt(np.clip(combo_core, 0, 1)), 0.02, 0.98)

tect_support = 0.45 * grid["prox_magm"] + 0.35 * grid["prox_struct"] + 0.20 * grid["prox_paleo"]
grid["tect_only_penalty"] = robust_normalize_01(np.clip(grid["tect_combo"] - tect_support, 0, 1), 0.02, 0.98)

# =========================================================
# GEO SCORE - ослабленный, как стабилизатор
# =========================================================
grid["geo_score_raw"] = (
    0.12 * grid["prox_tect1"] +
    0.12 * grid["prox_tect2"] +
    0.14 * grid["prox_paleo"] +
    0.11 * grid["prox_struct"] +
    0.08 * grid["prox_facies"] +
    0.08 * grid["prox_magm"] +
    0.09 * grid["tect_intersection"] +
    0.09 * grid["tect_magm_intersection"] +
    0.05 * grid["tect_struct_intersection"] +
    0.04 * grid["paleo_struct_intersection"] +
    0.08 * grid["coincidence_score"] -
    0.09 * grid["tect_only_penalty"]
)
grid["geo_score"] = robust_normalize_01(grid["geo_score_raw"], 0.02, 0.98)
grid["geo_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "geo_score", grid_shape, passes=2), 0.02, 0.98)

# =========================================================
# REGRESSION AS BASE
# =========================================================
grid["target"] = 0
grid["regression_score"] = grid["geo_score_sm"]
use_supervised = False
reg_test_auc_proxy = None

feature_cols = [
    "prox_facies", "prox_paleo", "prox_struct", "prox_magm",
    "prox_tect1", "prox_tect2", "tect_combo", "tect_intersection",
    "tect_magm_intersection", "tect_struct_intersection",
    "paleo_struct_intersection", "coincidence_score", "tect_only_penalty",
    "geo_score_sm"
]

if USE_SUPERVISED and points is not None and len(points) > 0:
    try:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", predicate="within")
    except Exception:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", op="within")

    positive_cells = joined["cell_id"].dropna().astype(int).unique().tolist()
    grid.loc[grid["cell_id"].isin(positive_cells), "target"] = 1

    pos = int(grid["target"].sum())
    neg = int((grid["target"] == 0).sum())

    if pos >= 20 and neg > pos:
        X = grid[feature_cols].fillna(0).to_numpy()
        y = grid["target"].to_numpy()
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # Небольшая проверка на holdout, потом учим на всей выборке
        X_train, X_test, y_train, y_test = train_test_split(
            X_scaled, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
        )
        lr_eval = LogisticRegression(
            random_state=RANDOM_STATE,
            max_iter=4000,
            class_weight="balanced",
            C=0.8
        )
        lr_eval.fit(X_train, y_train)
        test_prob = lr_eval.predict_proba(X_test)[:, 1]

        # proxy-метрика без дополнительных библиотек
        y_test = np.asarray(y_test)
        pos_mean = float(np.mean(test_prob[y_test == 1])) if np.any(y_test == 1) else np.nan
        neg_mean = float(np.mean(test_prob[y_test == 0])) if np.any(y_test == 0) else np.nan
        reg_test_auc_proxy = pos_mean - neg_mean

        lr = LogisticRegression(
            random_state=RANDOM_STATE,
            max_iter=4000,
            class_weight="balanced",
            C=0.8
        )
        lr.fit(X_scaled, y)
        grid["regression_score"] = robust_normalize_01(lr.predict_proba(X_scaled)[:, 1], 0.02, 0.98)
        use_supervised = True

grid["regression_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "regression_score", grid_shape, passes=2), 0.02, 0.98)
grid["ml_score"] = grid["regression_score_sm"]

# =========================================================
# SOM + KMEANS
# =========================================================
X = grid[feature_cols].fillna(0).to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

som = MiniSom(
    x=SOM_X, y=SOM_Y, input_len=X_scaled.shape[1],
    sigma=1.1, learning_rate=0.38, random_seed=RANDOM_STATE
)
som.random_weights_init(X_scaled)
som.train_random(X_scaled, SOM_ITERS)

winners = np.array([som.winner(x) for x in X_scaled])
grid["som_x"] = winners[:, 0]
grid["som_y"] = winners[:, 1]
grid["som_node"] = grid["som_x"].astype(str) + "_" + grid["som_y"].astype(str)

som_weights = som.get_weights().reshape(SOM_X * SOM_Y, X_scaled.shape[1])
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=20)
neuron_cluster = kmeans.fit_predict(som_weights)

node_to_cluster = {}
idx = 0
for i in range(SOM_X):
    for j in range(SOM_Y):
        node_to_cluster[f"{i}_{j}"] = int(neuron_cluster[idx])
        idx += 1
grid["cluster"] = grid["som_node"].map(node_to_cluster).astype(int)

cluster_geo = grid.groupby("cluster")["geo_score_sm"].mean().reset_index(name="cluster_geo_mean")
cluster_reg = grid.groupby("cluster")["regression_score"].mean().reset_index(name="cluster_reg_mean")
cluster_coinc = grid.groupby("cluster")["coincidence_score"].mean().reset_index(name="cluster_coinc_mean")
cluster_stats = cluster_geo.merge(cluster_reg, on="cluster", how="outer").merge(cluster_coinc, on="cluster", how="outer")

if use_supervised:
    cluster_hits = grid.groupby("cluster").agg(cells=("cell_id", "count"), positives=("target", "sum")).reset_index()
    cluster_hits["hit_rate"] = cluster_hits["positives"] / cluster_hits["cells"]
    cluster_stats = cluster_stats.merge(cluster_hits, on="cluster", how="left")
    cluster_stats["hit_rate"] = cluster_stats["hit_rate"].fillna(0)
    cluster_stats["cluster_score"] = robust_normalize_01(
        0.35 * cluster_stats["cluster_geo_mean"] +
        0.35 * cluster_stats["cluster_reg_mean"] +
        0.15 * cluster_stats["cluster_coinc_mean"] +
        0.15 * robust_normalize_01(cluster_stats["hit_rate"], 0.0, 1.0),
        0.02, 0.98,
    )
else:
    cluster_stats["cluster_score"] = robust_normalize_01(
        0.45 * cluster_stats["cluster_geo_mean"] +
        0.30 * cluster_stats["cluster_reg_mean"] +
        0.25 * cluster_stats["cluster_coinc_mean"],
        0.02, 0.98,
    )

grid = grid.merge(cluster_stats[["cluster", "cluster_score"]], on="cluster", how="left")
grid["cluster_score"] = grid["cluster_score"].fillna(grid["geo_score_sm"])

# =========================================================
# ИТОГ
# =========================================================
grid["local_bonus"] = robust_normalize_01(
    0.38 * grid["tect_intersection"] +
    0.37 * grid["tect_magm_intersection"] +
    0.25 * grid["tect_struct_intersection"],
    0.02, 0.98,
)

grid["prospectivity_raw"] = (
    W_REG * grid["regression_score_sm"] +
    W_GEO * grid["geo_score_sm"] +
    W_COINCIDENCE * grid["coincidence_score"] +
    W_CLUSTER * grid["cluster_score"] +
    W_LOCAL * grid["local_bonus"]
)

# Дополнительное слабое ослабление tectonic-only после сборки
grid["prospectivity_raw"] = grid["prospectivity_raw"] - 0.06 * grid["tect_only_penalty"]

grid["prospectivity"] = robust_normalize_01(grid["prospectivity_raw"], 0.02, 0.98)

# в логике презентации: меньше = лучше
grid["prognoz"] = 1.0 - grid["prospectivity"]

top_thr = float(grid["prospectivity"].quantile(0.90))
grid["top10"] = (grid["prospectivity"] >= top_thr).astype(int)

try:
    grid["prospect_class"] = pd.qcut(
        grid["prospectivity"],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop"
    )
except Exception:
    grid["prospect_class"] = "medium"

grid = make_display_classes(grid)
grid = mark_gold_zones(grid, grid_shape, mask_union)

# =========================================================
# СОХРАНЕНИЕ
# =========================================================
if OUT_GPKG.exists():
    OUT_GPKG.unlink()

grid.to_file(OUT_GPKG, layer="forecast_grid", driver="GPKG")
if points is not None and len(points) > 0:
    points.to_file(OUT_GPKG, layer="evidence_points", driver="GPKG")

grid.drop(columns="geometry").to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

plot_prox(grid, mask, OUT_PROX)
plot_final(grid, mask, points, OUT_PNG)
plot_compare(grid, mask, points, OUT_COMPARE)

metrics = {
    "base_dir": str(BASE_DIR),
    "grid_cells": int(len(grid)),
    "cell_size": CELL_SIZE,
    "use_supervised_requested": bool(USE_SUPERVISED),
    "use_supervised_applied": bool(use_supervised),
    "positive_cells": int(grid["target"].sum()),
    "top10_threshold": float(top_thr),
    "prospectivity_min": float(np.nanmin(grid["prospectivity"])),
    "prospectivity_p05": float(np.nanquantile(grid["prospectivity"], 0.05)),
    "prospectivity_p50": float(np.nanquantile(grid["prospectivity"], 0.50)),
    "prospectivity_p95": float(np.nanquantile(grid["prospectivity"], 0.95)),
    "prospectivity_max": float(np.nanmax(grid["prospectivity"])),
    "prognoz_min": float(np.nanmin(grid["prognoz"])),
    "prognoz_max": float(np.nanmax(grid["prognoz"])),
    "display_score_min": float(np.nanmin(grid["display_score"])),
    "display_score_max": float(np.nanmax(grid["display_score"])),
    "regression_score_min": float(np.nanmin(grid["regression_score"])),
    "regression_score_max": float(np.nanmax(grid["regression_score"])),
    "regression_score_sm_min": float(np.nanmin(grid["regression_score_sm"])),
    "regression_score_sm_max": float(np.nanmax(grid["regression_score_sm"])),
    "reg_test_auc_proxy": None if reg_test_auc_proxy is None else float(reg_test_auc_proxy),
    "gold_zone_count": int(grid["gold_zone"].sum()),
    "point_count": int(len(points)) if points is not None else 0,
}
Path(OUT_JSON).write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

print("Готово.")
print(f"BASE_DIR: {BASE_DIR}")
print(f"PNG: {OUT_PNG}")
print(f"COMPARE: {OUT_COMPARE}")
print(f"GPKG: {OUT_GPKG}")
print(f"CSV: {OUT_CSV}")
print(f"JSON: {OUT_JSON}")
print("Диагностика:")
print(grid[["prospectivity", "prognoz", "display_score", "regression_score", "regression_score_sm", "gold_zone"]].describe())

Готово.
BASE_DIR: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз
PNG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v5_regression_base\gold_forecast_same_methods_fixed_v5.png
COMPARE: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v5_regression_base\compare_same_methods_fixed_v5.png
GPKG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v5_regression_base\gold_forecast_same_methods_fixed_v5.gpkg
CSV: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v5_regression_base\grid_attributes_same_methods_fixed_v5.csv
JSON: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v5_regression_base\metrics_same_methods_fixed_v5.json
Диагностика:
       prospectivity       prognoz  display_score  regression_score  \
count   15684.000000  15684.000000   15684.000000      15684.000000   
mean        0.565030      0.434970       0.434959          0.533817   
std         0.252496      0.252496       0.252510          0.

In [9]:

import os
import re
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm

from pyproj import CRS
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from minisom import MiniSom

warnings.filterwarnings("ignore")

# =========================================================
# НАСТРОЙКИ
# =========================================================
CELL_SIZE = 500
RANDOM_STATE = 42

# методы сохраняем
SOM_X = 12
SOM_Y = 12
SOM_ITERS = 4000
N_CLUSTERS = 6
USE_SUPERVISED = True

# регрессия становится основой,
# но геологический блок и кластеры остаются как стабилизаторы
W_REG = 0.40
W_GEO = 0.22
W_EVID = 0.20
W_COINCIDENCE = 0.10
W_CLUSTER = 0.04
W_LOCAL = 0.04
SHOW_POINTS = False

# proximity
Q_FACIES = 0.78
Q_PALEO = 0.76
Q_STRUCT = 0.72
Q_MAGM = 0.42
Q_TECT1 = 0.74
Q_TECT2 = 0.74

# визуализация и gold-зоны
N_DISPLAY_CLASSES = 20
TOP_GOLD_Q = 0.035
TOP_LOCAL_Q = 0.96

# =========================================================
# ПУТИ
# =========================================================
def find_existing_base_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path("/mnt/data/prog_zip"),
        Path("/mnt/data"),
        Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз"),
    ]
    for base in candidates:
        shp_dir = base / "shp_dbf"
        if shp_dir.exists() and (shp_dir / "svita_new.shp").exists():
            return base
    raise FileNotFoundError("Не найден каталог с shp_dbf. Укажи BASE_DIR вручную.")

BASE_DIR = find_existing_base_dir()
SHP_DIR = BASE_DIR / "shp_dbf"
OUT_DIR = BASE_DIR / "same_methods_fixed_v6_regression_base"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAFE_ALIAS_DIR = OUT_DIR / "_safe_shp_aliases"
SAFE_ALIAS_DIR.mkdir(parents=True, exist_ok=True)

OUT_GPKG = OUT_DIR / "gold_forecast_same_methods_fixed_v6.gpkg"
OUT_PNG = OUT_DIR / "gold_forecast_same_methods_fixed_v6.png"
OUT_PROX = OUT_DIR / "prox_magm_same_methods_fixed_v6.png"
OUT_COMPARE = OUT_DIR / "compare_same_methods_fixed_v6.png"
OUT_CSV = OUT_DIR / "grid_attributes_same_methods_fixed_v6.csv"
OUT_JSON = OUT_DIR / "metrics_same_methods_fixed_v6.json"

# =========================================================
# ВСПОМОГАТЕЛЬНЫЕ
# =========================================================
def normalize_01(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    mn = np.nanmin(arr[finite])
    mx = np.nanmax(arr[finite])
    if np.isclose(mx, mn):
        return np.full_like(arr, 0.5, dtype=float)
    out = np.full_like(arr, np.nan, dtype=float)
    out[finite] = (arr[finite] - mn) / (mx - mn)
    return out

def robust_normalize_01(values, q_low=0.03, q_high=0.97):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    lo = np.nanquantile(arr[finite], q_low)
    hi = np.nanquantile(arr[finite], q_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or np.isclose(lo, hi):
        return normalize_01(arr)
    out = (arr - lo) / (hi - lo)
    return np.clip(out, 0, 1)

def read_sidecar_proj4(shp_path: Path):
    sidecar = shp_path.with_name(shp_path.stem + "_shp.pj4")
    if sidecar.exists():
        txt = sidecar.read_text(encoding="utf-8", errors="ignore")
        m = re.search(r"pj4=(.+)", txt)
        if m:
            return m.group(1).strip()
    return None

def prepare_ascii_aliases(shp_dir: Path, alias_dir: Path):
    aliases = {}
    stems = {}
    for name_b in os.listdir(os.fsencode(shp_dir)):
        if not name_b.endswith((b".shp", b".shx", b".dbf", b".prj", b".pj4")):
            continue
        if name_b.endswith(b"_shp.pj4"):
            continue
        base_b, ext_b = os.path.splitext(name_b)
        stems.setdefault(base_b, set()).add(ext_b)

    alias_idx = 0
    for base_b, exts in sorted(stems.items()):
        try:
            base_s = os.fsdecode(base_b)
            safe = all(ord(ch) < 128 and (ch.isalnum() or ch in "_-. ") for ch in base_s)
        except Exception:
            base_s = None
            safe = False

        if safe:
            aliases[base_s] = shp_dir / f"{base_s}.shp"
            continue

        alias_name = f"evidence_{alias_idx:02d}"
        alias_idx += 1
        for ext_b in exts:
            src = os.path.join(os.fsencode(shp_dir), base_b + ext_b)
            dst = alias_dir / f"{alias_name}{os.fsdecode(ext_b)}"
            shutil.copyfile(src, dst)
        pj4_src = os.path.join(os.fsencode(shp_dir), base_b + b"_shp.pj4")
        if os.path.exists(pj4_src):
            shutil.copyfile(pj4_src, alias_dir / f"{alias_name}_shp.pj4")
        aliases[alias_name] = alias_dir / f"{alias_name}.shp"
    return aliases

def load_layer(shp_path: Path):
    gdf = gpd.read_file(shp_path)
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    if gdf.crs is None:
        proj4 = read_sidecar_proj4(shp_path)
        if proj4:
            gdf = gdf.set_crs(CRS.from_proj4(proj4), allow_override=True)
    return gdf

def to_crs_safe(gdf, target_crs):
    if gdf.crs is None and target_crs is not None:
        return gdf.set_crs(target_crs, allow_override=True)
    if target_crs is None or gdf.crs == target_crs:
        return gdf
    return gdf.to_crs(target_crs)

def build_grid(mask, cell_size):
    mask_union = unary_union(mask.geometry)
    prepared_mask = prep(mask_union)
    minx, miny, maxx, maxy = mask.total_bounds
    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)
    rows = []
    cell_id = 0
    for r, y in enumerate(ys):
        for c, x in enumerate(xs):
            geom = box(x, y, x + cell_size, y + cell_size)
            if prepared_mask.intersects(geom):
                rows.append((cell_id, r, c, geom))
                cell_id += 1
    grid = gpd.GeoDataFrame(rows, columns=["cell_id", "row", "col", "geometry"], geometry="geometry", crs=mask.crs)
    return grid, mask_union, (len(ys), len(xs))

def add_distance_feature(grid, source, name):
    source_union = unary_union(source.geometry)
    distances = np.empty(len(grid), dtype=float)
    for i, geom in enumerate(grid.geometry.values):
        distances[i] = 0.0 if geom.intersects(source_union) else geom.distance(source_union)
    grid[name] = distances
    return grid

def distance_to_proximity(distance, transform="sqrt", q=0.75):
    d = np.asarray(distance, dtype=float)
    d = np.clip(d, 0, None)
    if transform == "sqrt":
        dt = np.sqrt(d)
    elif transform == "cbrt":
        dt = np.cbrt(d)
    elif transform == "log1p":
        dt = np.log1p(d)
    else:
        dt = d
    scale = float(np.nanquantile(dt, q))
    if not np.isfinite(scale) or scale <= 0:
        scale = max(float(np.nanmean(dt)), 1.0)
    prox = np.exp(-dt / scale)
    return np.clip(prox, 0, 1)

def smooth_on_regular_grid(grid, value_col, shape, passes=1):
    try:
        from scipy.signal import convolve2d
    except Exception:
        return grid[value_col].to_numpy()
    arr = np.full(shape, np.nan, dtype=float)
    arr[grid["row"].to_numpy(), grid["col"].to_numpy()] = grid[value_col].to_numpy()
    kernel = np.array([[1.0, 1.2, 1.0], [1.2, 3.0, 1.2], [1.0, 1.2, 1.0]], dtype=float)
    smoothed = arr.copy()
    for _ in range(max(1, passes)):
        valid = np.isfinite(smoothed).astype(float)
        filled = np.nan_to_num(smoothed, nan=0.0)
        num = convolve2d(filled, kernel, mode="same", boundary="fill", fillvalue=0)
        den = convolve2d(valid, kernel, mode="same", boundary="fill", fillvalue=0)
        with np.errstate(divide="ignore", invalid="ignore"):
            smoothed = np.where(den > 0, num / den, np.nan)
    return smoothed[grid["row"].to_numpy(), grid["col"].to_numpy()]

def collect_points(mask_crs, aliases):
    point_layers = []
    for name, shp_path in aliases.items():
        if name in {"svita_new", "fasii", "glub_raz_nw", "glub_r_nw", "gr_dol_vp_poly", "kory", "dayki_buf"}:
            continue
        gdf = load_layer(shp_path)
        gdf = to_crs_safe(gdf, mask_crs)
        geom_types = {str(x) for x in gdf.geom_type.unique()}
        if "Point" in geom_types or "MultiPoint" in geom_types:
            point_layers.append(gdf)
    if not point_layers:
        return None
    pts = pd.concat(point_layers, ignore_index=True)
    return gpd.GeoDataFrame(pts, geometry="geometry", crs=mask_crs)


def add_point_support(grid, points, shape):
    # weak spatial guidance from known mineralization/evidence points:
    # helps suppress distant false positives and keeps regression as main block.
    if points is None or len(points) == 0:
        grid["evidence_support"] = 0.5
        grid["evidence_support_sm"] = 0.5
        return grid
    pts_union = unary_union(points.geometry)
    d = np.empty(len(grid), dtype=float)
    for i, geom in enumerate(grid.geometry.values):
        d[i] = 0.0 if geom.intersects(pts_union) else geom.distance(pts_union)
    grid["dist_points"] = d
    # relatively local support
    grid["evidence_support"] = distance_to_proximity(grid["dist_points"], transform="sqrt", q=0.35)
    grid["evidence_support_sm"] = robust_normalize_01(
        smooth_on_regular_grid(grid, "evidence_support", shape, passes=3), 0.02, 0.98
    )
    return grid

def set_mask_extent(ax, mask):
    minx, miny, maxx, maxy = mask.total_bounds
    padx = (maxx - minx) * 0.02
    pady = (maxy - miny) * 0.02
    ax.set_xlim(minx - padx, maxx + padx)
    ax.set_ylim(miny - pady, maxy + pady)


def keep_large_components(grid, bool_col, shape, min_cells=4):
    try:
        from scipy import ndimage
    except Exception:
        return grid[bool_col].to_numpy().astype(bool)
    arr = np.zeros(shape, dtype=np.uint8)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    arr[rows, cols] = grid[bool_col].to_numpy().astype(np.uint8)
    structure = np.ones((3, 3), dtype=np.uint8)
    labeled, n = ndimage.label(arr, structure=structure)
    if n == 0:
        return grid[bool_col].to_numpy().astype(bool)
    sizes = np.bincount(labeled.ravel())
    keep = np.isin(labeled, np.where(sizes >= min_cells)[0])
    keep &= labeled > 0
    return keep[rows, cols]

def local_max_mask(grid, value_col, shape):
    try:
        from scipy.ndimage import maximum_filter
    except Exception:
        vals = grid[value_col].to_numpy()
        thr = np.nanquantile(vals, 0.98)
        return vals >= thr
    arr = np.full(shape, np.nan, dtype=float)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    vals = grid[value_col].to_numpy()
    arr[rows, cols] = vals
    filled = np.nan_to_num(arr, nan=-9999.0)
    locmax = maximum_filter(filled, size=3, mode="nearest")
    mask = np.isfinite(arr) & (filled >= locmax)
    return mask[rows, cols]

def make_display_classes(grid):
    prog_disp = robust_normalize_01(grid["prognoz"].to_numpy(), 0.02, 0.98)
    grid["display_score"] = prog_disp
    bins = np.linspace(0, 1, N_DISPLAY_CLASSES + 1)
    grid["display_class"] = np.digitize(prog_disp, bins[1:-1], right=False)
    return grid



def mark_gold_zones(grid, shape, mask_union):
    q_best = float(grid["prognoz"].quantile(TOP_GOLD_Q))
    q_local = float(grid["local_bonus"].quantile(TOP_LOCAL_Q))
    q_coinc = float(grid["coincidence_score"].quantile(TOP_LOCAL_Q))
    q_magm = float(grid["prox_magm"].quantile(0.88))
    q_tmagm = float(grid["tect_magm_intersection"].quantile(0.80))
    q_reg = float(grid["regression_score_sm"].quantile(0.92))
    q_evid = float(grid["evidence_support_sm"].quantile(0.72))

    local_peak = local_max_mask(grid, "prospectivity", shape)

    seed_gold = (
        (grid["prognoz"] <= q_best) &
        (grid["regression_score_sm"] >= q_reg) &
        (grid["evidence_support_sm"] >= q_evid) &
        (
            ((grid["local_bonus"] >= q_local) & (grid["tect_magm_intersection"] >= q_tmagm)) |
            ((grid["coincidence_score"] >= q_coinc) & (grid["prox_magm"] >= q_magm))
        )
    )

    # Edge rule is retained, but much stricter than before.
    edge_gold = (
        (grid["dist_to_boundary"] <= CELL_SIZE * 0.90) &
        (grid["prox_magm"] >= q_magm) &
        (grid["tect_magm_intersection"] >= q_tmagm) &
        (grid["evidence_support_sm"] >= float(grid["evidence_support_sm"].quantile(0.65))) &
        (grid["regression_score_sm"] >= float(grid["regression_score_sm"].quantile(0.82)))
    )

    raw_gold = ((seed_gold & local_peak) | edge_gold)
    grid["gold_seed"] = raw_gold.astype(int)
    large = keep_large_components(grid, "gold_seed", shape, min_cells=5)
    grid["gold_zone"] = large.astype(int)
    return grid

def plot_prox(grid, mask, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    grid.plot(column="prox_magm", ax=ax, cmap="RdYlBu_r", linewidth=0, legend=True)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    set_mask_extent(ax, mask)
    ax.set_title("prox_magm")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_final(grid, mask, points, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr.N)
    grid.plot(column="display_class", ax=ax, cmap="bwr", norm=norm, linewidth=0, legend=True)
    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=ax, color="#f2d200", linewidth=0)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=ax, color="yellow", markersize=8, edgecolor="black", linewidth=0.25)
    set_mask_extent(ax, mask)
    ax.set_title("Итоговый прогноз")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_compare(grid, mask, points, out_png):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    grid.plot(column="prox_magm", ax=axes[0], cmap="RdYlBu_r", linewidth=0)
    mask.boundary.plot(ax=axes[0], color="black", linewidth=0.5)
    set_mask_extent(axes[0], mask)
    axes[0].set_title("prox_magm")
    axes[0].set_axis_off()

    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr.N)
    grid.plot(column="display_class", ax=axes[1], cmap="bwr", norm=norm, linewidth=0)
    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=axes[1], color="#f2d200", linewidth=0)
    mask.boundary.plot(ax=axes[1], color="black", linewidth=0.5)
    if points is not None and len(points) > 0:
        points.plot(ax=axes[1], color="yellow", markersize=8, edgecolor="black", linewidth=0.25)
    set_mask_extent(axes[1], mask)
    axes[1].set_title("Итоговый прогноз")
    axes[1].set_axis_off()

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

# =========================================================
# ЗАГРУЗКА
# =========================================================
aliases = prepare_ascii_aliases(SHP_DIR, SAFE_ALIAS_DIR)
mask = load_layer(aliases["svita_new"])
facies = to_crs_safe(load_layer(aliases["fasii"]), mask.crs)
tect1 = to_crs_safe(load_layer(aliases["glub_raz_nw"]), mask.crs)
tect2 = to_crs_safe(load_layer(aliases["glub_r_nw"]), mask.crs)
paleo = to_crs_safe(load_layer(aliases["gr_dol_vp_poly"]), mask.crs)
struct = to_crs_safe(load_layer(aliases["kory"]), mask.crs)
magm = to_crs_safe(load_layer(aliases["dayki_buf"]), mask.crs)
points = collect_points(mask.crs, aliases)

# =========================================================
# СЕТКА
# =========================================================
grid, mask_union, grid_shape = build_grid(mask, CELL_SIZE)

# =========================================================
# ДИСТАНЦИИ
# =========================================================
grid = add_distance_feature(grid, facies, "dist_facies")
grid = add_distance_feature(grid, paleo, "dist_paleo")
grid = add_distance_feature(grid, struct, "dist_struct")
grid = add_distance_feature(grid, magm, "dist_magm")
grid = add_distance_feature(grid, tect1, "dist_tect1")
grid = add_distance_feature(grid, tect2, "dist_tect2")

# =========================================================
# PROXIMITY
# =========================================================
grid["prox_facies"] = distance_to_proximity(grid["dist_facies"], transform="cbrt", q=Q_FACIES)
grid["prox_paleo"] = distance_to_proximity(grid["dist_paleo"], transform="cbrt", q=Q_PALEO)
grid["prox_struct"] = distance_to_proximity(grid["dist_struct"], transform="sqrt", q=Q_STRUCT)
grid["prox_magm"] = distance_to_proximity(grid["dist_magm"], transform="sqrt", q=Q_MAGM)
grid["prox_tect1"] = distance_to_proximity(grid["dist_tect1"], transform="cbrt", q=Q_TECT1)
grid["prox_tect2"] = distance_to_proximity(grid["dist_tect2"], transform="cbrt", q=Q_TECT2)

# =========================================================
# SUPPORT FROM EVIDENCE POINTS
# =========================================================
grid = add_point_support(grid, points, grid_shape)

# =========================================================
# INTERACTIONS
# =========================================================
grid["tect_combo"] = 0.5 * (grid["prox_tect1"] + grid["prox_tect2"])
grid["tect_intersection"] = grid["prox_tect1"] * grid["prox_tect2"]
grid["tect_magm_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_magm"])
grid["tect_struct_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_struct"])
grid["paleo_struct_intersection"] = np.sqrt(grid["prox_paleo"] * grid["prox_struct"])

combo_core = (
    np.clip(grid["tect_combo"], 0, 1) *
    np.clip(0.55 * grid["prox_magm"] + 0.45 * grid["prox_struct"], 0, 1) *
    np.clip(0.60 * grid["prox_paleo"] + 0.40 * grid["prox_facies"], 0, 1)
)
grid["coincidence_score"] = robust_normalize_01(np.sqrt(np.clip(combo_core, 0, 1)), 0.02, 0.98)

tect_support = 0.40 * grid["prox_magm"] + 0.30 * grid["prox_struct"] + 0.20 * grid["prox_paleo"] + 0.10 * grid["evidence_support_sm"]
grid["tect_only_penalty"] = robust_normalize_01(np.clip(grid["tect_combo"] - tect_support, 0, 1), 0.02, 0.98)

# =========================================================
# GEO SCORE - ослабленный, как стабилизатор
# =========================================================
grid["geo_score_raw"] = (
    0.12 * grid["prox_tect1"] +
    0.12 * grid["prox_tect2"] +
    0.14 * grid["prox_paleo"] +
    0.11 * grid["prox_struct"] +
    0.08 * grid["prox_facies"] +
    0.08 * grid["prox_magm"] +
    0.09 * grid["tect_intersection"] +
    0.09 * grid["tect_magm_intersection"] +
    0.05 * grid["tect_struct_intersection"] +
    0.04 * grid["paleo_struct_intersection"] +
    0.08 * grid["coincidence_score"] -
    0.09 * grid["tect_only_penalty"]
)
grid["geo_score"] = robust_normalize_01(grid["geo_score_raw"], 0.02, 0.98)
grid["geo_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "geo_score", grid_shape, passes=2), 0.02, 0.98)

# =========================================================
# REGRESSION AS BASE
# =========================================================
grid["target"] = 0
grid["regression_score"] = grid["geo_score_sm"]
use_supervised = False
reg_test_auc_proxy = None

feature_cols = [
    "prox_facies", "prox_paleo", "prox_struct", "prox_magm",
    "prox_tect1", "prox_tect2", "tect_combo", "tect_intersection",
    "tect_magm_intersection", "tect_struct_intersection",
    "paleo_struct_intersection", "coincidence_score", "tect_only_penalty",
    "evidence_support_sm", "geo_score_sm"
]

if USE_SUPERVISED and points is not None and len(points) > 0:
    try:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", predicate="within")
    except Exception:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", op="within")

    positive_cells = joined["cell_id"].dropna().astype(int).unique().tolist()
    grid.loc[grid["cell_id"].isin(positive_cells), "target"] = 1

    pos = int(grid["target"].sum())
    neg = int((grid["target"] == 0).sum())

    if pos >= 20 and neg > pos:
        X = grid[feature_cols].fillna(0).to_numpy()
        y = grid["target"].to_numpy()
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # Небольшая проверка на holdout, потом учим на всей выборке
        X_train, X_test, y_train, y_test = train_test_split(
            X_scaled, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
        )
        lr_eval = LogisticRegression(
            random_state=RANDOM_STATE,
            max_iter=4000,
            class_weight="balanced",
            C=0.8
        )
        lr_eval.fit(X_train, y_train)
        test_prob = lr_eval.predict_proba(X_test)[:, 1]

        # proxy-метрика без дополнительных библиотек
        y_test = np.asarray(y_test)
        pos_mean = float(np.mean(test_prob[y_test == 1])) if np.any(y_test == 1) else np.nan
        neg_mean = float(np.mean(test_prob[y_test == 0])) if np.any(y_test == 0) else np.nan
        reg_test_auc_proxy = pos_mean - neg_mean

        lr = LogisticRegression(
            random_state=RANDOM_STATE,
            max_iter=4000,
            class_weight="balanced",
            C=0.8
        )
        lr.fit(X_scaled, y)
        grid["regression_score"] = robust_normalize_01(lr.predict_proba(X_scaled)[:, 1], 0.02, 0.98)
        use_supervised = True

grid["regression_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "regression_score", grid_shape, passes=2), 0.02, 0.98)
grid["ml_score"] = grid["regression_score_sm"]

# =========================================================
# SOM + KMEANS
# =========================================================
X = grid[feature_cols].fillna(0).to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

som = MiniSom(
    x=SOM_X, y=SOM_Y, input_len=X_scaled.shape[1],
    sigma=1.1, learning_rate=0.38, random_seed=RANDOM_STATE
)
som.random_weights_init(X_scaled)
som.train_random(X_scaled, SOM_ITERS)

winners = np.array([som.winner(x) for x in X_scaled])
grid["som_x"] = winners[:, 0]
grid["som_y"] = winners[:, 1]
grid["som_node"] = grid["som_x"].astype(str) + "_" + grid["som_y"].astype(str)

som_weights = som.get_weights().reshape(SOM_X * SOM_Y, X_scaled.shape[1])
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=20)
neuron_cluster = kmeans.fit_predict(som_weights)

node_to_cluster = {}
idx = 0
for i in range(SOM_X):
    for j in range(SOM_Y):
        node_to_cluster[f"{i}_{j}"] = int(neuron_cluster[idx])
        idx += 1
grid["cluster"] = grid["som_node"].map(node_to_cluster).astype(int)

cluster_geo = grid.groupby("cluster")["geo_score_sm"].mean().reset_index(name="cluster_geo_mean")
cluster_reg = grid.groupby("cluster")["regression_score"].mean().reset_index(name="cluster_reg_mean")
cluster_coinc = grid.groupby("cluster")["coincidence_score"].mean().reset_index(name="cluster_coinc_mean")
cluster_stats = cluster_geo.merge(cluster_reg, on="cluster", how="outer").merge(cluster_coinc, on="cluster", how="outer")

if use_supervised:
    cluster_hits = grid.groupby("cluster").agg(cells=("cell_id", "count"), positives=("target", "sum")).reset_index()
    cluster_hits["hit_rate"] = cluster_hits["positives"] / cluster_hits["cells"]
    cluster_stats = cluster_stats.merge(cluster_hits, on="cluster", how="left")
    cluster_stats["hit_rate"] = cluster_stats["hit_rate"].fillna(0)
    cluster_stats["cluster_score"] = robust_normalize_01(
        0.30 * cluster_stats["cluster_geo_mean"] +
        0.35 * cluster_stats["cluster_reg_mean"] +
        0.15 * cluster_stats["cluster_coinc_mean"] +
        0.20 * robust_normalize_01(cluster_stats["hit_rate"], 0.0, 1.0),
        0.02, 0.98,
    )
else:
    cluster_stats["cluster_score"] = robust_normalize_01(
        0.40 * cluster_stats["cluster_geo_mean"] +
        0.25 * cluster_stats["cluster_reg_mean"] +
        0.20 * cluster_stats["cluster_coinc_mean"] +
        0.15 * grid["evidence_support_sm"].mean(),
        0.02, 0.98,
    )

grid = grid.merge(cluster_stats[["cluster", "cluster_score"]], on="cluster", how="left")
grid["cluster_score"] = grid["cluster_score"].fillna(grid["geo_score_sm"])

# =========================================================
# ИТОГ
# =========================================================
grid["local_bonus"] = robust_normalize_01(
    0.38 * grid["tect_intersection"] +
    0.37 * grid["tect_magm_intersection"] +
    0.25 * grid["tect_struct_intersection"],
    0.02, 0.98,
)

grid["prospectivity_raw"] = (
    W_REG * grid["regression_score_sm"] +
    W_GEO * grid["geo_score_sm"] +
    W_EVID * grid["evidence_support_sm"] +
    W_COINCIDENCE * grid["coincidence_score"] +
    W_CLUSTER * grid["cluster_score"] +
    W_LOCAL * grid["local_bonus"]
)

# Дальняя от известных проявлений и краевая тектоника часто дает ложноположительные зоны.
mask_boundary = mask_union.boundary
grid["dist_to_boundary"] = np.array([geom.distance(mask_boundary) for geom in grid.geometry])
edge_near = np.exp(-grid["dist_to_boundary"].to_numpy() / (CELL_SIZE * 2.2))
grid["edge_false_penalty"] = robust_normalize_01(
    edge_near * np.clip(1.0 - grid["evidence_support_sm"], 0, 1) * np.clip(grid["tect_only_penalty"], 0, 1),
    0.02, 0.98
)

grid["prospectivity_raw"] = (
    grid["prospectivity_raw"]
    - 0.08 * grid["tect_only_penalty"]
    - 0.10 * grid["edge_false_penalty"]
)

grid["prospectivity_raw_sm"] = smooth_on_regular_grid(grid, "prospectivity_raw", grid_shape, passes=2)
grid["prospectivity"] = robust_normalize_01(grid["prospectivity_raw_sm"], 0.02, 0.98)

# в логике презентации: меньше = лучше
grid["prognoz"] = 1.0 - grid["prospectivity"]

top_thr = float(grid["prospectivity"].quantile(0.90))
grid["top10"] = (grid["prospectivity"] >= top_thr).astype(int)

try:
    grid["prospect_class"] = pd.qcut(
        grid["prospectivity"],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop"
    )
except Exception:
    grid["prospect_class"] = "medium"

grid = make_display_classes(grid)
grid = mark_gold_zones(grid, grid_shape, mask_union)

# =========================================================
# СОХРАНЕНИЕ
# =========================================================
if OUT_GPKG.exists():
    OUT_GPKG.unlink()

grid.to_file(OUT_GPKG, layer="forecast_grid", driver="GPKG")
if points is not None and len(points) > 0:
    points.to_file(OUT_GPKG, layer="evidence_points", driver="GPKG")

grid.drop(columns="geometry").to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

plot_prox(grid, mask, OUT_PROX)
plot_final(grid, mask, points, OUT_PNG)
plot_compare(grid, mask, points, OUT_COMPARE)

metrics = {
    "base_dir": str(BASE_DIR),
    "grid_cells": int(len(grid)),
    "cell_size": CELL_SIZE,
    "use_supervised_requested": bool(USE_SUPERVISED),
    "use_supervised_applied": bool(use_supervised),
    "positive_cells": int(grid["target"].sum()),
    "top10_threshold": float(top_thr),
    "prospectivity_min": float(np.nanmin(grid["prospectivity"])),
    "prospectivity_p05": float(np.nanquantile(grid["prospectivity"], 0.05)),
    "prospectivity_p50": float(np.nanquantile(grid["prospectivity"], 0.50)),
    "prospectivity_p95": float(np.nanquantile(grid["prospectivity"], 0.95)),
    "prospectivity_max": float(np.nanmax(grid["prospectivity"])),
    "prognoz_min": float(np.nanmin(grid["prognoz"])),
    "prognoz_max": float(np.nanmax(grid["prognoz"])),
    "display_score_min": float(np.nanmin(grid["display_score"])),
    "display_score_max": float(np.nanmax(grid["display_score"])),
    "regression_score_min": float(np.nanmin(grid["regression_score"])),
    "regression_score_max": float(np.nanmax(grid["regression_score"])),
    "regression_score_sm_min": float(np.nanmin(grid["regression_score_sm"])),
    "regression_score_sm_max": float(np.nanmax(grid["regression_score_sm"])),
    "evidence_support_min": float(np.nanmin(grid["evidence_support_sm"])),
    "evidence_support_max": float(np.nanmax(grid["evidence_support_sm"])),
    "edge_false_penalty_min": float(np.nanmin(grid["edge_false_penalty"])),
    "edge_false_penalty_max": float(np.nanmax(grid["edge_false_penalty"])),
    "reg_test_auc_proxy": None if reg_test_auc_proxy is None else float(reg_test_auc_proxy),
    "gold_zone_count": int(grid["gold_zone"].sum()),
    "point_count": int(len(points)) if points is not None else 0,
}
Path(OUT_JSON).write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

print("Готово.")
print(f"BASE_DIR: {BASE_DIR}")
print(f"PNG: {OUT_PNG}")
print(f"COMPARE: {OUT_COMPARE}")
print(f"GPKG: {OUT_GPKG}")
print(f"CSV: {OUT_CSV}")
print(f"JSON: {OUT_JSON}")
print("Диагностика:")
print(grid[["prospectivity", "prognoz", "display_score", "regression_score_sm", "evidence_support_sm", "edge_false_penalty", "gold_zone"]].describe())


Готово.
BASE_DIR: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз
PNG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v6_regression_base\gold_forecast_same_methods_fixed_v6.png
COMPARE: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v6_regression_base\compare_same_methods_fixed_v6.png
GPKG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v6_regression_base\gold_forecast_same_methods_fixed_v6.gpkg
CSV: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v6_regression_base\grid_attributes_same_methods_fixed_v6.csv
JSON: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v6_regression_base\metrics_same_methods_fixed_v6.json
Диагностика:
       prospectivity       prognoz  display_score  regression_score_sm  \
count   15684.000000  15684.000000   15684.000000         1.568400e+04   
mean        0.377053      0.622947       0.622846         5.927533e-02   
std         0.217285      0.217285       0.217341   

In [11]:
import os
import re
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm

from pyproj import CRS
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from minisom import MiniSom

warnings.filterwarnings("ignore")

# =========================================================
# НАСТРОЙКИ
# =========================================================
CELL_SIZE = 500
RANDOM_STATE = 42

# методы сохраняем
SOM_X = 12
SOM_Y = 12
SOM_ITERS = 4000
N_CLUSTERS = 6
USE_SUPERVISED = True

# Random Forest как основной ML-блок, но не единственный источник результата
RF_N_ESTIMATORS = 400
RF_MAX_DEPTH = 12
RF_MIN_SAMPLES_LEAF = 4
RF_MIN_SAMPLES_SPLIT = 8

# итоговые веса
W_GEO = 0.50
W_RF = 0.25
W_COINCIDENCE = 0.15
W_CLUSTER = 0.05
W_LOCAL = 0.05

# proximity
Q_FACIES = 0.78
Q_PALEO = 0.76
Q_STRUCT = 0.72
Q_MAGM = 0.42
Q_TECT1 = 0.74
Q_TECT2 = 0.74

# визуализация и gold-зоны
N_DISPLAY_CLASSES = 20
TOP_GOLD_Q = 0.03
TOP_LOCAL_Q = 0.96
SHOW_POINTS = False

# =========================================================
# ПУТИ
# =========================================================
def find_existing_base_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path("/mnt/data/prog_zip"),
        Path("/mnt/data"),
        Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз"),
    ]
    for base in candidates:
        shp_dir = base / "shp_dbf"
        if shp_dir.exists() and (shp_dir / "svita_new.shp").exists():
            return base
    raise FileNotFoundError("Не найден каталог с shp_dbf. Укажи BASE_DIR вручную.")

BASE_DIR = find_existing_base_dir()
SHP_DIR = BASE_DIR / "shp_dbf"
OUT_DIR = BASE_DIR / "same_methods_fixed_v8_random_forest"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAFE_ALIAS_DIR = OUT_DIR / "_safe_shp_aliases"
SAFE_ALIAS_DIR.mkdir(parents=True, exist_ok=True)

OUT_GPKG = OUT_DIR / "gold_forecast_same_methods_fixed_v8_rf.gpkg"
OUT_PNG = OUT_DIR / "gold_forecast_same_methods_fixed_v8_rf.png"
OUT_PROX = OUT_DIR / "prox_magm_same_methods_fixed_v8_rf.png"
OUT_COMPARE = OUT_DIR / "compare_same_methods_fixed_v8_rf.png"
OUT_CSV = OUT_DIR / "grid_attributes_same_methods_fixed_v8_rf.csv"
OUT_JSON = OUT_DIR / "metrics_same_methods_fixed_v8_rf.json"

# =========================================================
# ВСПОМОГАТЕЛЬНЫЕ
# =========================================================
def normalize_01(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    mn = np.nanmin(arr[finite])
    mx = np.nanmax(arr[finite])
    if np.isclose(mx, mn):
        return np.full_like(arr, 0.5, dtype=float)
    out = np.full_like(arr, np.nan, dtype=float)
    out[finite] = (arr[finite] - mn) / (mx - mn)
    return out

def robust_normalize_01(values, q_low=0.03, q_high=0.97):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    lo = np.nanquantile(arr[finite], q_low)
    hi = np.nanquantile(arr[finite], q_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or np.isclose(lo, hi):
        return normalize_01(arr)
    out = (arr - lo) / (hi - lo)
    return np.clip(out, 0, 1)

def read_sidecar_proj4(shp_path: Path):
    sidecar = shp_path.with_name(shp_path.stem + "_shp.pj4")
    if sidecar.exists():
        txt = sidecar.read_text(encoding="utf-8", errors="ignore")
        m = re.search(r"pj4=(.+)", txt)
        if m:
            return m.group(1).strip()
    return None

def prepare_ascii_aliases(shp_dir: Path, alias_dir: Path):
    aliases = {}
    stems = {}
    for name_b in os.listdir(os.fsencode(shp_dir)):
        if not name_b.endswith((b".shp", b".shx", b".dbf", b".prj", b".pj4")):
            continue
        if name_b.endswith(b"_shp.pj4"):
            continue
        base_b, ext_b = os.path.splitext(name_b)
        stems.setdefault(base_b, set()).add(ext_b)

    alias_idx = 0
    for base_b, exts in sorted(stems.items()):
        try:
            base_s = os.fsdecode(base_b)
            safe = all(ord(ch) < 128 and (ch.isalnum() or ch in "_-. ") for ch in base_s)
        except Exception:
            base_s = None
            safe = False

        if safe:
            aliases[base_s] = shp_dir / f"{base_s}.shp"
            continue

        alias_name = f"evidence_{alias_idx:02d}"
        alias_idx += 1
        for ext_b in exts:
            src = os.path.join(os.fsencode(shp_dir), base_b + ext_b)
            dst = alias_dir / f"{alias_name}{os.fsdecode(ext_b)}"
            shutil.copyfile(src, dst)
        pj4_src = os.path.join(os.fsencode(shp_dir), base_b + b"_shp.pj4")
        if os.path.exists(pj4_src):
            shutil.copyfile(pj4_src, alias_dir / f"{alias_name}_shp.pj4")
        aliases[alias_name] = alias_dir / f"{alias_name}.shp"
    return aliases

def load_layer(shp_path: Path):
    gdf = gpd.read_file(shp_path)
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    if gdf.crs is None:
        proj4 = read_sidecar_proj4(shp_path)
        if proj4:
            gdf = gdf.set_crs(CRS.from_proj4(proj4), allow_override=True)
    return gdf

def to_crs_safe(gdf, target_crs):
    if gdf.crs is None and target_crs is not None:
        return gdf.set_crs(target_crs, allow_override=True)
    if target_crs is None or gdf.crs == target_crs:
        return gdf
    return gdf.to_crs(target_crs)

def build_grid(mask, cell_size):
    mask_union = unary_union(mask.geometry)
    prepared_mask = prep(mask_union)
    minx, miny, maxx, maxy = mask.total_bounds
    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)
    rows = []
    cell_id = 0
    for r, y in enumerate(ys):
        for c, x in enumerate(xs):
            geom = box(x, y, x + cell_size, y + cell_size)
            if prepared_mask.intersects(geom):
                rows.append((cell_id, r, c, geom))
                cell_id += 1
    grid = gpd.GeoDataFrame(rows, columns=["cell_id", "row", "col", "geometry"], geometry="geometry", crs=mask.crs)
    return grid, mask_union, (len(ys), len(xs))

def add_distance_feature(grid, source, name):
    source_union = unary_union(source.geometry)
    distances = np.empty(len(grid), dtype=float)
    for i, geom in enumerate(grid.geometry.values):
        distances[i] = 0.0 if geom.intersects(source_union) else geom.distance(source_union)
    grid[name] = distances
    return grid

def distance_to_proximity(distance, transform="sqrt", q=0.75):
    d = np.asarray(distance, dtype=float)
    d = np.clip(d, 0, None)
    if transform == "sqrt":
        dt = np.sqrt(d)
    elif transform == "cbrt":
        dt = np.cbrt(d)
    elif transform == "log1p":
        dt = np.log1p(d)
    else:
        dt = d
    scale = float(np.nanquantile(dt, q))
    if not np.isfinite(scale) or scale <= 0:
        scale = max(float(np.nanmean(dt)), 1.0)
    prox = np.exp(-dt / scale)
    return np.clip(prox, 0, 1)

def smooth_on_regular_grid(grid, value_col, shape, passes=1):
    try:
        from scipy.signal import convolve2d
    except Exception:
        return grid[value_col].to_numpy()
    arr = np.full(shape, np.nan, dtype=float)
    arr[grid["row"].to_numpy(), grid["col"].to_numpy()] = grid[value_col].to_numpy()
    kernel = np.array([[1.0, 1.2, 1.0], [1.2, 3.0, 1.2], [1.0, 1.2, 1.0]], dtype=float)
    smoothed = arr.copy()
    for _ in range(max(1, passes)):
        valid = np.isfinite(smoothed).astype(float)
        filled = np.nan_to_num(smoothed, nan=0.0)
        num = convolve2d(filled, kernel, mode="same", boundary="fill", fillvalue=0)
        den = convolve2d(valid, kernel, mode="same", boundary="fill", fillvalue=0)
        with np.errstate(divide="ignore", invalid="ignore"):
            smoothed = np.where(den > 0, num / den, np.nan)
    return smoothed[grid["row"].to_numpy(), grid["col"].to_numpy()]

def collect_points(mask_crs, aliases):
    point_layers = []
    for name, shp_path in aliases.items():
        if name in {"svita_new", "fasii", "glub_raz_nw", "glub_r_nw", "gr_dol_vp_poly", "kory", "dayki_buf"}:
            continue
        gdf = load_layer(shp_path)
        gdf = to_crs_safe(gdf, mask_crs)
        geom_types = {str(x) for x in gdf.geom_type.unique()}
        if "Point" in geom_types or "MultiPoint" in geom_types:
            point_layers.append(gdf)
    if not point_layers:
        return None
    pts = pd.concat(point_layers, ignore_index=True)
    return gpd.GeoDataFrame(pts, geometry="geometry", crs=mask_crs)

def set_mask_extent(ax, mask):
    minx, miny, maxx, maxy = mask.total_bounds
    padx = (maxx - minx) * 0.02
    pady = (maxy - miny) * 0.02
    ax.set_xlim(minx - padx, maxx + padx)
    ax.set_ylim(miny - pady, maxy + pady)

def local_max_mask(grid, value_col, shape):
    try:
        from scipy.ndimage import maximum_filter
    except Exception:
        vals = grid[value_col].to_numpy()
        thr = np.nanquantile(vals, 0.98)
        return vals >= thr
    arr = np.full(shape, np.nan, dtype=float)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    vals = grid[value_col].to_numpy()
    arr[rows, cols] = vals
    filled = np.nan_to_num(arr, nan=-9999.0)
    locmax = maximum_filter(filled, size=3, mode="nearest")
    mask = np.isfinite(arr) & (filled >= locmax)
    return mask[rows, cols]

def keep_large_components(grid, bool_col, shape, min_cells=4):
    try:
        from scipy import ndimage
    except Exception:
        return grid[bool_col].to_numpy().astype(bool)
    arr = np.zeros(shape, dtype=np.uint8)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    arr[rows, cols] = grid[bool_col].to_numpy().astype(np.uint8)
    structure = np.ones((3, 3), dtype=np.uint8)
    labeled, n = ndimage.label(arr, structure=structure)
    if n == 0:
        return grid[bool_col].to_numpy().astype(bool)
    sizes = np.bincount(labeled.ravel())
    keep = np.isin(labeled, np.where(sizes >= min_cells)[0])
    keep &= labeled > 0
    return keep[rows, cols]

def make_display_classes(grid):
    prog_disp = robust_normalize_01(grid["prognoz"].to_numpy(), 0.02, 0.98)
    grid["display_score"] = prog_disp
    bins = np.linspace(0, 1, N_DISPLAY_CLASSES + 1)
    grid["display_class"] = np.digitize(prog_disp, bins[1:-1], right=False)
    return grid


def mark_gold_zones(grid, shape, mask_union):
    # Золото должно выделяться из уже правильной перспективной поверхности,
    # а не из отдельного RF-hotspot слоя.
    q_best = float(grid["prospectivity"].quantile(0.94))
    q_support = float(grid["prospectivity"].quantile(0.80))

    q_local = float(grid["local_bonus"].quantile(0.96))
    q_coinc = float(grid["coincidence_score"].quantile(0.90))
    q_magm = float(grid["prox_magm"].quantile(0.86))
    q_tmagm = float(grid["tect_magm_intersection"].quantile(0.78))

    local_peak = local_max_mask(grid, "prospectivity", shape)

    mask_boundary = mask_union.boundary
    grid["dist_to_boundary"] = np.array([geom.distance(mask_boundary) for geom in grid.geometry])

    # Золотые зоны разрешены только внутри уже хороших синих зон
    support_zone = grid["prospectivity"] >= q_support

    # 1. Компактные локальные ядра
    core_gold = (
        support_zone &
        local_peak &
        (grid["prospectivity"] >= q_best) &
        (
            (
                (grid["local_bonus"] >= q_local) &
                (grid["tect_magm_intersection"] >= q_tmagm)
            ) |
            (
                (grid["coincidence_score"] >= q_coinc) &
                (grid["prox_magm"] >= q_magm)
            )
        )
    )

    # 2. Линейные золотые зоны вдоль магматизма/тектоники
    linear_gold = (
        support_zone &
        (grid["prox_magm"] >= q_magm) &
        (grid["tect_magm_intersection"] >= q_tmagm) &
        (grid["coincidence_score"] >= q_coinc) &
        (grid["prospectivity"] >= float(grid["prospectivity"].quantile(0.90)))
    )

    # На краях допускаем только действительно сильные линейные зоны
    edge_linear_gold = (
        linear_gold &
        (grid["dist_to_boundary"] <= CELL_SIZE * 0.85)
    )

    raw_gold = core_gold | edge_linear_gold | (
        linear_gold & (grid["dist_to_boundary"] > CELL_SIZE * 0.85)
    )

    grid["gold_seed"] = raw_gold.astype(int)

    # Чистим мелкий мусор, но не слишком жестко
    large = keep_large_components(grid, "gold_seed", shape, min_cells=2)
    grid["gold_zone"] = large.astype(int)

    return grid

def plot_prox(grid, mask, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    grid.plot(column="prox_magm", ax=ax, cmap="RdYlBu_r", linewidth=0, legend=True)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    set_mask_extent(ax, mask)
    ax.set_title("prox_magm")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_final(grid, mask, points, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr.N)
    grid.plot(column="display_class", ax=ax, cmap="bwr", norm=norm, linewidth=0, legend=True)
    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=ax, color="#f2d200", linewidth=0)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=ax, color="yellow", markersize=8, edgecolor="black", linewidth=0.25)
    set_mask_extent(ax, mask)
    ax.set_title("Итоговый прогноз")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_compare(grid, mask, points, out_png):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    grid.plot(column="prox_magm", ax=axes[0], cmap="RdYlBu_r", linewidth=0)
    mask.boundary.plot(ax=axes[0], color="black", linewidth=0.5)
    set_mask_extent(axes[0], mask)
    axes[0].set_title("prox_magm")
    axes[0].set_axis_off()

    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr.N)
    grid.plot(column="display_class", ax=axes[1], cmap="bwr", norm=norm, linewidth=0)
    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=axes[1], color="#f2d200", linewidth=0)
    mask.boundary.plot(ax=axes[1], color="black", linewidth=0.5)
    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=axes[1], color="yellow", markersize=8, edgecolor="black", linewidth=0.25)
    set_mask_extent(axes[1], mask)
    axes[1].set_title("Итоговый прогноз")
    axes[1].set_axis_off()

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

# =========================================================
# ЗАГРУЗКА
# =========================================================
aliases = prepare_ascii_aliases(SHP_DIR, SAFE_ALIAS_DIR)
mask = load_layer(aliases["svita_new"])
facies = to_crs_safe(load_layer(aliases["fasii"]), mask.crs)
tect1 = to_crs_safe(load_layer(aliases["glub_raz_nw"]), mask.crs)
tect2 = to_crs_safe(load_layer(aliases["glub_r_nw"]), mask.crs)
paleo = to_crs_safe(load_layer(aliases["gr_dol_vp_poly"]), mask.crs)
struct = to_crs_safe(load_layer(aliases["kory"]), mask.crs)
magm = to_crs_safe(load_layer(aliases["dayki_buf"]), mask.crs)
points = collect_points(mask.crs, aliases)

# =========================================================
# СЕТКА
# =========================================================
grid, mask_union, grid_shape = build_grid(mask, CELL_SIZE)

# =========================================================
# ДИСТАНЦИИ
# =========================================================
grid = add_distance_feature(grid, facies, "dist_facies")
grid = add_distance_feature(grid, paleo, "dist_paleo")
grid = add_distance_feature(grid, struct, "dist_struct")
grid = add_distance_feature(grid, magm, "dist_magm")
grid = add_distance_feature(grid, tect1, "dist_tect1")
grid = add_distance_feature(grid, tect2, "dist_tect2")

# =========================================================
# PROXIMITY
# =========================================================
grid["prox_facies"] = distance_to_proximity(grid["dist_facies"], transform="cbrt", q=Q_FACIES)
grid["prox_paleo"] = distance_to_proximity(grid["dist_paleo"], transform="cbrt", q=Q_PALEO)
grid["prox_struct"] = distance_to_proximity(grid["dist_struct"], transform="sqrt", q=Q_STRUCT)
grid["prox_magm"] = distance_to_proximity(grid["dist_magm"], transform="sqrt", q=Q_MAGM)
grid["prox_tect1"] = distance_to_proximity(grid["dist_tect1"], transform="cbrt", q=Q_TECT1)
grid["prox_tect2"] = distance_to_proximity(grid["dist_tect2"], transform="cbrt", q=Q_TECT2)

# =========================================================
# INTERACTIONS
# =========================================================
grid["tect_combo"] = 0.5 * (grid["prox_tect1"] + grid["prox_tect2"])
grid["tect_intersection"] = grid["prox_tect1"] * grid["prox_tect2"]
grid["tect_magm_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_magm"])
grid["tect_struct_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_struct"])
grid["paleo_struct_intersection"] = np.sqrt(grid["prox_paleo"] * grid["prox_struct"])

combo_core = (
    np.clip(grid["tect_combo"], 0, 1) *
    np.clip(0.55 * grid["prox_magm"] + 0.45 * grid["prox_struct"], 0, 1) *
    np.clip(0.60 * grid["prox_paleo"] + 0.40 * grid["prox_facies"], 0, 1)
)
grid["coincidence_score"] = robust_normalize_01(np.sqrt(np.clip(combo_core, 0, 1)), 0.02, 0.98)

tect_support = 0.40 * grid["prox_magm"] + 0.35 * grid["prox_struct"] + 0.25 * grid["prox_paleo"]
grid["tect_only_penalty"] = robust_normalize_01(np.clip(grid["tect_combo"] - tect_support, 0, 1), 0.02, 0.98)

# =========================================================
# GEO SCORE - базовая геологическая поверхность
# =========================================================
grid["geo_score_raw"] = (
    0.14 * grid["prox_tect1"] +
    0.14 * grid["prox_tect2"] +
    0.14 * grid["prox_paleo"] +
    0.11 * grid["prox_struct"] +
    0.08 * grid["prox_facies"] +
    0.08 * grid["prox_magm"] +
    0.08 * grid["tect_intersection"] +
    0.08 * grid["tect_magm_intersection"] +
    0.05 * grid["tect_struct_intersection"] +
    0.04 * grid["paleo_struct_intersection"] +
    0.10 * grid["coincidence_score"] -
    0.08 * grid["tect_only_penalty"]
)
grid["geo_score"] = robust_normalize_01(grid["geo_score_raw"], 0.02, 0.98)
grid["geo_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "geo_score", grid_shape, passes=2), 0.02, 0.98)

# =========================================================
# RANDOM FOREST AS ML BLOCK
# =========================================================
grid["target"] = 0
grid["rf_score"] = grid["geo_score_sm"]
use_supervised = False
rf_test_proxy = None
feature_importance = {}

feature_cols = [
    "prox_facies", "prox_paleo", "prox_struct", "prox_magm",
    "prox_tect1", "prox_tect2", "tect_combo", "tect_intersection",
    "tect_magm_intersection", "tect_struct_intersection",
    "paleo_struct_intersection", "coincidence_score", "tect_only_penalty",
    "geo_score_sm"
]

if USE_SUPERVISED and points is not None and len(points) > 0:
    try:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", predicate="within")
    except Exception:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", op="within")

    positive_cells = joined["cell_id"].dropna().astype(int).unique().tolist()
    grid.loc[grid["cell_id"].isin(positive_cells), "target"] = 1

    pos = int(grid["target"].sum())
    neg = int((grid["target"] == 0).sum())

    if pos >= 20 and neg > pos:
        X = grid[feature_cols].fillna(0).to_numpy()
        y = grid["target"].to_numpy()

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
        )

        rf_eval = RandomForestClassifier(
            n_estimators=RF_N_ESTIMATORS,
            max_depth=RF_MAX_DEPTH,
            min_samples_leaf=RF_MIN_SAMPLES_LEAF,
            min_samples_split=RF_MIN_SAMPLES_SPLIT,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        rf_eval.fit(X_train, y_train)
        test_prob = rf_eval.predict_proba(X_test)[:, 1]
        y_test = np.asarray(y_test)
        pos_mean = float(np.mean(test_prob[y_test == 1])) if np.any(y_test == 1) else np.nan
        neg_mean = float(np.mean(test_prob[y_test == 0])) if np.any(y_test == 0) else np.nan
        rf_test_proxy = pos_mean - neg_mean

        rf = RandomForestClassifier(
            n_estimators=RF_N_ESTIMATORS,
            max_depth=RF_MAX_DEPTH,
            min_samples_leaf=RF_MIN_SAMPLES_LEAF,
            min_samples_split=RF_MIN_SAMPLES_SPLIT,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        rf.fit(X, y)
        grid["rf_score"] = robust_normalize_01(rf.predict_proba(X)[:, 1], 0.02, 0.98)
        feature_importance = dict(zip(feature_cols, rf.feature_importances_.tolist()))
        use_supervised = True

grid["rf_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "rf_score", grid_shape, passes=2), 0.02, 0.98)
grid["ml_score"] = grid["rf_score_sm"]

# =========================================================
# SOM + KMEANS
# =========================================================
X = grid[feature_cols].fillna(0).to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

som = MiniSom(
    x=SOM_X, y=SOM_Y, input_len=X_scaled.shape[1],
    sigma=1.1, learning_rate=0.38, random_seed=RANDOM_STATE
)
som.random_weights_init(X_scaled)
som.train_random(X_scaled, SOM_ITERS)

winners = np.array([som.winner(x) for x in X_scaled])
grid["som_x"] = winners[:, 0]
grid["som_y"] = winners[:, 1]
grid["som_node"] = grid["som_x"].astype(str) + "_" + grid["som_y"].astype(str)

som_weights = som.get_weights().reshape(SOM_X * SOM_Y, X_scaled.shape[1])
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=20)
neuron_cluster = kmeans.fit_predict(som_weights)

node_to_cluster = {}
idx = 0
for i in range(SOM_X):
    for j in range(SOM_Y):
        node_to_cluster[f"{i}_{j}"] = int(neuron_cluster[idx])
        idx += 1
grid["cluster"] = grid["som_node"].map(node_to_cluster).astype(int)

cluster_geo = grid.groupby("cluster")["geo_score_sm"].mean().reset_index(name="cluster_geo_mean")
cluster_rf = grid.groupby("cluster")["rf_score_sm"].mean().reset_index(name="cluster_rf_mean")
cluster_coinc = grid.groupby("cluster")["coincidence_score"].mean().reset_index(name="cluster_coinc_mean")
cluster_stats = cluster_geo.merge(cluster_rf, on="cluster", how="outer").merge(cluster_coinc, on="cluster", how="outer")

if use_supervised:
    cluster_hits = grid.groupby("cluster").agg(cells=("cell_id", "count"), positives=("target", "sum")).reset_index()
    cluster_hits["hit_rate"] = cluster_hits["positives"] / cluster_hits["cells"]
    cluster_stats = cluster_stats.merge(cluster_hits, on="cluster", how="left")
    cluster_stats["hit_rate"] = cluster_stats["hit_rate"].fillna(0)
    cluster_stats["cluster_score"] = robust_normalize_01(
        0.35 * cluster_stats["cluster_geo_mean"] +
        0.35 * cluster_stats["cluster_rf_mean"] +
        0.15 * cluster_stats["cluster_coinc_mean"] +
        0.15 * robust_normalize_01(cluster_stats["hit_rate"], 0.0, 1.0),
        0.02, 0.98,
    )
else:
    cluster_stats["cluster_score"] = robust_normalize_01(
        0.45 * cluster_stats["cluster_geo_mean"] +
        0.35 * cluster_stats["cluster_rf_mean"] +
        0.20 * cluster_stats["cluster_coinc_mean"],
        0.02, 0.98,
    )

grid = grid.merge(cluster_stats[["cluster", "cluster_score"]], on="cluster", how="left")
grid["cluster_score"] = grid["cluster_score"].fillna(grid["geo_score_sm"])

# =========================================================
# ИТОГ
# =========================================================
grid["local_bonus"] = robust_normalize_01(
    0.38 * grid["tect_intersection"] +
    0.37 * grid["tect_magm_intersection"] +
    0.25 * grid["tect_struct_intersection"],
    0.02, 0.98,
)

mask_boundary = mask_union.boundary
grid["dist_to_boundary"] = np.array([geom.distance(mask_boundary) for geom in grid.geometry])
edge_near = np.exp(-grid["dist_to_boundary"].to_numpy() / (CELL_SIZE * 2.2))
grid["edge_false_penalty"] = robust_normalize_01(
    edge_near * np.clip(grid["tect_only_penalty"], 0, 1),
    0.02, 0.98
)

grid["prospectivity_raw"] = (
    W_GEO * grid["geo_score_sm"] +
    W_RF * grid["rf_score_sm"] +
    W_COINCIDENCE * grid["coincidence_score"] +
    W_CLUSTER * grid["cluster_score"] +
    W_LOCAL * grid["local_bonus"]
)
grid["prospectivity_raw"] = (
    grid["prospectivity_raw"]
    - 0.05 * grid["tect_only_penalty"]
    - 0.04 * grid["edge_false_penalty"]
)
grid["prospectivity_raw_sm"] = smooth_on_regular_grid(grid, "prospectivity_raw", grid_shape, passes=2)
grid["prospectivity"] = robust_normalize_01(grid["prospectivity_raw_sm"], 0.02, 0.98)

# в логике презентации: меньше = лучше
grid["prognoz"] = 1.0 - grid["prospectivity"]

top_thr = float(grid["prospectivity"].quantile(0.90))
grid["top10"] = (grid["prospectivity"] >= top_thr).astype(int)

try:
    grid["prospect_class"] = pd.qcut(
        grid["prospectivity"],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop"
    )
except Exception:
    grid["prospect_class"] = "medium"

grid = make_display_classes(grid)
grid = mark_gold_zones(grid, grid_shape, mask_union)

# =========================================================
# СОХРАНЕНИЕ
# =========================================================
if OUT_GPKG.exists():
    OUT_GPKG.unlink()

grid.to_file(OUT_GPKG, layer="forecast_grid", driver="GPKG")
if points is not None and len(points) > 0:
    points.to_file(OUT_GPKG, layer="evidence_points", driver="GPKG")

grid.drop(columns="geometry").to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

plot_prox(grid, mask, OUT_PROX)
plot_final(grid, mask, points, OUT_PNG)
plot_compare(grid, mask, points, OUT_COMPARE)

metrics = {
    "base_dir": str(BASE_DIR),
    "grid_cells": int(len(grid)),
    "cell_size": CELL_SIZE,
    "use_supervised_requested": bool(USE_SUPERVISED),
    "use_supervised_applied": bool(use_supervised),
    "positive_cells": int(grid["target"].sum()),
    "top10_threshold": float(top_thr),
    "prospectivity_min": float(np.nanmin(grid["prospectivity"])),
    "prospectivity_p05": float(np.nanquantile(grid["prospectivity"], 0.05)),
    "prospectivity_p50": float(np.nanquantile(grid["prospectivity"], 0.50)),
    "prospectivity_p95": float(np.nanquantile(grid["prospectivity"], 0.95)),
    "prospectivity_max": float(np.nanmax(grid["prospectivity"])),
    "prognoz_min": float(np.nanmin(grid["prognoz"])),
    "prognoz_max": float(np.nanmax(grid["prognoz"])),
    "display_score_min": float(np.nanmin(grid["display_score"])),
    "display_score_max": float(np.nanmax(grid["display_score"])),
    "rf_score_min": float(np.nanmin(grid["rf_score"])),
    "rf_score_max": float(np.nanmax(grid["rf_score"])),
    "rf_score_sm_min": float(np.nanmin(grid["rf_score_sm"])),
    "rf_score_sm_max": float(np.nanmax(grid["rf_score_sm"])),
    "rf_test_proxy": None if rf_test_proxy is None else float(rf_test_proxy),
    "edge_false_penalty_min": float(np.nanmin(grid["edge_false_penalty"])),
    "edge_false_penalty_max": float(np.nanmax(grid["edge_false_penalty"])),
    "gold_zone_count": int(grid["gold_zone"].sum()),
    "gold_zone_share": float(grid["gold_zone"].mean()),
    "prospectivity_q80": float(np.nanquantile(grid["prospectivity"], 0.80)),
    "prospectivity_q90": float(np.nanquantile(grid["prospectivity"], 0.90)),
    "prospectivity_q94": float(np.nanquantile(grid["prospectivity"], 0.94)),
    "point_count": int(len(points)) if points is not None else 0,
    "rf_feature_importance": feature_importance,
}
Path(OUT_JSON).write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

print("Готово.")
print(f"BASE_DIR: {BASE_DIR}")
print(f"PNG: {OUT_PNG}")
print(f"COMPARE: {OUT_COMPARE}")
print(f"GPKG: {OUT_GPKG}")
print(f"CSV: {OUT_CSV}")
print(f"JSON: {OUT_JSON}")
print("Диагностика:")
print(grid[["prospectivity", "prognoz", "display_score", "rf_score_sm", "gold_zone", "prox_magm", "tect_magm_intersection", "coincidence_score"]].describe())


Готово.
BASE_DIR: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз
PNG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v8_random_forest\gold_forecast_same_methods_fixed_v8_rf.png
COMPARE: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v8_random_forest\compare_same_methods_fixed_v8_rf.png
GPKG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v8_random_forest\gold_forecast_same_methods_fixed_v8_rf.gpkg
CSV: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v8_random_forest\grid_attributes_same_methods_fixed_v8_rf.csv
JSON: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v8_random_forest\metrics_same_methods_fixed_v8_rf.json
Диагностика:
       prospectivity       prognoz  display_score   rf_score_sm     gold_zone  \
count   15684.000000  15684.000000   15684.000000  15684.000000  15684.000000   
mean        0.418994      0.581006       0.580971      0.190389      0.035514   
std         0.254734      

In [12]:
import os
import re
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm

from pyproj import CRS
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from minisom import MiniSom

warnings.filterwarnings("ignore")

# =========================================================
# НАСТРОЙКИ
# =========================================================
CELL_SIZE = 500
RANDOM_STATE = 42

# методы сохраняем
SOM_X = 12
SOM_Y = 12
SOM_ITERS = 4000
N_CLUSTERS = 6
USE_SUPERVISED = True

# Random Forest как основной ML-блок, но не единственный источник результата
RF_N_ESTIMATORS = 400
RF_MAX_DEPTH = 12
RF_MIN_SAMPLES_LEAF = 4
RF_MIN_SAMPLES_SPLIT = 8

# итоговые веса
W_GEO = 0.50
W_RF = 0.25
W_COINCIDENCE = 0.15
W_CLUSTER = 0.05
W_LOCAL = 0.05

# proximity
Q_FACIES = 0.78
Q_PALEO = 0.76
Q_STRUCT = 0.72
Q_MAGM = 0.42
Q_TECT1 = 0.74
Q_TECT2 = 0.74

# визуализация и gold-зоны
N_DISPLAY_CLASSES = 20
TOP_GOLD_Q = 0.03
TOP_LOCAL_Q = 0.96
SHOW_POINTS = False

# =========================================================
# ПУТИ
# =========================================================
def find_existing_base_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path("/mnt/data/prog_zip"),
        Path("/mnt/data"),
        Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз"),
    ]
    for base in candidates:
        shp_dir = base / "shp_dbf"
        if shp_dir.exists() and (shp_dir / "svita_new.shp").exists():
            return base
    raise FileNotFoundError("Не найден каталог с shp_dbf. Укажи BASE_DIR вручную.")

BASE_DIR = find_existing_base_dir()
SHP_DIR = BASE_DIR / "shp_dbf"
OUT_DIR = BASE_DIR / "same_methods_fixed_v9_random_forest"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAFE_ALIAS_DIR = OUT_DIR / "_safe_shp_aliases"
SAFE_ALIAS_DIR.mkdir(parents=True, exist_ok=True)

OUT_GPKG = OUT_DIR / "gold_forecast_same_methods_fixed_v9_rf.gpkg"
OUT_PNG = OUT_DIR / "gold_forecast_same_methods_fixed_v9_rf.png"
OUT_PROX = OUT_DIR / "prox_magm_same_methods_fixed_v9_rf.png"
OUT_COMPARE = OUT_DIR / "compare_same_methods_fixed_v9_rf.png"
OUT_CSV = OUT_DIR / "grid_attributes_same_methods_fixed_v9_rf.csv"
OUT_JSON = OUT_DIR / "metrics_same_methods_fixed_v9_rf.json"

# =========================================================
# ВСПОМОГАТЕЛЬНЫЕ
# =========================================================
def normalize_01(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    mn = np.nanmin(arr[finite])
    mx = np.nanmax(arr[finite])
    if np.isclose(mx, mn):
        return np.full_like(arr, 0.5, dtype=float)
    out = np.full_like(arr, np.nan, dtype=float)
    out[finite] = (arr[finite] - mn) / (mx - mn)
    return out

def robust_normalize_01(values, q_low=0.03, q_high=0.97):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    lo = np.nanquantile(arr[finite], q_low)
    hi = np.nanquantile(arr[finite], q_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or np.isclose(lo, hi):
        return normalize_01(arr)
    out = (arr - lo) / (hi - lo)
    return np.clip(out, 0, 1)

def read_sidecar_proj4(shp_path: Path):
    sidecar = shp_path.with_name(shp_path.stem + "_shp.pj4")
    if sidecar.exists():
        txt = sidecar.read_text(encoding="utf-8", errors="ignore")
        m = re.search(r"pj4=(.+)", txt)
        if m:
            return m.group(1).strip()
    return None

def prepare_ascii_aliases(shp_dir: Path, alias_dir: Path):
    aliases = {}
    stems = {}
    for name_b in os.listdir(os.fsencode(shp_dir)):
        if not name_b.endswith((b".shp", b".shx", b".dbf", b".prj", b".pj4")):
            continue
        if name_b.endswith(b"_shp.pj4"):
            continue
        base_b, ext_b = os.path.splitext(name_b)
        stems.setdefault(base_b, set()).add(ext_b)

    alias_idx = 0
    for base_b, exts in sorted(stems.items()):
        try:
            base_s = os.fsdecode(base_b)
            safe = all(ord(ch) < 128 and (ch.isalnum() or ch in "_-. ") for ch in base_s)
        except Exception:
            base_s = None
            safe = False

        if safe:
            aliases[base_s] = shp_dir / f"{base_s}.shp"
            continue

        alias_name = f"evidence_{alias_idx:02d}"
        alias_idx += 1
        for ext_b in exts:
            src = os.path.join(os.fsencode(shp_dir), base_b + ext_b)
            dst = alias_dir / f"{alias_name}{os.fsdecode(ext_b)}"
            shutil.copyfile(src, dst)
        pj4_src = os.path.join(os.fsencode(shp_dir), base_b + b"_shp.pj4")
        if os.path.exists(pj4_src):
            shutil.copyfile(pj4_src, alias_dir / f"{alias_name}_shp.pj4")
        aliases[alias_name] = alias_dir / f"{alias_name}.shp"
    return aliases

def load_layer(shp_path: Path):
    gdf = gpd.read_file(shp_path)
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    if gdf.crs is None:
        proj4 = read_sidecar_proj4(shp_path)
        if proj4:
            gdf = gdf.set_crs(CRS.from_proj4(proj4), allow_override=True)
    return gdf

def to_crs_safe(gdf, target_crs):
    if gdf.crs is None and target_crs is not None:
        return gdf.set_crs(target_crs, allow_override=True)
    if target_crs is None or gdf.crs == target_crs:
        return gdf
    return gdf.to_crs(target_crs)

def build_grid(mask, cell_size):
    mask_union = unary_union(mask.geometry)
    prepared_mask = prep(mask_union)
    minx, miny, maxx, maxy = mask.total_bounds
    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)
    rows = []
    cell_id = 0
    for r, y in enumerate(ys):
        for c, x in enumerate(xs):
            geom = box(x, y, x + cell_size, y + cell_size)
            if prepared_mask.intersects(geom):
                rows.append((cell_id, r, c, geom))
                cell_id += 1
    grid = gpd.GeoDataFrame(rows, columns=["cell_id", "row", "col", "geometry"], geometry="geometry", crs=mask.crs)
    return grid, mask_union, (len(ys), len(xs))

def add_distance_feature(grid, source, name):
    source_union = unary_union(source.geometry)
    distances = np.empty(len(grid), dtype=float)
    for i, geom in enumerate(grid.geometry.values):
        distances[i] = 0.0 if geom.intersects(source_union) else geom.distance(source_union)
    grid[name] = distances
    return grid

def distance_to_proximity(distance, transform="sqrt", q=0.75):
    d = np.asarray(distance, dtype=float)
    d = np.clip(d, 0, None)
    if transform == "sqrt":
        dt = np.sqrt(d)
    elif transform == "cbrt":
        dt = np.cbrt(d)
    elif transform == "log1p":
        dt = np.log1p(d)
    else:
        dt = d
    scale = float(np.nanquantile(dt, q))
    if not np.isfinite(scale) or scale <= 0:
        scale = max(float(np.nanmean(dt)), 1.0)
    prox = np.exp(-dt / scale)
    return np.clip(prox, 0, 1)

def smooth_on_regular_grid(grid, value_col, shape, passes=1):
    try:
        from scipy.signal import convolve2d
    except Exception:
        return grid[value_col].to_numpy()
    arr = np.full(shape, np.nan, dtype=float)
    arr[grid["row"].to_numpy(), grid["col"].to_numpy()] = grid[value_col].to_numpy()
    kernel = np.array([[1.0, 1.2, 1.0], [1.2, 3.0, 1.2], [1.0, 1.2, 1.0]], dtype=float)
    smoothed = arr.copy()
    for _ in range(max(1, passes)):
        valid = np.isfinite(smoothed).astype(float)
        filled = np.nan_to_num(smoothed, nan=0.0)
        num = convolve2d(filled, kernel, mode="same", boundary="fill", fillvalue=0)
        den = convolve2d(valid, kernel, mode="same", boundary="fill", fillvalue=0)
        with np.errstate(divide="ignore", invalid="ignore"):
            smoothed = np.where(den > 0, num / den, np.nan)
    return smoothed[grid["row"].to_numpy(), grid["col"].to_numpy()]

def collect_points(mask_crs, aliases):
    point_layers = []
    for name, shp_path in aliases.items():
        if name in {"svita_new", "fasii", "glub_raz_nw", "glub_r_nw", "gr_dol_vp_poly", "kory", "dayki_buf"}:
            continue
        gdf = load_layer(shp_path)
        gdf = to_crs_safe(gdf, mask_crs)
        geom_types = {str(x) for x in gdf.geom_type.unique()}
        if "Point" in geom_types or "MultiPoint" in geom_types:
            point_layers.append(gdf)
    if not point_layers:
        return None
    pts = pd.concat(point_layers, ignore_index=True)
    return gpd.GeoDataFrame(pts, geometry="geometry", crs=mask_crs)

def set_mask_extent(ax, mask):
    minx, miny, maxx, maxy = mask.total_bounds
    padx = (maxx - minx) * 0.02
    pady = (maxy - miny) * 0.02
    ax.set_xlim(minx - padx, maxx + padx)
    ax.set_ylim(miny - pady, maxy + pady)

def local_max_mask(grid, value_col, shape):
    try:
        from scipy.ndimage import maximum_filter
    except Exception:
        vals = grid[value_col].to_numpy()
        thr = np.nanquantile(vals, 0.98)
        return vals >= thr
    arr = np.full(shape, np.nan, dtype=float)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    vals = grid[value_col].to_numpy()
    arr[rows, cols] = vals
    filled = np.nan_to_num(arr, nan=-9999.0)
    locmax = maximum_filter(filled, size=3, mode="nearest")
    mask = np.isfinite(arr) & (filled >= locmax)
    return mask[rows, cols]

def keep_large_components(grid, bool_col, shape, min_cells=4):
    try:
        from scipy import ndimage
    except Exception:
        return grid[bool_col].to_numpy().astype(bool)
    arr = np.zeros(shape, dtype=np.uint8)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    arr[rows, cols] = grid[bool_col].to_numpy().astype(np.uint8)
    structure = np.ones((3, 3), dtype=np.uint8)
    labeled, n = ndimage.label(arr, structure=structure)
    if n == 0:
        return grid[bool_col].to_numpy().astype(bool)
    sizes = np.bincount(labeled.ravel())
    keep = np.isin(labeled, np.where(sizes >= min_cells)[0])
    keep &= labeled > 0
    return keep[rows, cols]

def make_display_classes(grid):
    prog_disp = robust_normalize_01(grid["prognoz"].to_numpy(), 0.02, 0.98)
    grid["display_score"] = prog_disp
    bins = np.linspace(0, 1, N_DISPLAY_CLASSES + 1)
    grid["display_class"] = np.digitize(prog_disp, bins[1:-1], right=False)
    return grid



def mark_gold_zones(grid, shape, mask_union):
    q_best = float(grid["prospectivity"].quantile(0.97))
    q_support = float(grid["prospectivity"].quantile(0.88))

    q_local = float(grid["local_bonus"].quantile(0.97))
    q_coinc = float(grid["coincidence_score"].quantile(0.94))
    q_magm = float(grid["prox_magm"].quantile(0.90))
    q_tmagm = float(grid["tect_magm_intersection"].quantile(0.84))

    local_peak = local_max_mask(grid, "prospectivity", shape)

    mask_boundary = mask_union.boundary
    grid["dist_to_boundary"] = np.array([geom.distance(mask_boundary) for geom in grid.geometry])

    support_zone = grid["prospectivity"] >= q_support

    core_gold = (
        support_zone &
        local_peak &
        (grid["prospectivity"] >= q_best) &
        (
            (
                (grid["local_bonus"] >= q_local) &
                (grid["tect_magm_intersection"] >= q_tmagm)
            ) |
            (
                (grid["coincidence_score"] >= q_coinc) &
                (grid["prox_magm"] >= q_magm)
            )
        )
    )

    linear_gold = (
        support_zone &
        (grid["prospectivity"] >= float(grid["prospectivity"].quantile(0.94))) &
        (grid["prox_magm"] >= q_magm) &
        (grid["tect_magm_intersection"] >= q_tmagm) &
        (grid["coincidence_score"] >= q_coinc)
    )

    edge_linear_gold = (
        linear_gold &
        (grid["dist_to_boundary"] <= CELL_SIZE * 0.75)
    )

    raw_gold = core_gold | edge_linear_gold

    grid["gold_seed"] = raw_gold.astype(int)

    large = keep_large_components(grid, "gold_seed", shape, min_cells=4)
    grid["gold_zone"] = large.astype(int)

    return grid

def plot_prox(grid, mask, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    grid.plot(column="prox_magm", ax=ax, cmap="RdYlBu_r", linewidth=0, legend=True)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    set_mask_extent(ax, mask)
    ax.set_title("prox_magm")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_final(grid, mask, points, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr.N)
    grid.plot(column="display_class", ax=ax, cmap="bwr", norm=norm, linewidth=0, legend=True)
    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=ax, color="#f2d200", linewidth=0)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=ax, color="yellow", markersize=8, edgecolor="black", linewidth=0.25)
    set_mask_extent(ax, mask)
    ax.set_title("Итоговый прогноз")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_compare(grid, mask, points, out_png):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    grid.plot(column="prox_magm", ax=axes[0], cmap="RdYlBu_r", linewidth=0)
    mask.boundary.plot(ax=axes[0], color="black", linewidth=0.5)
    set_mask_extent(axes[0], mask)
    axes[0].set_title("prox_magm")
    axes[0].set_axis_off()

    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr.N)
    grid.plot(column="display_class", ax=axes[1], cmap="bwr", norm=norm, linewidth=0)
    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=axes[1], color="#f2d200", linewidth=0)
    mask.boundary.plot(ax=axes[1], color="black", linewidth=0.5)
    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=axes[1], color="yellow", markersize=8, edgecolor="black", linewidth=0.25)
    set_mask_extent(axes[1], mask)
    axes[1].set_title("Итоговый прогноз")
    axes[1].set_axis_off()

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

# =========================================================
# ЗАГРУЗКА
# =========================================================
aliases = prepare_ascii_aliases(SHP_DIR, SAFE_ALIAS_DIR)
mask = load_layer(aliases["svita_new"])
facies = to_crs_safe(load_layer(aliases["fasii"]), mask.crs)
tect1 = to_crs_safe(load_layer(aliases["glub_raz_nw"]), mask.crs)
tect2 = to_crs_safe(load_layer(aliases["glub_r_nw"]), mask.crs)
paleo = to_crs_safe(load_layer(aliases["gr_dol_vp_poly"]), mask.crs)
struct = to_crs_safe(load_layer(aliases["kory"]), mask.crs)
magm = to_crs_safe(load_layer(aliases["dayki_buf"]), mask.crs)
points = collect_points(mask.crs, aliases)

# =========================================================
# СЕТКА
# =========================================================
grid, mask_union, grid_shape = build_grid(mask, CELL_SIZE)

# =========================================================
# ДИСТАНЦИИ
# =========================================================
grid = add_distance_feature(grid, facies, "dist_facies")
grid = add_distance_feature(grid, paleo, "dist_paleo")
grid = add_distance_feature(grid, struct, "dist_struct")
grid = add_distance_feature(grid, magm, "dist_magm")
grid = add_distance_feature(grid, tect1, "dist_tect1")
grid = add_distance_feature(grid, tect2, "dist_tect2")

# =========================================================
# PROXIMITY
# =========================================================
grid["prox_facies"] = distance_to_proximity(grid["dist_facies"], transform="cbrt", q=Q_FACIES)
grid["prox_paleo"] = distance_to_proximity(grid["dist_paleo"], transform="cbrt", q=Q_PALEO)
grid["prox_struct"] = distance_to_proximity(grid["dist_struct"], transform="sqrt", q=Q_STRUCT)
grid["prox_magm"] = distance_to_proximity(grid["dist_magm"], transform="sqrt", q=Q_MAGM)
grid["prox_tect1"] = distance_to_proximity(grid["dist_tect1"], transform="cbrt", q=Q_TECT1)
grid["prox_tect2"] = distance_to_proximity(grid["dist_tect2"], transform="cbrt", q=Q_TECT2)

# =========================================================
# INTERACTIONS
# =========================================================
grid["tect_combo"] = 0.5 * (grid["prox_tect1"] + grid["prox_tect2"])
grid["tect_intersection"] = grid["prox_tect1"] * grid["prox_tect2"]
grid["tect_magm_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_magm"])
grid["tect_struct_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_struct"])
grid["paleo_struct_intersection"] = np.sqrt(grid["prox_paleo"] * grid["prox_struct"])

combo_core = (
    np.clip(grid["tect_combo"], 0, 1) *
    np.clip(0.55 * grid["prox_magm"] + 0.45 * grid["prox_struct"], 0, 1) *
    np.clip(0.60 * grid["prox_paleo"] + 0.40 * grid["prox_facies"], 0, 1)
)
grid["coincidence_score"] = robust_normalize_01(np.sqrt(np.clip(combo_core, 0, 1)), 0.02, 0.98)

tect_support = 0.40 * grid["prox_magm"] + 0.35 * grid["prox_struct"] + 0.25 * grid["prox_paleo"]
grid["tect_only_penalty"] = robust_normalize_01(np.clip(grid["tect_combo"] - tect_support, 0, 1), 0.02, 0.98)

# =========================================================
# GEO SCORE - базовая геологическая поверхность
# =========================================================
grid["geo_score_raw"] = (
    0.14 * grid["prox_tect1"] +
    0.14 * grid["prox_tect2"] +
    0.14 * grid["prox_paleo"] +
    0.11 * grid["prox_struct"] +
    0.08 * grid["prox_facies"] +
    0.08 * grid["prox_magm"] +
    0.08 * grid["tect_intersection"] +
    0.08 * grid["tect_magm_intersection"] +
    0.05 * grid["tect_struct_intersection"] +
    0.04 * grid["paleo_struct_intersection"] +
    0.10 * grid["coincidence_score"] -
    0.08 * grid["tect_only_penalty"]
)
grid["geo_score"] = robust_normalize_01(grid["geo_score_raw"], 0.02, 0.98)
grid["geo_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "geo_score", grid_shape, passes=2), 0.02, 0.98)

# =========================================================
# RANDOM FOREST AS ML BLOCK
# =========================================================
grid["target"] = 0
grid["rf_score"] = grid["geo_score_sm"]
use_supervised = False
rf_test_proxy = None
feature_importance = {}

feature_cols = [
    "prox_facies", "prox_paleo", "prox_struct", "prox_magm",
    "prox_tect1", "prox_tect2", "tect_combo", "tect_intersection",
    "tect_magm_intersection", "tect_struct_intersection",
    "paleo_struct_intersection", "coincidence_score", "tect_only_penalty",
    "geo_score_sm"
]

if USE_SUPERVISED and points is not None and len(points) > 0:
    try:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", predicate="within")
    except Exception:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", op="within")

    positive_cells = joined["cell_id"].dropna().astype(int).unique().tolist()
    grid.loc[grid["cell_id"].isin(positive_cells), "target"] = 1

    pos = int(grid["target"].sum())
    neg = int((grid["target"] == 0).sum())

    if pos >= 20 and neg > pos:
        X = grid[feature_cols].fillna(0).to_numpy()
        y = grid["target"].to_numpy()

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
        )

        rf_eval = RandomForestClassifier(
            n_estimators=RF_N_ESTIMATORS,
            max_depth=RF_MAX_DEPTH,
            min_samples_leaf=RF_MIN_SAMPLES_LEAF,
            min_samples_split=RF_MIN_SAMPLES_SPLIT,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        rf_eval.fit(X_train, y_train)
        test_prob = rf_eval.predict_proba(X_test)[:, 1]
        y_test = np.asarray(y_test)
        pos_mean = float(np.mean(test_prob[y_test == 1])) if np.any(y_test == 1) else np.nan
        neg_mean = float(np.mean(test_prob[y_test == 0])) if np.any(y_test == 0) else np.nan
        rf_test_proxy = pos_mean - neg_mean

        rf = RandomForestClassifier(
            n_estimators=RF_N_ESTIMATORS,
            max_depth=RF_MAX_DEPTH,
            min_samples_leaf=RF_MIN_SAMPLES_LEAF,
            min_samples_split=RF_MIN_SAMPLES_SPLIT,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        rf.fit(X, y)
        grid["rf_score"] = robust_normalize_01(rf.predict_proba(X)[:, 1], 0.02, 0.98)
        feature_importance = dict(zip(feature_cols, rf.feature_importances_.tolist()))
        use_supervised = True

grid["rf_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "rf_score", grid_shape, passes=2), 0.02, 0.98)
grid["ml_score"] = grid["rf_score_sm"]

# =========================================================
# SOM + KMEANS
# =========================================================
X = grid[feature_cols].fillna(0).to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

som = MiniSom(
    x=SOM_X, y=SOM_Y, input_len=X_scaled.shape[1],
    sigma=1.1, learning_rate=0.38, random_seed=RANDOM_STATE
)
som.random_weights_init(X_scaled)
som.train_random(X_scaled, SOM_ITERS)

winners = np.array([som.winner(x) for x in X_scaled])
grid["som_x"] = winners[:, 0]
grid["som_y"] = winners[:, 1]
grid["som_node"] = grid["som_x"].astype(str) + "_" + grid["som_y"].astype(str)

som_weights = som.get_weights().reshape(SOM_X * SOM_Y, X_scaled.shape[1])
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=20)
neuron_cluster = kmeans.fit_predict(som_weights)

node_to_cluster = {}
idx = 0
for i in range(SOM_X):
    for j in range(SOM_Y):
        node_to_cluster[f"{i}_{j}"] = int(neuron_cluster[idx])
        idx += 1
grid["cluster"] = grid["som_node"].map(node_to_cluster).astype(int)

cluster_geo = grid.groupby("cluster")["geo_score_sm"].mean().reset_index(name="cluster_geo_mean")
cluster_rf = grid.groupby("cluster")["rf_score_sm"].mean().reset_index(name="cluster_rf_mean")
cluster_coinc = grid.groupby("cluster")["coincidence_score"].mean().reset_index(name="cluster_coinc_mean")
cluster_stats = cluster_geo.merge(cluster_rf, on="cluster", how="outer").merge(cluster_coinc, on="cluster", how="outer")

if use_supervised:
    cluster_hits = grid.groupby("cluster").agg(cells=("cell_id", "count"), positives=("target", "sum")).reset_index()
    cluster_hits["hit_rate"] = cluster_hits["positives"] / cluster_hits["cells"]
    cluster_stats = cluster_stats.merge(cluster_hits, on="cluster", how="left")
    cluster_stats["hit_rate"] = cluster_stats["hit_rate"].fillna(0)
    cluster_stats["cluster_score"] = robust_normalize_01(
        0.35 * cluster_stats["cluster_geo_mean"] +
        0.35 * cluster_stats["cluster_rf_mean"] +
        0.15 * cluster_stats["cluster_coinc_mean"] +
        0.15 * robust_normalize_01(cluster_stats["hit_rate"], 0.0, 1.0),
        0.02, 0.98,
    )
else:
    cluster_stats["cluster_score"] = robust_normalize_01(
        0.45 * cluster_stats["cluster_geo_mean"] +
        0.35 * cluster_stats["cluster_rf_mean"] +
        0.20 * cluster_stats["cluster_coinc_mean"],
        0.02, 0.98,
    )

grid = grid.merge(cluster_stats[["cluster", "cluster_score"]], on="cluster", how="left")
grid["cluster_score"] = grid["cluster_score"].fillna(grid["geo_score_sm"])

# =========================================================
# ИТОГ
# =========================================================
grid["local_bonus"] = robust_normalize_01(
    0.38 * grid["tect_intersection"] +
    0.37 * grid["tect_magm_intersection"] +
    0.25 * grid["tect_struct_intersection"],
    0.02, 0.98,
)

mask_boundary = mask_union.boundary
grid["dist_to_boundary"] = np.array([geom.distance(mask_boundary) for geom in grid.geometry])
edge_near = np.exp(-grid["dist_to_boundary"].to_numpy() / (CELL_SIZE * 2.2))
grid["edge_false_penalty"] = robust_normalize_01(
    edge_near * np.clip(grid["tect_only_penalty"], 0, 1),
    0.02, 0.98
)

grid["prospectivity_raw"] = (
    W_GEO * grid["geo_score_sm"] +
    W_RF * grid["rf_score_sm"] +
    W_COINCIDENCE * grid["coincidence_score"] +
    W_CLUSTER * grid["cluster_score"] +
    W_LOCAL * grid["local_bonus"]
)
grid["prospectivity_raw"] = (
    grid["prospectivity_raw"]
    - 0.05 * grid["tect_only_penalty"]
    - 0.04 * grid["edge_false_penalty"]
)
grid["prospectivity_raw_sm"] = smooth_on_regular_grid(grid, "prospectivity_raw", grid_shape, passes=2)
grid["prospectivity"] = robust_normalize_01(grid["prospectivity_raw_sm"], 0.02, 0.98)

# в логике презентации: меньше = лучше
grid["prognoz"] = 1.0 - grid["prospectivity"]

top_thr = float(grid["prospectivity"].quantile(0.90))
grid["top10"] = (grid["prospectivity"] >= top_thr).astype(int)

try:
    grid["prospect_class"] = pd.qcut(
        grid["prospectivity"],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop"
    )
except Exception:
    grid["prospect_class"] = "medium"

grid = make_display_classes(grid)
grid = mark_gold_zones(grid, grid_shape, mask_union)

# =========================================================
# СОХРАНЕНИЕ
# =========================================================
if OUT_GPKG.exists():
    OUT_GPKG.unlink()

grid.to_file(OUT_GPKG, layer="forecast_grid", driver="GPKG")
if points is not None and len(points) > 0:
    points.to_file(OUT_GPKG, layer="evidence_points", driver="GPKG")

grid.drop(columns="geometry").to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

plot_prox(grid, mask, OUT_PROX)
plot_final(grid, mask, points, OUT_PNG)
plot_compare(grid, mask, points, OUT_COMPARE)

metrics = {
    "base_dir": str(BASE_DIR),
    "grid_cells": int(len(grid)),
    "cell_size": CELL_SIZE,
    "use_supervised_requested": bool(USE_SUPERVISED),
    "use_supervised_applied": bool(use_supervised),
    "positive_cells": int(grid["target"].sum()),
    "top10_threshold": float(top_thr),
    "prospectivity_min": float(np.nanmin(grid["prospectivity"])),
    "prospectivity_p05": float(np.nanquantile(grid["prospectivity"], 0.05)),
    "prospectivity_p50": float(np.nanquantile(grid["prospectivity"], 0.50)),
    "prospectivity_p95": float(np.nanquantile(grid["prospectivity"], 0.95)),
    "prospectivity_max": float(np.nanmax(grid["prospectivity"])),
    "prognoz_min": float(np.nanmin(grid["prognoz"])),
    "prognoz_max": float(np.nanmax(grid["prognoz"])),
    "display_score_min": float(np.nanmin(grid["display_score"])),
    "display_score_max": float(np.nanmax(grid["display_score"])),
    "rf_score_min": float(np.nanmin(grid["rf_score"])),
    "rf_score_max": float(np.nanmax(grid["rf_score"])),
    "rf_score_sm_min": float(np.nanmin(grid["rf_score_sm"])),
    "rf_score_sm_max": float(np.nanmax(grid["rf_score_sm"])),
    "rf_test_proxy": None if rf_test_proxy is None else float(rf_test_proxy),
    "edge_false_penalty_min": float(np.nanmin(grid["edge_false_penalty"])),
    "edge_false_penalty_max": float(np.nanmax(grid["edge_false_penalty"])),
    "gold_zone_count": int(grid["gold_zone"].sum()),
    "gold_zone_share": float(grid["gold_zone"].mean()),
    "prospectivity_q80": float(np.nanquantile(grid["prospectivity"], 0.80)),
    "prospectivity_q90": float(np.nanquantile(grid["prospectivity"], 0.90)),
    "prospectivity_q94": float(np.nanquantile(grid["prospectivity"], 0.94)),
    "point_count": int(len(points)) if points is not None else 0,
    "rf_feature_importance": feature_importance,
}
Path(OUT_JSON).write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

print("Готово.")
print(f"BASE_DIR: {BASE_DIR}")
print(f"PNG: {OUT_PNG}")
print(f"COMPARE: {OUT_COMPARE}")
print(f"GPKG: {OUT_GPKG}")
print(f"CSV: {OUT_CSV}")
print(f"JSON: {OUT_JSON}")
print("Диагностика:")
print(grid[["prospectivity", "prognoz", "display_score", "rf_score_sm", "gold_zone", "prox_magm", "tect_magm_intersection", "coincidence_score"]].describe())


Готово.
BASE_DIR: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз
PNG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v9_random_forest\gold_forecast_same_methods_fixed_v9_rf.png
COMPARE: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v9_random_forest\compare_same_methods_fixed_v9_rf.png
GPKG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v9_random_forest\gold_forecast_same_methods_fixed_v9_rf.gpkg
CSV: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v9_random_forest\grid_attributes_same_methods_fixed_v9_rf.csv
JSON: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v9_random_forest\metrics_same_methods_fixed_v9_rf.json
Диагностика:
       prospectivity       prognoz  display_score   rf_score_sm     gold_zone  \
count   15684.000000  15684.000000   15684.000000  15684.000000  15684.000000   
mean        0.418994      0.581006       0.580971      0.190389      0.005611   
std         0.254734      

In [13]:
import os
import re
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm

from pyproj import CRS
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# =========================================================
# НАСТРОЙКИ
# =========================================================
CELL_SIZE = 500
RANDOM_STATE = 42

# основной ML-метод
USE_SUPERVISED = True

# Random Forest как основной ML-блок, но не единственный источник результата
RF_N_ESTIMATORS = 400
RF_MAX_DEPTH = 12
RF_MIN_SAMPLES_LEAF = 4
RF_MIN_SAMPLES_SPLIT = 8

# итоговые веса
W_GEO = 0.54
W_RF = 0.27
W_COINCIDENCE = 0.14
W_LOCAL = 0.05

# proximity
Q_FACIES = 0.78
Q_PALEO = 0.76
Q_STRUCT = 0.72
Q_MAGM = 0.42
Q_TECT1 = 0.74
Q_TECT2 = 0.74

# визуализация и gold-зоны
N_DISPLAY_CLASSES = 20
TOP_GOLD_Q = 0.03
TOP_LOCAL_Q = 0.96
SHOW_POINTS = False

# =========================================================
# ПУТИ
# =========================================================
def find_existing_base_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path("/mnt/data/prog_zip"),
        Path("/mnt/data"),
        Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз"),
    ]
    for base in candidates:
        shp_dir = base / "shp_dbf"
        if shp_dir.exists() and (shp_dir / "svita_new.shp").exists():
            return base
    raise FileNotFoundError("Не найден каталог с shp_dbf. Укажи BASE_DIR вручную.")

BASE_DIR = find_existing_base_dir()
SHP_DIR = BASE_DIR / "shp_dbf"
OUT_DIR = BASE_DIR / "same_methods_fixed_v10_random_forest_no_som"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAFE_ALIAS_DIR = OUT_DIR / "_safe_shp_aliases"
SAFE_ALIAS_DIR.mkdir(parents=True, exist_ok=True)

OUT_GPKG = OUT_DIR / "gold_forecast_same_methods_fixed_v10_rf.gpkg"
OUT_PNG = OUT_DIR / "gold_forecast_same_methods_fixed_v10_rf.png"
OUT_PROX = OUT_DIR / "prox_magm_same_methods_fixed_v10_rf.png"
OUT_COMPARE = OUT_DIR / "compare_same_methods_fixed_v10_rf.png"
OUT_CSV = OUT_DIR / "grid_attributes_same_methods_fixed_v10_rf.csv"
OUT_JSON = OUT_DIR / "metrics_same_methods_fixed_v10_rf.json"

# =========================================================
# ВСПОМОГАТЕЛЬНЫЕ
# =========================================================
def normalize_01(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    mn = np.nanmin(arr[finite])
    mx = np.nanmax(arr[finite])
    if np.isclose(mx, mn):
        return np.full_like(arr, 0.5, dtype=float)
    out = np.full_like(arr, np.nan, dtype=float)
    out[finite] = (arr[finite] - mn) / (mx - mn)
    return out

def robust_normalize_01(values, q_low=0.03, q_high=0.97):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    lo = np.nanquantile(arr[finite], q_low)
    hi = np.nanquantile(arr[finite], q_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or np.isclose(lo, hi):
        return normalize_01(arr)
    out = (arr - lo) / (hi - lo)
    return np.clip(out, 0, 1)

def read_sidecar_proj4(shp_path: Path):
    sidecar = shp_path.with_name(shp_path.stem + "_shp.pj4")
    if sidecar.exists():
        txt = sidecar.read_text(encoding="utf-8", errors="ignore")
        m = re.search(r"pj4=(.+)", txt)
        if m:
            return m.group(1).strip()
    return None

def prepare_ascii_aliases(shp_dir: Path, alias_dir: Path):
    aliases = {}
    stems = {}
    for name_b in os.listdir(os.fsencode(shp_dir)):
        if not name_b.endswith((b".shp", b".shx", b".dbf", b".prj", b".pj4")):
            continue
        if name_b.endswith(b"_shp.pj4"):
            continue
        base_b, ext_b = os.path.splitext(name_b)
        stems.setdefault(base_b, set()).add(ext_b)

    alias_idx = 0
    for base_b, exts in sorted(stems.items()):
        try:
            base_s = os.fsdecode(base_b)
            safe = all(ord(ch) < 128 and (ch.isalnum() or ch in "_-. ") for ch in base_s)
        except Exception:
            base_s = None
            safe = False

        if safe:
            aliases[base_s] = shp_dir / f"{base_s}.shp"
            continue

        alias_name = f"evidence_{alias_idx:02d}"
        alias_idx += 1
        for ext_b in exts:
            src = os.path.join(os.fsencode(shp_dir), base_b + ext_b)
            dst = alias_dir / f"{alias_name}{os.fsdecode(ext_b)}"
            shutil.copyfile(src, dst)
        pj4_src = os.path.join(os.fsencode(shp_dir), base_b + b"_shp.pj4")
        if os.path.exists(pj4_src):
            shutil.copyfile(pj4_src, alias_dir / f"{alias_name}_shp.pj4")
        aliases[alias_name] = alias_dir / f"{alias_name}.shp"
    return aliases

def load_layer(shp_path: Path):
    gdf = gpd.read_file(shp_path)
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    if gdf.crs is None:
        proj4 = read_sidecar_proj4(shp_path)
        if proj4:
            gdf = gdf.set_crs(CRS.from_proj4(proj4), allow_override=True)
    return gdf

def to_crs_safe(gdf, target_crs):
    if gdf.crs is None and target_crs is not None:
        return gdf.set_crs(target_crs, allow_override=True)
    if target_crs is None or gdf.crs == target_crs:
        return gdf
    return gdf.to_crs(target_crs)

def build_grid(mask, cell_size):
    mask_union = unary_union(mask.geometry)
    prepared_mask = prep(mask_union)
    minx, miny, maxx, maxy = mask.total_bounds
    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)
    rows = []
    cell_id = 0
    for r, y in enumerate(ys):
        for c, x in enumerate(xs):
            geom = box(x, y, x + cell_size, y + cell_size)
            if prepared_mask.intersects(geom):
                rows.append((cell_id, r, c, geom))
                cell_id += 1
    grid = gpd.GeoDataFrame(rows, columns=["cell_id", "row", "col", "geometry"], geometry="geometry", crs=mask.crs)
    return grid, mask_union, (len(ys), len(xs))

def add_distance_feature(grid, source, name):
    source_union = unary_union(source.geometry)
    distances = np.empty(len(grid), dtype=float)
    for i, geom in enumerate(grid.geometry.values):
        distances[i] = 0.0 if geom.intersects(source_union) else geom.distance(source_union)
    grid[name] = distances
    return grid

def distance_to_proximity(distance, transform="sqrt", q=0.75):
    d = np.asarray(distance, dtype=float)
    d = np.clip(d, 0, None)
    if transform == "sqrt":
        dt = np.sqrt(d)
    elif transform == "cbrt":
        dt = np.cbrt(d)
    elif transform == "log1p":
        dt = np.log1p(d)
    else:
        dt = d
    scale = float(np.nanquantile(dt, q))
    if not np.isfinite(scale) or scale <= 0:
        scale = max(float(np.nanmean(dt)), 1.0)
    prox = np.exp(-dt / scale)
    return np.clip(prox, 0, 1)

def smooth_on_regular_grid(grid, value_col, shape, passes=1):
    try:
        from scipy.signal import convolve2d
    except Exception:
        return grid[value_col].to_numpy()
    arr = np.full(shape, np.nan, dtype=float)
    arr[grid["row"].to_numpy(), grid["col"].to_numpy()] = grid[value_col].to_numpy()
    kernel = np.array([[1.0, 1.2, 1.0], [1.2, 3.0, 1.2], [1.0, 1.2, 1.0]], dtype=float)
    smoothed = arr.copy()
    for _ in range(max(1, passes)):
        valid = np.isfinite(smoothed).astype(float)
        filled = np.nan_to_num(smoothed, nan=0.0)
        num = convolve2d(filled, kernel, mode="same", boundary="fill", fillvalue=0)
        den = convolve2d(valid, kernel, mode="same", boundary="fill", fillvalue=0)
        with np.errstate(divide="ignore", invalid="ignore"):
            smoothed = np.where(den > 0, num / den, np.nan)
    return smoothed[grid["row"].to_numpy(), grid["col"].to_numpy()]

def collect_points(mask_crs, aliases):
    point_layers = []
    for name, shp_path in aliases.items():
        if name in {"svita_new", "fasii", "glub_raz_nw", "glub_r_nw", "gr_dol_vp_poly", "kory", "dayki_buf"}:
            continue
        gdf = load_layer(shp_path)
        gdf = to_crs_safe(gdf, mask_crs)
        geom_types = {str(x) for x in gdf.geom_type.unique()}
        if "Point" in geom_types or "MultiPoint" in geom_types:
            point_layers.append(gdf)
    if not point_layers:
        return None
    pts = pd.concat(point_layers, ignore_index=True)
    return gpd.GeoDataFrame(pts, geometry="geometry", crs=mask_crs)

def set_mask_extent(ax, mask):
    minx, miny, maxx, maxy = mask.total_bounds
    padx = (maxx - minx) * 0.02
    pady = (maxy - miny) * 0.02
    ax.set_xlim(minx - padx, maxx + padx)
    ax.set_ylim(miny - pady, maxy + pady)

def local_max_mask(grid, value_col, shape):
    try:
        from scipy.ndimage import maximum_filter
    except Exception:
        vals = grid[value_col].to_numpy()
        thr = np.nanquantile(vals, 0.98)
        return vals >= thr
    arr = np.full(shape, np.nan, dtype=float)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    vals = grid[value_col].to_numpy()
    arr[rows, cols] = vals
    filled = np.nan_to_num(arr, nan=-9999.0)
    locmax = maximum_filter(filled, size=3, mode="nearest")
    mask = np.isfinite(arr) & (filled >= locmax)
    return mask[rows, cols]

def keep_large_components(grid, bool_col, shape, min_cells=4):
    try:
        from scipy import ndimage
    except Exception:
        return grid[bool_col].to_numpy().astype(bool)
    arr = np.zeros(shape, dtype=np.uint8)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    arr[rows, cols] = grid[bool_col].to_numpy().astype(np.uint8)
    structure = np.ones((3, 3), dtype=np.uint8)
    labeled, n = ndimage.label(arr, structure=structure)
    if n == 0:
        return grid[bool_col].to_numpy().astype(bool)
    sizes = np.bincount(labeled.ravel())
    keep = np.isin(labeled, np.where(sizes >= min_cells)[0])
    keep &= labeled > 0
    return keep[rows, cols]

def make_display_classes(grid):
    prog_disp = robust_normalize_01(grid["prognoz"].to_numpy(), 0.02, 0.98)
    grid["display_score"] = prog_disp
    bins = np.linspace(0, 1, N_DISPLAY_CLASSES + 1)
    grid["display_class"] = np.digitize(prog_disp, bins[1:-1], right=False)
    return grid




def mark_gold_zones(grid, shape, mask_union):
    q_best = float(grid["prospectivity"].quantile(0.965))
    q_support = float(grid["prospectivity"].quantile(0.88))

    q_local = float(grid["local_bonus"].quantile(0.97))
    q_coinc = float(grid["coincidence_score"].quantile(0.94))
    q_magm = float(grid["prox_magm"].quantile(0.90))
    q_tmagm = float(grid["tect_magm_intersection"].quantile(0.80))

    local_peak = local_max_mask(grid, "prospectivity", shape)

    mask_boundary = mask_union.boundary
    grid["dist_to_boundary"] = np.array([geom.distance(mask_boundary) for geom in grid.geometry])

    support_zone = grid["prospectivity"] >= q_support

    core_gold = (
        support_zone &
        local_peak &
        (grid["prospectivity"] >= q_best) &
        (
            (
                (grid["local_bonus"] >= q_local) &
                (grid["tect_magm_intersection"] >= q_tmagm)
            ) |
            (
                (grid["coincidence_score"] >= q_coinc) &
                (grid["prox_magm"] >= q_magm)
            )
        )
    )

    linear_gold = (
        support_zone &
        (grid["prospectivity"] >= float(grid["prospectivity"].quantile(0.94))) &
        (grid["prox_magm"] >= q_magm) &
        (grid["tect_magm_intersection"] >= q_tmagm) &
        (grid["coincidence_score"] >= q_coinc)
    )

    edge_linear_gold = (
        linear_gold &
        (grid["dist_to_boundary"] <= CELL_SIZE * 0.75)
    )

    inner_linear_gold = (
        linear_gold &
        (grid["dist_to_boundary"] > CELL_SIZE * 0.75)
    )

    raw_gold = core_gold | edge_linear_gold | inner_linear_gold

    grid["gold_seed"] = raw_gold.astype(int)

    large = keep_large_components(grid, "gold_seed", shape, min_cells=3)
    grid["gold_zone"] = large.astype(int)

    return grid

def plot_prox(grid, mask, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    grid.plot(column="prox_magm", ax=ax, cmap="RdYlBu_r", linewidth=0, legend=True)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    set_mask_extent(ax, mask)
    ax.set_title("prox_magm")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_final(grid, mask, points, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr.N)
    grid.plot(column="display_class", ax=ax, cmap="bwr", norm=norm, linewidth=0, legend=True)
    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=ax, color="#f2d200", linewidth=0)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=ax, color="yellow", markersize=8, edgecolor="black", linewidth=0.25)
    set_mask_extent(ax, mask)
    ax.set_title("Итоговый прогноз")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_compare(grid, mask, points, out_png):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    grid.plot(column="prox_magm", ax=axes[0], cmap="RdYlBu_r", linewidth=0)
    mask.boundary.plot(ax=axes[0], color="black", linewidth=0.5)
    set_mask_extent(axes[0], mask)
    axes[0].set_title("prox_magm")
    axes[0].set_axis_off()

    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr.N)
    grid.plot(column="display_class", ax=axes[1], cmap="bwr", norm=norm, linewidth=0)
    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=axes[1], color="#f2d200", linewidth=0)
    mask.boundary.plot(ax=axes[1], color="black", linewidth=0.5)
    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=axes[1], color="yellow", markersize=8, edgecolor="black", linewidth=0.25)
    set_mask_extent(axes[1], mask)
    axes[1].set_title("Итоговый прогноз")
    axes[1].set_axis_off()

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

# =========================================================
# ЗАГРУЗКА
# =========================================================
aliases = prepare_ascii_aliases(SHP_DIR, SAFE_ALIAS_DIR)
mask = load_layer(aliases["svita_new"])
facies = to_crs_safe(load_layer(aliases["fasii"]), mask.crs)
tect1 = to_crs_safe(load_layer(aliases["glub_raz_nw"]), mask.crs)
tect2 = to_crs_safe(load_layer(aliases["glub_r_nw"]), mask.crs)
paleo = to_crs_safe(load_layer(aliases["gr_dol_vp_poly"]), mask.crs)
struct = to_crs_safe(load_layer(aliases["kory"]), mask.crs)
magm = to_crs_safe(load_layer(aliases["dayki_buf"]), mask.crs)
points = collect_points(mask.crs, aliases)

# =========================================================
# СЕТКА
# =========================================================
grid, mask_union, grid_shape = build_grid(mask, CELL_SIZE)

# =========================================================
# ДИСТАНЦИИ
# =========================================================
grid = add_distance_feature(grid, facies, "dist_facies")
grid = add_distance_feature(grid, paleo, "dist_paleo")
grid = add_distance_feature(grid, struct, "dist_struct")
grid = add_distance_feature(grid, magm, "dist_magm")
grid = add_distance_feature(grid, tect1, "dist_tect1")
grid = add_distance_feature(grid, tect2, "dist_tect2")

# =========================================================
# PROXIMITY
# =========================================================
grid["prox_facies"] = distance_to_proximity(grid["dist_facies"], transform="cbrt", q=Q_FACIES)
grid["prox_paleo"] = distance_to_proximity(grid["dist_paleo"], transform="cbrt", q=Q_PALEO)
grid["prox_struct"] = distance_to_proximity(grid["dist_struct"], transform="sqrt", q=Q_STRUCT)
grid["prox_magm"] = distance_to_proximity(grid["dist_magm"], transform="sqrt", q=Q_MAGM)
grid["prox_tect1"] = distance_to_proximity(grid["dist_tect1"], transform="cbrt", q=Q_TECT1)
grid["prox_tect2"] = distance_to_proximity(grid["dist_tect2"], transform="cbrt", q=Q_TECT2)

# =========================================================
# INTERACTIONS
# =========================================================
grid["tect_combo"] = 0.5 * (grid["prox_tect1"] + grid["prox_tect2"])
grid["tect_intersection"] = grid["prox_tect1"] * grid["prox_tect2"]
grid["tect_magm_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_magm"])
grid["tect_struct_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_struct"])
grid["paleo_struct_intersection"] = np.sqrt(grid["prox_paleo"] * grid["prox_struct"])

combo_core = (
    np.clip(grid["tect_combo"], 0, 1) *
    np.clip(0.55 * grid["prox_magm"] + 0.45 * grid["prox_struct"], 0, 1) *
    np.clip(0.60 * grid["prox_paleo"] + 0.40 * grid["prox_facies"], 0, 1)
)
grid["coincidence_score"] = robust_normalize_01(np.sqrt(np.clip(combo_core, 0, 1)), 0.02, 0.98)

tect_support = 0.40 * grid["prox_magm"] + 0.35 * grid["prox_struct"] + 0.25 * grid["prox_paleo"]
grid["tect_only_penalty"] = robust_normalize_01(np.clip(grid["tect_combo"] - tect_support, 0, 1), 0.02, 0.98)

# =========================================================
# GEO SCORE - базовая геологическая поверхность
# =========================================================
grid["geo_score_raw"] = (
    0.14 * grid["prox_tect1"] +
    0.14 * grid["prox_tect2"] +
    0.14 * grid["prox_paleo"] +
    0.11 * grid["prox_struct"] +
    0.08 * grid["prox_facies"] +
    0.08 * grid["prox_magm"] +
    0.08 * grid["tect_intersection"] +
    0.08 * grid["tect_magm_intersection"] +
    0.05 * grid["tect_struct_intersection"] +
    0.04 * grid["paleo_struct_intersection"] +
    0.10 * grid["coincidence_score"] -
    0.08 * grid["tect_only_penalty"]
)
grid["geo_score"] = robust_normalize_01(grid["geo_score_raw"], 0.02, 0.98)
grid["geo_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "geo_score", grid_shape, passes=2), 0.02, 0.98)

# =========================================================
# RANDOM FOREST AS ML BLOCK
# =========================================================
grid["target"] = 0
grid["rf_score"] = grid["geo_score_sm"]
use_supervised = False
rf_test_proxy = None
feature_importance = {}

feature_cols = [
    "prox_facies", "prox_paleo", "prox_struct", "prox_magm",
    "prox_tect1", "prox_tect2", "tect_combo", "tect_intersection",
    "tect_magm_intersection", "tect_struct_intersection",
    "paleo_struct_intersection", "coincidence_score", "tect_only_penalty",
    "geo_score_sm"
]

if USE_SUPERVISED and points is not None and len(points) > 0:
    try:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", predicate="within")
    except Exception:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", op="within")

    positive_cells = joined["cell_id"].dropna().astype(int).unique().tolist()
    grid.loc[grid["cell_id"].isin(positive_cells), "target"] = 1

    pos = int(grid["target"].sum())
    neg = int((grid["target"] == 0).sum())

    if pos >= 20 and neg > pos:
        X = grid[feature_cols].fillna(0).to_numpy()
        y = grid["target"].to_numpy()

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
        )

        rf_eval = RandomForestClassifier(
            n_estimators=RF_N_ESTIMATORS,
            max_depth=RF_MAX_DEPTH,
            min_samples_leaf=RF_MIN_SAMPLES_LEAF,
            min_samples_split=RF_MIN_SAMPLES_SPLIT,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        rf_eval.fit(X_train, y_train)
        test_prob = rf_eval.predict_proba(X_test)[:, 1]
        y_test = np.asarray(y_test)
        pos_mean = float(np.mean(test_prob[y_test == 1])) if np.any(y_test == 1) else np.nan
        neg_mean = float(np.mean(test_prob[y_test == 0])) if np.any(y_test == 0) else np.nan
        rf_test_proxy = pos_mean - neg_mean

        rf = RandomForestClassifier(
            n_estimators=RF_N_ESTIMATORS,
            max_depth=RF_MAX_DEPTH,
            min_samples_leaf=RF_MIN_SAMPLES_LEAF,
            min_samples_split=RF_MIN_SAMPLES_SPLIT,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        rf.fit(X, y)
        grid["rf_score"] = robust_normalize_01(rf.predict_proba(X)[:, 1], 0.02, 0.98)
        feature_importance = dict(zip(feature_cols, rf.feature_importances_.tolist()))
        use_supervised = True

grid["rf_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "rf_score", grid_shape, passes=2), 0.02, 0.98)
grid["ml_score"] = grid["rf_score_sm"]

# =========================================================
# БЕЗ SOM/KMEANS
# =========================================================
# cluster_score больше не выделяется отдельным unsupervised-блоком.
# Оставляем нейтральный стабилизатор, чтобы не ломать остальную логику кода.
grid["cluster_score"] = grid["geo_score_sm"].copy()

# =========================================================
# ИТОГ
# =========================================================
grid["local_bonus"] = robust_normalize_01(
    0.38 * grid["tect_intersection"] +
    0.37 * grid["tect_magm_intersection"] +
    0.25 * grid["tect_struct_intersection"],
    0.02, 0.98,
)

mask_boundary = mask_union.boundary
grid["dist_to_boundary"] = np.array([geom.distance(mask_boundary) for geom in grid.geometry])
edge_near = np.exp(-grid["dist_to_boundary"].to_numpy() / (CELL_SIZE * 2.2))
grid["edge_false_penalty"] = robust_normalize_01(
    edge_near * np.clip(grid["tect_only_penalty"], 0, 1),
    0.02, 0.98
)

grid["prospectivity_raw"] = (
    W_GEO * grid["geo_score_sm"] +
    W_RF * grid["rf_score_sm"] +
    W_COINCIDENCE * grid["coincidence_score"] +
    W_CLUSTER * W_LOCAL * grid["local_bonus"]
)
grid["prospectivity_raw"] = (
    grid["prospectivity_raw"]
    - 0.05 * grid["tect_only_penalty"]
    - 0.04 * grid["edge_false_penalty"]
)
grid["prospectivity_raw_sm"] = smooth_on_regular_grid(grid, "prospectivity_raw", grid_shape, passes=2)
grid["prospectivity"] = robust_normalize_01(grid["prospectivity_raw_sm"], 0.02, 0.98)

# в логике презентации: меньше = лучше
grid["prognoz"] = 1.0 - grid["prospectivity"]

top_thr = float(grid["prospectivity"].quantile(0.90))
grid["top10"] = (grid["prospectivity"] >= top_thr).astype(int)

try:
    grid["prospect_class"] = pd.qcut(
        grid["prospectivity"],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop"
    )
except Exception:
    grid["prospect_class"] = "medium"

grid = make_display_classes(grid)
grid = mark_gold_zones(grid, grid_shape, mask_union)

# =========================================================
# СОХРАНЕНИЕ
# =========================================================
if OUT_GPKG.exists():
    OUT_GPKG.unlink()

grid.to_file(OUT_GPKG, layer="forecast_grid", driver="GPKG")
if points is not None and len(points) > 0:
    points.to_file(OUT_GPKG, layer="evidence_points", driver="GPKG")

grid.drop(columns="geometry").to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

plot_prox(grid, mask, OUT_PROX)
plot_final(grid, mask, points, OUT_PNG)
plot_compare(grid, mask, points, OUT_COMPARE)

metrics = {
    "base_dir": str(BASE_DIR),
    "grid_cells": int(len(grid)),
    "cell_size": CELL_SIZE,
    "use_supervised_requested": bool(USE_SUPERVISED),
    "use_supervised_applied": bool(use_supervised),
    "positive_cells": int(grid["target"].sum()),
    "top10_threshold": float(top_thr),
    "prospectivity_min": float(np.nanmin(grid["prospectivity"])),
    "prospectivity_p05": float(np.nanquantile(grid["prospectivity"], 0.05)),
    "prospectivity_p50": float(np.nanquantile(grid["prospectivity"], 0.50)),
    "prospectivity_p95": float(np.nanquantile(grid["prospectivity"], 0.95)),
    "prospectivity_max": float(np.nanmax(grid["prospectivity"])),
    "prognoz_min": float(np.nanmin(grid["prognoz"])),
    "prognoz_max": float(np.nanmax(grid["prognoz"])),
    "display_score_min": float(np.nanmin(grid["display_score"])),
    "display_score_max": float(np.nanmax(grid["display_score"])),
    "rf_score_min": float(np.nanmin(grid["rf_score"])),
    "rf_score_max": float(np.nanmax(grid["rf_score"])),
    "rf_score_sm_min": float(np.nanmin(grid["rf_score_sm"])),
    "rf_score_sm_max": float(np.nanmax(grid["rf_score_sm"])),
    "rf_test_proxy": None if rf_test_proxy is None else float(rf_test_proxy),
    "edge_false_penalty_min": float(np.nanmin(grid["edge_false_penalty"])),
    "edge_false_penalty_max": float(np.nanmax(grid["edge_false_penalty"])),
    "gold_zone_count": int(grid["gold_zone"].sum()),
    "gold_zone_share": float(grid["gold_zone"].mean()),
    "prospectivity_q80": float(np.nanquantile(grid["prospectivity"], 0.80)),
    "prospectivity_q90": float(np.nanquantile(grid["prospectivity"], 0.90)),
    "prospectivity_q94": float(np.nanquantile(grid["prospectivity"], 0.94)),
    "som_used": False,
    "kmeans_used": False,
    "point_count": int(len(points)) if points is not None else 0,
    "rf_feature_importance": feature_importance,
}
Path(OUT_JSON).write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

print("Готово.")
print(f"BASE_DIR: {BASE_DIR}")
print(f"PNG: {OUT_PNG}")
print(f"COMPARE: {OUT_COMPARE}")
print(f"GPKG: {OUT_GPKG}")
print(f"CSV: {OUT_CSV}")
print(f"JSON: {OUT_JSON}")
print("Диагностика:")
print(grid[["prospectivity", "prognoz", "display_score", "rf_score_sm", "gold_zone", "prox_magm", "tect_magm_intersection", "coincidence_score"]].describe())


Готово.
BASE_DIR: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз
PNG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v10_random_forest_no_som\gold_forecast_same_methods_fixed_v10_rf.png
COMPARE: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v10_random_forest_no_som\compare_same_methods_fixed_v10_rf.png
GPKG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v10_random_forest_no_som\gold_forecast_same_methods_fixed_v10_rf.gpkg
CSV: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v10_random_forest_no_som\grid_attributes_same_methods_fixed_v10_rf.csv
JSON: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v10_random_forest_no_som\metrics_same_methods_fixed_v10_rf.json
Диагностика:
       prospectivity       prognoz  display_score   rf_score_sm     gold_zone  \
count   15684.000000  15684.000000   15684.000000  15684.000000  15684.000000   
mean        0.411144      0.588856       0.588843      0.19038

In [14]:
import os
import re
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm

from pyproj import CRS
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# =========================================================
# НАСТРОЙКИ
# =========================================================
CELL_SIZE = 500
RANDOM_STATE = 42

# основной ML-метод
USE_SUPERVISED = True

# Random Forest как основной ML-блок, но не единственный источник результата
RF_N_ESTIMATORS = 400
RF_MAX_DEPTH = 12
RF_MIN_SAMPLES_LEAF = 4
RF_MIN_SAMPLES_SPLIT = 8

# итоговые веса
W_GEO = 0.54
W_RF = 0.27
W_COINCIDENCE = 0.14
W_LOCAL = 0.05

# proximity
Q_FACIES = 0.78
Q_PALEO = 0.76
Q_STRUCT = 0.72
Q_MAGM = 0.42
Q_TECT1 = 0.74
Q_TECT2 = 0.74

# визуализация и gold-зоны
N_DISPLAY_CLASSES = 20
TOP_GOLD_Q = 0.03
TOP_LOCAL_Q = 0.96
SHOW_POINTS = False

# =========================================================
# ПУТИ
# =========================================================
def find_existing_base_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path("/mnt/data/prog_zip"),
        Path("/mnt/data"),
        Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз"),
    ]
    for base in candidates:
        shp_dir = base / "shp_dbf"
        if shp_dir.exists() and (shp_dir / "svita_new.shp").exists():
            return base
    raise FileNotFoundError("Не найден каталог с shp_dbf. Укажи BASE_DIR вручную.")

BASE_DIR = find_existing_base_dir()
SHP_DIR = BASE_DIR / "shp_dbf"
OUT_DIR = BASE_DIR / "same_methods_fixed_v11_random_forest_no_som"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAFE_ALIAS_DIR = OUT_DIR / "_safe_shp_aliases"
SAFE_ALIAS_DIR.mkdir(parents=True, exist_ok=True)

OUT_GPKG = OUT_DIR / "gold_forecast_same_methods_fixed_v11_rf.gpkg"
OUT_PNG = OUT_DIR / "gold_forecast_same_methods_fixed_v11_rf.png"
OUT_PROX = OUT_DIR / "prox_magm_same_methods_fixed_v11_rf.png"
OUT_COMPARE = OUT_DIR / "compare_same_methods_fixed_v11_rf.png"
OUT_CSV = OUT_DIR / "grid_attributes_same_methods_fixed_v11_rf.csv"
OUT_JSON = OUT_DIR / "metrics_same_methods_fixed_v11_rf.json"

# =========================================================
# ВСПОМОГАТЕЛЬНЫЕ
# =========================================================
def normalize_01(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    mn = np.nanmin(arr[finite])
    mx = np.nanmax(arr[finite])
    if np.isclose(mx, mn):
        return np.full_like(arr, 0.5, dtype=float)
    out = np.full_like(arr, np.nan, dtype=float)
    out[finite] = (arr[finite] - mn) / (mx - mn)
    return out

def robust_normalize_01(values, q_low=0.03, q_high=0.97):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    lo = np.nanquantile(arr[finite], q_low)
    hi = np.nanquantile(arr[finite], q_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or np.isclose(lo, hi):
        return normalize_01(arr)
    out = (arr - lo) / (hi - lo)
    return np.clip(out, 0, 1)

def read_sidecar_proj4(shp_path: Path):
    sidecar = shp_path.with_name(shp_path.stem + "_shp.pj4")
    if sidecar.exists():
        txt = sidecar.read_text(encoding="utf-8", errors="ignore")
        m = re.search(r"pj4=(.+)", txt)
        if m:
            return m.group(1).strip()
    return None

def prepare_ascii_aliases(shp_dir: Path, alias_dir: Path):
    aliases = {}
    stems = {}
    for name_b in os.listdir(os.fsencode(shp_dir)):
        if not name_b.endswith((b".shp", b".shx", b".dbf", b".prj", b".pj4")):
            continue
        if name_b.endswith(b"_shp.pj4"):
            continue
        base_b, ext_b = os.path.splitext(name_b)
        stems.setdefault(base_b, set()).add(ext_b)

    alias_idx = 0
    for base_b, exts in sorted(stems.items()):
        try:
            base_s = os.fsdecode(base_b)
            safe = all(ord(ch) < 128 and (ch.isalnum() or ch in "_-. ") for ch in base_s)
        except Exception:
            base_s = None
            safe = False

        if safe:
            aliases[base_s] = shp_dir / f"{base_s}.shp"
            continue

        alias_name = f"evidence_{alias_idx:02d}"
        alias_idx += 1
        for ext_b in exts:
            src = os.path.join(os.fsencode(shp_dir), base_b + ext_b)
            dst = alias_dir / f"{alias_name}{os.fsdecode(ext_b)}"
            shutil.copyfile(src, dst)
        pj4_src = os.path.join(os.fsencode(shp_dir), base_b + b"_shp.pj4")
        if os.path.exists(pj4_src):
            shutil.copyfile(pj4_src, alias_dir / f"{alias_name}_shp.pj4")
        aliases[alias_name] = alias_dir / f"{alias_name}.shp"
    return aliases

def load_layer(shp_path: Path):
    gdf = gpd.read_file(shp_path)
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    if gdf.crs is None:
        proj4 = read_sidecar_proj4(shp_path)
        if proj4:
            gdf = gdf.set_crs(CRS.from_proj4(proj4), allow_override=True)
    return gdf

def to_crs_safe(gdf, target_crs):
    if gdf.crs is None and target_crs is not None:
        return gdf.set_crs(target_crs, allow_override=True)
    if target_crs is None or gdf.crs == target_crs:
        return gdf
    return gdf.to_crs(target_crs)

def build_grid(mask, cell_size):
    mask_union = unary_union(mask.geometry)
    prepared_mask = prep(mask_union)
    minx, miny, maxx, maxy = mask.total_bounds
    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)
    rows = []
    cell_id = 0
    for r, y in enumerate(ys):
        for c, x in enumerate(xs):
            geom = box(x, y, x + cell_size, y + cell_size)
            if prepared_mask.intersects(geom):
                rows.append((cell_id, r, c, geom))
                cell_id += 1
    grid = gpd.GeoDataFrame(rows, columns=["cell_id", "row", "col", "geometry"], geometry="geometry", crs=mask.crs)
    return grid, mask_union, (len(ys), len(xs))

def add_distance_feature(grid, source, name):
    source_union = unary_union(source.geometry)
    distances = np.empty(len(grid), dtype=float)
    for i, geom in enumerate(grid.geometry.values):
        distances[i] = 0.0 if geom.intersects(source_union) else geom.distance(source_union)
    grid[name] = distances
    return grid

def distance_to_proximity(distance, transform="sqrt", q=0.75):
    d = np.asarray(distance, dtype=float)
    d = np.clip(d, 0, None)
    if transform == "sqrt":
        dt = np.sqrt(d)
    elif transform == "cbrt":
        dt = np.cbrt(d)
    elif transform == "log1p":
        dt = np.log1p(d)
    else:
        dt = d
    scale = float(np.nanquantile(dt, q))
    if not np.isfinite(scale) or scale <= 0:
        scale = max(float(np.nanmean(dt)), 1.0)
    prox = np.exp(-dt / scale)
    return np.clip(prox, 0, 1)

def smooth_on_regular_grid(grid, value_col, shape, passes=1):
    try:
        from scipy.signal import convolve2d
    except Exception:
        return grid[value_col].to_numpy()
    arr = np.full(shape, np.nan, dtype=float)
    arr[grid["row"].to_numpy(), grid["col"].to_numpy()] = grid[value_col].to_numpy()
    kernel = np.array([[1.0, 1.2, 1.0], [1.2, 3.0, 1.2], [1.0, 1.2, 1.0]], dtype=float)
    smoothed = arr.copy()
    for _ in range(max(1, passes)):
        valid = np.isfinite(smoothed).astype(float)
        filled = np.nan_to_num(smoothed, nan=0.0)
        num = convolve2d(filled, kernel, mode="same", boundary="fill", fillvalue=0)
        den = convolve2d(valid, kernel, mode="same", boundary="fill", fillvalue=0)
        with np.errstate(divide="ignore", invalid="ignore"):
            smoothed = np.where(den > 0, num / den, np.nan)
    return smoothed[grid["row"].to_numpy(), grid["col"].to_numpy()]

def collect_points(mask_crs, aliases):
    point_layers = []
    for name, shp_path in aliases.items():
        if name in {"svita_new", "fasii", "glub_raz_nw", "glub_r_nw", "gr_dol_vp_poly", "kory", "dayki_buf"}:
            continue
        gdf = load_layer(shp_path)
        gdf = to_crs_safe(gdf, mask_crs)
        geom_types = {str(x) for x in gdf.geom_type.unique()}
        if "Point" in geom_types or "MultiPoint" in geom_types:
            point_layers.append(gdf)
    if not point_layers:
        return None
    pts = pd.concat(point_layers, ignore_index=True)
    return gpd.GeoDataFrame(pts, geometry="geometry", crs=mask_crs)

def set_mask_extent(ax, mask):
    minx, miny, maxx, maxy = mask.total_bounds
    padx = (maxx - minx) * 0.02
    pady = (maxy - miny) * 0.02
    ax.set_xlim(minx - padx, maxx + padx)
    ax.set_ylim(miny - pady, maxy + pady)

def local_max_mask(grid, value_col, shape):
    try:
        from scipy.ndimage import maximum_filter
    except Exception:
        vals = grid[value_col].to_numpy()
        thr = np.nanquantile(vals, 0.98)
        return vals >= thr
    arr = np.full(shape, np.nan, dtype=float)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    vals = grid[value_col].to_numpy()
    arr[rows, cols] = vals
    filled = np.nan_to_num(arr, nan=-9999.0)
    locmax = maximum_filter(filled, size=3, mode="nearest")
    mask = np.isfinite(arr) & (filled >= locmax)
    return mask[rows, cols]

def keep_large_components(grid, bool_col, shape, min_cells=4):
    try:
        from scipy import ndimage
    except Exception:
        return grid[bool_col].to_numpy().astype(bool)
    arr = np.zeros(shape, dtype=np.uint8)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    arr[rows, cols] = grid[bool_col].to_numpy().astype(np.uint8)
    structure = np.ones((3, 3), dtype=np.uint8)
    labeled, n = ndimage.label(arr, structure=structure)
    if n == 0:
        return grid[bool_col].to_numpy().astype(bool)
    sizes = np.bincount(labeled.ravel())
    keep = np.isin(labeled, np.where(sizes >= min_cells)[0])
    keep &= labeled > 0
    return keep[rows, cols]

def make_display_classes(grid):
    prog_disp = robust_normalize_01(grid["prognoz"].to_numpy(), 0.02, 0.98)
    grid["display_score"] = prog_disp
    bins = np.linspace(0, 1, N_DISPLAY_CLASSES + 1)
    grid["display_class"] = np.digitize(prog_disp, bins[1:-1], right=False)
    return grid





def mark_gold_zones(grid, shape, mask_union):
    # Без SOM оставляем gold-mask ближе к строгой v9,
    # чтобы не раздувать жёлтые зоны внутри широких синих областей.
    q_best = float(grid["prospectivity"].quantile(0.97))
    q_support = float(grid["prospectivity"].quantile(0.88))

    q_local = float(grid["local_bonus"].quantile(0.97))
    q_coinc = float(grid["coincidence_score"].quantile(0.94))
    q_magm = float(grid["prox_magm"].quantile(0.90))
    q_tmagm = float(grid["tect_magm_intersection"].quantile(0.84))

    local_peak = local_max_mask(grid, "prospectivity", shape)

    mask_boundary = mask_union.boundary
    grid["dist_to_boundary"] = np.array([geom.distance(mask_boundary) for geom in grid.geometry])

    support_zone = grid["prospectivity"] >= q_support

    core_gold = (
        support_zone &
        local_peak &
        (grid["prospectivity"] >= q_best) &
        (
            (
                (grid["local_bonus"] >= q_local) &
                (grid["tect_magm_intersection"] >= q_tmagm)
            ) |
            (
                (grid["coincidence_score"] >= q_coinc) &
                (grid["prox_magm"] >= q_magm)
            )
        )
    )

    linear_gold = (
        support_zone &
        (grid["prospectivity"] >= float(grid["prospectivity"].quantile(0.94))) &
        (grid["prox_magm"] >= q_magm) &
        (grid["tect_magm_intersection"] >= q_tmagm) &
        (grid["coincidence_score"] >= q_coinc)
    )

    # Только действительно сильные краевые линейные зоны
    edge_linear_gold = (
        linear_gold &
        (grid["dist_to_boundary"] <= CELL_SIZE * 0.75)
    )

    raw_gold = core_gold | edge_linear_gold

    grid["gold_seed"] = raw_gold.astype(int)

    # Возвращаем более строгую фильтрацию, чтобы убрать разрастание gold-mask
    large = keep_large_components(grid, "gold_seed", shape, min_cells=4)
    grid["gold_zone"] = large.astype(int)

    return grid

def plot_prox(grid, mask, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    grid.plot(column="prox_magm", ax=ax, cmap="RdYlBu_r", linewidth=0, legend=True)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    set_mask_extent(ax, mask)
    ax.set_title("prox_magm")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_final(grid, mask, points, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr.N)
    grid.plot(column="display_class", ax=ax, cmap="bwr", norm=norm, linewidth=0, legend=True)
    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=ax, color="#f2d200", linewidth=0)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=ax, color="yellow", markersize=8, edgecolor="black", linewidth=0.25)
    set_mask_extent(ax, mask)
    ax.set_title("Итоговый прогноз")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_compare(grid, mask, points, out_png):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    grid.plot(column="prox_magm", ax=axes[0], cmap="RdYlBu_r", linewidth=0)
    mask.boundary.plot(ax=axes[0], color="black", linewidth=0.5)
    set_mask_extent(axes[0], mask)
    axes[0].set_title("prox_magm")
    axes[0].set_axis_off()

    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr.N)
    grid.plot(column="display_class", ax=axes[1], cmap="bwr", norm=norm, linewidth=0)
    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=axes[1], color="#f2d200", linewidth=0)
    mask.boundary.plot(ax=axes[1], color="black", linewidth=0.5)
    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=axes[1], color="yellow", markersize=8, edgecolor="black", linewidth=0.25)
    set_mask_extent(axes[1], mask)
    axes[1].set_title("Итоговый прогноз")
    axes[1].set_axis_off()

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

# =========================================================
# ЗАГРУЗКА
# =========================================================
aliases = prepare_ascii_aliases(SHP_DIR, SAFE_ALIAS_DIR)
mask = load_layer(aliases["svita_new"])
facies = to_crs_safe(load_layer(aliases["fasii"]), mask.crs)
tect1 = to_crs_safe(load_layer(aliases["glub_raz_nw"]), mask.crs)
tect2 = to_crs_safe(load_layer(aliases["glub_r_nw"]), mask.crs)
paleo = to_crs_safe(load_layer(aliases["gr_dol_vp_poly"]), mask.crs)
struct = to_crs_safe(load_layer(aliases["kory"]), mask.crs)
magm = to_crs_safe(load_layer(aliases["dayki_buf"]), mask.crs)
points = collect_points(mask.crs, aliases)

# =========================================================
# СЕТКА
# =========================================================
grid, mask_union, grid_shape = build_grid(mask, CELL_SIZE)

# =========================================================
# ДИСТАНЦИИ
# =========================================================
grid = add_distance_feature(grid, facies, "dist_facies")
grid = add_distance_feature(grid, paleo, "dist_paleo")
grid = add_distance_feature(grid, struct, "dist_struct")
grid = add_distance_feature(grid, magm, "dist_magm")
grid = add_distance_feature(grid, tect1, "dist_tect1")
grid = add_distance_feature(grid, tect2, "dist_tect2")

# =========================================================
# PROXIMITY
# =========================================================
grid["prox_facies"] = distance_to_proximity(grid["dist_facies"], transform="cbrt", q=Q_FACIES)
grid["prox_paleo"] = distance_to_proximity(grid["dist_paleo"], transform="cbrt", q=Q_PALEO)
grid["prox_struct"] = distance_to_proximity(grid["dist_struct"], transform="sqrt", q=Q_STRUCT)
grid["prox_magm"] = distance_to_proximity(grid["dist_magm"], transform="sqrt", q=Q_MAGM)
grid["prox_tect1"] = distance_to_proximity(grid["dist_tect1"], transform="cbrt", q=Q_TECT1)
grid["prox_tect2"] = distance_to_proximity(grid["dist_tect2"], transform="cbrt", q=Q_TECT2)

# =========================================================
# INTERACTIONS
# =========================================================
grid["tect_combo"] = 0.5 * (grid["prox_tect1"] + grid["prox_tect2"])
grid["tect_intersection"] = grid["prox_tect1"] * grid["prox_tect2"]
grid["tect_magm_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_magm"])
grid["tect_struct_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_struct"])
grid["paleo_struct_intersection"] = np.sqrt(grid["prox_paleo"] * grid["prox_struct"])

combo_core = (
    np.clip(grid["tect_combo"], 0, 1) *
    np.clip(0.55 * grid["prox_magm"] + 0.45 * grid["prox_struct"], 0, 1) *
    np.clip(0.60 * grid["prox_paleo"] + 0.40 * grid["prox_facies"], 0, 1)
)
grid["coincidence_score"] = robust_normalize_01(np.sqrt(np.clip(combo_core, 0, 1)), 0.02, 0.98)

tect_support = 0.40 * grid["prox_magm"] + 0.35 * grid["prox_struct"] + 0.25 * grid["prox_paleo"]
grid["tect_only_penalty"] = robust_normalize_01(np.clip(grid["tect_combo"] - tect_support, 0, 1), 0.02, 0.98)

# =========================================================
# GEO SCORE - базовая геологическая поверхность
# =========================================================
grid["geo_score_raw"] = (
    0.14 * grid["prox_tect1"] +
    0.14 * grid["prox_tect2"] +
    0.14 * grid["prox_paleo"] +
    0.11 * grid["prox_struct"] +
    0.08 * grid["prox_facies"] +
    0.08 * grid["prox_magm"] +
    0.08 * grid["tect_intersection"] +
    0.08 * grid["tect_magm_intersection"] +
    0.05 * grid["tect_struct_intersection"] +
    0.04 * grid["paleo_struct_intersection"] +
    0.10 * grid["coincidence_score"] -
    0.08 * grid["tect_only_penalty"]
)
grid["geo_score"] = robust_normalize_01(grid["geo_score_raw"], 0.02, 0.98)
grid["geo_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "geo_score", grid_shape, passes=2), 0.02, 0.98)

# =========================================================
# RANDOM FOREST AS ML BLOCK
# =========================================================
grid["target"] = 0
grid["rf_score"] = grid["geo_score_sm"]
use_supervised = False
rf_test_proxy = None
feature_importance = {}

feature_cols = [
    "prox_facies", "prox_paleo", "prox_struct", "prox_magm",
    "prox_tect1", "prox_tect2", "tect_combo", "tect_intersection",
    "tect_magm_intersection", "tect_struct_intersection",
    "paleo_struct_intersection", "coincidence_score", "tect_only_penalty",
    "geo_score_sm"
]

if USE_SUPERVISED and points is not None and len(points) > 0:
    try:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", predicate="within")
    except Exception:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", op="within")

    positive_cells = joined["cell_id"].dropna().astype(int).unique().tolist()
    grid.loc[grid["cell_id"].isin(positive_cells), "target"] = 1

    pos = int(grid["target"].sum())
    neg = int((grid["target"] == 0).sum())

    if pos >= 20 and neg > pos:
        X = grid[feature_cols].fillna(0).to_numpy()
        y = grid["target"].to_numpy()

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
        )

        rf_eval = RandomForestClassifier(
            n_estimators=RF_N_ESTIMATORS,
            max_depth=RF_MAX_DEPTH,
            min_samples_leaf=RF_MIN_SAMPLES_LEAF,
            min_samples_split=RF_MIN_SAMPLES_SPLIT,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        rf_eval.fit(X_train, y_train)
        test_prob = rf_eval.predict_proba(X_test)[:, 1]
        y_test = np.asarray(y_test)
        pos_mean = float(np.mean(test_prob[y_test == 1])) if np.any(y_test == 1) else np.nan
        neg_mean = float(np.mean(test_prob[y_test == 0])) if np.any(y_test == 0) else np.nan
        rf_test_proxy = pos_mean - neg_mean

        rf = RandomForestClassifier(
            n_estimators=RF_N_ESTIMATORS,
            max_depth=RF_MAX_DEPTH,
            min_samples_leaf=RF_MIN_SAMPLES_LEAF,
            min_samples_split=RF_MIN_SAMPLES_SPLIT,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        rf.fit(X, y)
        grid["rf_score"] = robust_normalize_01(rf.predict_proba(X)[:, 1], 0.02, 0.98)
        feature_importance = dict(zip(feature_cols, rf.feature_importances_.tolist()))
        use_supervised = True

grid["rf_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "rf_score", grid_shape, passes=2), 0.02, 0.98)
grid["ml_score"] = grid["rf_score_sm"]

# =========================================================
# БЕЗ SOM/KMEANS
# =========================================================
# cluster_score больше не выделяется отдельным unsupervised-блоком.
# Оставляем нейтральный стабилизатор, чтобы не ломать остальную логику кода.
grid["cluster_score"] = grid["geo_score_sm"].copy()

# =========================================================
# ИТОГ
# =========================================================
grid["local_bonus"] = robust_normalize_01(
    0.38 * grid["tect_intersection"] +
    0.37 * grid["tect_magm_intersection"] +
    0.25 * grid["tect_struct_intersection"],
    0.02, 0.98,
)

mask_boundary = mask_union.boundary
grid["dist_to_boundary"] = np.array([geom.distance(mask_boundary) for geom in grid.geometry])
edge_near = np.exp(-grid["dist_to_boundary"].to_numpy() / (CELL_SIZE * 2.2))
grid["edge_false_penalty"] = robust_normalize_01(
    edge_near * np.clip(grid["tect_only_penalty"], 0, 1),
    0.02, 0.98
)

grid["prospectivity_raw"] = (
    W_GEO * grid["geo_score_sm"] +
    W_RF * grid["rf_score_sm"] +
    W_COINCIDENCE * grid["coincidence_score"] +
    W_CLUSTER * W_LOCAL * grid["local_bonus"]
)
grid["prospectivity_raw"] = (
    grid["prospectivity_raw"]
    - 0.05 * grid["tect_only_penalty"]
    - 0.04 * grid["edge_false_penalty"]
)
grid["prospectivity_raw_sm"] = smooth_on_regular_grid(grid, "prospectivity_raw", grid_shape, passes=2)
grid["prospectivity"] = robust_normalize_01(grid["prospectivity_raw_sm"], 0.02, 0.98)

# в логике презентации: меньше = лучше
grid["prognoz"] = 1.0 - grid["prospectivity"]

top_thr = float(grid["prospectivity"].quantile(0.90))
grid["top10"] = (grid["prospectivity"] >= top_thr).astype(int)

try:
    grid["prospect_class"] = pd.qcut(
        grid["prospectivity"],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop"
    )
except Exception:
    grid["prospect_class"] = "medium"

grid = make_display_classes(grid)
grid = mark_gold_zones(grid, grid_shape, mask_union)

# =========================================================
# СОХРАНЕНИЕ
# =========================================================
if OUT_GPKG.exists():
    OUT_GPKG.unlink()

grid.to_file(OUT_GPKG, layer="forecast_grid", driver="GPKG")
if points is not None and len(points) > 0:
    points.to_file(OUT_GPKG, layer="evidence_points", driver="GPKG")

grid.drop(columns="geometry").to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

plot_prox(grid, mask, OUT_PROX)
plot_final(grid, mask, points, OUT_PNG)
plot_compare(grid, mask, points, OUT_COMPARE)

metrics = {
    "base_dir": str(BASE_DIR),
    "grid_cells": int(len(grid)),
    "cell_size": CELL_SIZE,
    "use_supervised_requested": bool(USE_SUPERVISED),
    "use_supervised_applied": bool(use_supervised),
    "positive_cells": int(grid["target"].sum()),
    "top10_threshold": float(top_thr),
    "prospectivity_min": float(np.nanmin(grid["prospectivity"])),
    "prospectivity_p05": float(np.nanquantile(grid["prospectivity"], 0.05)),
    "prospectivity_p50": float(np.nanquantile(grid["prospectivity"], 0.50)),
    "prospectivity_p95": float(np.nanquantile(grid["prospectivity"], 0.95)),
    "prospectivity_max": float(np.nanmax(grid["prospectivity"])),
    "prognoz_min": float(np.nanmin(grid["prognoz"])),
    "prognoz_max": float(np.nanmax(grid["prognoz"])),
    "display_score_min": float(np.nanmin(grid["display_score"])),
    "display_score_max": float(np.nanmax(grid["display_score"])),
    "rf_score_min": float(np.nanmin(grid["rf_score"])),
    "rf_score_max": float(np.nanmax(grid["rf_score"])),
    "rf_score_sm_min": float(np.nanmin(grid["rf_score_sm"])),
    "rf_score_sm_max": float(np.nanmax(grid["rf_score_sm"])),
    "rf_test_proxy": None if rf_test_proxy is None else float(rf_test_proxy),
    "edge_false_penalty_min": float(np.nanmin(grid["edge_false_penalty"])),
    "edge_false_penalty_max": float(np.nanmax(grid["edge_false_penalty"])),
    "gold_zone_count": int(grid["gold_zone"].sum()),
    "gold_zone_share": float(grid["gold_zone"].mean()),
    "prospectivity_q80": float(np.nanquantile(grid["prospectivity"], 0.80)),
    "prospectivity_q90": float(np.nanquantile(grid["prospectivity"], 0.90)),
    "prospectivity_q94": float(np.nanquantile(grid["prospectivity"], 0.94)),
    "som_used": False,
    "kmeans_used": False,
    "point_count": int(len(points)) if points is not None else 0,
    "rf_feature_importance": feature_importance,
}
Path(OUT_JSON).write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

print("Готово.")
print(f"BASE_DIR: {BASE_DIR}")
print(f"PNG: {OUT_PNG}")
print(f"COMPARE: {OUT_COMPARE}")
print(f"GPKG: {OUT_GPKG}")
print(f"CSV: {OUT_CSV}")
print(f"JSON: {OUT_JSON}")
print("Диагностика:")
print(grid[["prospectivity", "prognoz", "display_score", "rf_score_sm", "gold_zone", "prox_magm", "tect_magm_intersection", "coincidence_score"]].describe())


Готово.
BASE_DIR: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз
PNG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v11_random_forest_no_som\gold_forecast_same_methods_fixed_v11_rf.png
COMPARE: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v11_random_forest_no_som\compare_same_methods_fixed_v11_rf.png
GPKG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v11_random_forest_no_som\gold_forecast_same_methods_fixed_v11_rf.gpkg
CSV: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v11_random_forest_no_som\grid_attributes_same_methods_fixed_v11_rf.csv
JSON: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\same_methods_fixed_v11_random_forest_no_som\metrics_same_methods_fixed_v11_rf.json
Диагностика:
       prospectivity       prognoz  display_score   rf_score_sm     gold_zone  \
count   15684.000000  15684.000000   15684.000000  15684.000000  15684.000000   
mean        0.411144      0.588856       0.588843      0.19038

In [15]:
import json
import os
import re
import shutil
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import BoundaryNorm
from matplotlib.patches import Patch
from pyproj import CRS
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# =========================
# SETTINGS
# =========================
CELL_SIZE = 500
RANDOM_STATE = 42
USE_SUPERVISED = True

RF_N_ESTIMATORS = 400
RF_MAX_DEPTH = 12
RF_MIN_SAMPLES_LEAF = 4
RF_MIN_SAMPLES_SPLIT = 8

W_GEO = 0.54
W_RF = 0.27
W_COINCIDENCE = 0.14
W_LOCAL = 0.05

Q_FACIES = 0.78
Q_PALEO = 0.76
Q_STRUCT = 0.72
Q_MAGM = 0.42
Q_TECT1 = 0.74
Q_TECT2 = 0.74

N_DISPLAY_CLASSES = 20
SHOW_POINTS = False

# =========================
# PATHS
# =========================
def find_base_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path("/mnt/data/prog_zip"),
        Path("/mnt/data"),
        Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз"),
    ]
    for base in candidates:
        shp_dir = base / "shp_dbf"
        if shp_dir.exists() and (shp_dir / "svita_new.shp").exists():
            return base
    raise FileNotFoundError("Не найден каталог с shp_dbf.")

BASE_DIR = find_base_dir()
SHP_DIR = BASE_DIR / "shp_dbf"
OUT_DIR = BASE_DIR / "rf_clean_result"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAFE_ALIAS_DIR = OUT_DIR / "_safe_aliases"
SAFE_ALIAS_DIR.mkdir(parents=True, exist_ok=True)

OUT_GPKG = OUT_DIR / "forecast_rf_clean.gpkg"
OUT_PNG = OUT_DIR / "forecast_rf_clean.png"
OUT_COMPARE = OUT_DIR / "compare_rf_clean.png"
OUT_PROX = OUT_DIR / "prox_magm_rf_clean.png"
OUT_CSV = OUT_DIR / "grid_rf_clean.csv"
OUT_JSON = OUT_DIR / "metrics_rf_clean.json"

# =========================
# HELPERS
# =========================
def normalize_01(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    mn = np.nanmin(arr[finite])
    mx = np.nanmax(arr[finite])
    if np.isclose(mx, mn):
        return np.full_like(arr, 0.5, dtype=float)
    out = np.full_like(arr, np.nan, dtype=float)
    out[finite] = (arr[finite] - mn) / (mx - mn)
    return out

def robust_normalize_01(values, q_low=0.03, q_high=0.97):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    lo = np.nanquantile(arr[finite], q_low)
    hi = np.nanquantile(arr[finite], q_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or np.isclose(lo, hi):
        return normalize_01(arr)
    return np.clip((arr - lo) / (hi - lo), 0, 1)

def smooth_on_regular_grid(grid, value_col, shape, passes=1):
    try:
        from scipy.signal import convolve2d
    except Exception:
        return grid[value_col].to_numpy()
    arr = np.full(shape, np.nan, dtype=float)
    arr[grid["row"].to_numpy(), grid["col"].to_numpy()] = grid[value_col].to_numpy()
    kernel = np.array([[1.0, 1.2, 1.0], [1.2, 3.0, 1.2], [1.0, 1.2, 1.0]], dtype=float)
    smoothed = arr.copy()
    for _ in range(max(1, passes)):
        valid = np.isfinite(smoothed).astype(float)
        filled = np.nan_to_num(smoothed, nan=0.0)
        num = convolve2d(filled, kernel, mode="same", boundary="fill", fillvalue=0)
        den = convolve2d(valid, kernel, mode="same", boundary="fill", fillvalue=0)
        smoothed = np.where(den > 0, num / den, np.nan)
    return smoothed[grid["row"].to_numpy(), grid["col"].to_numpy()]

def local_max_mask(grid, value_col, shape):
    try:
        from scipy.ndimage import maximum_filter
    except Exception:
        vals = grid[value_col].to_numpy()
        thr = np.nanquantile(vals, 0.98)
        return vals >= thr
    arr = np.full(shape, np.nan, dtype=float)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    vals = grid[value_col].to_numpy()
    arr[rows, cols] = vals
    filled = np.nan_to_num(arr, nan=-9999.0)
    locmax = maximum_filter(filled, size=3, mode="nearest")
    return (np.isfinite(arr) & (filled >= locmax))[rows, cols]

def keep_large_components(grid, bool_col, shape, min_cells=4):
    try:
        from scipy import ndimage
    except Exception:
        return grid[bool_col].to_numpy().astype(bool)
    arr = np.zeros(shape, dtype=np.uint8)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    arr[rows, cols] = grid[bool_col].to_numpy().astype(np.uint8)
    structure = np.ones((3, 3), dtype=np.uint8)
    labeled, _ = ndimage.label(arr, structure=structure)
    sizes = np.bincount(labeled.ravel())
    keep = np.isin(labeled, np.where(sizes >= min_cells)[0]) & (labeled > 0)
    return keep[rows, cols]

def read_sidecar_proj4(shp_path: Path):
    sidecar = shp_path.with_name(shp_path.stem + "_shp.pj4")
    if sidecar.exists():
        txt = sidecar.read_text(encoding="utf-8", errors="ignore")
        m = re.search(r"pj4=(.+)", txt)
        if m:
            return m.group(1).strip()
    return None

def prepare_ascii_aliases(shp_dir: Path, alias_dir: Path):
    aliases, stems = {}, {}
    for name_b in os.listdir(os.fsencode(shp_dir)):
        if not name_b.endswith((b".shp", b".shx", b".dbf", b".prj", b".pj4")) or name_b.endswith(b"_shp.pj4"):
            continue
        base_b, ext_b = os.path.splitext(name_b)
        stems.setdefault(base_b, set()).add(ext_b)

    alias_idx = 0
    for base_b, exts in sorted(stems.items()):
        try:
            base_s = os.fsdecode(base_b)
            safe = all(ord(ch) < 128 and (ch.isalnum() or ch in "_-. ") for ch in base_s)
        except Exception:
            safe = False
            base_s = None

        if safe:
            aliases[base_s] = shp_dir / f"{base_s}.shp"
            continue

        alias = f"layer_{alias_idx:02d}"
        alias_idx += 1
        for ext_b in exts:
            src = os.path.join(os.fsencode(shp_dir), base_b + ext_b)
            dst = alias_dir / f"{alias}{os.fsdecode(ext_b)}"
            shutil.copyfile(src, dst)
        pj4_src = os.path.join(os.fsencode(shp_dir), base_b + b"_shp.pj4")
        if os.path.exists(pj4_src):
            shutil.copyfile(pj4_src, alias_dir / f"{alias}_shp.pj4")
        aliases[alias] = alias_dir / f"{alias}.shp"
    return aliases

def load_layer(path: Path):
    gdf = gpd.read_file(path)
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    if gdf.crs is None:
        proj4 = read_sidecar_proj4(path)
        if proj4:
            gdf = gdf.set_crs(CRS.from_proj4(proj4), allow_override=True)
    return gdf

def to_crs_safe(gdf, target_crs):
    if gdf.crs is None and target_crs is not None:
        return gdf.set_crs(target_crs, allow_override=True)
    if target_crs is None or gdf.crs == target_crs:
        return gdf
    return gdf.to_crs(target_crs)

def build_grid(mask, cell_size):
    mask_union = unary_union(mask.geometry)
    prepared_mask = prep(mask_union)
    minx, miny, maxx, maxy = mask.total_bounds
    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)
    rows = []
    cell_id = 0
    for r, y in enumerate(ys):
        for c, x in enumerate(xs):
            geom = box(x, y, x + cell_size, y + cell_size)
            if prepared_mask.intersects(geom):
                rows.append((cell_id, r, c, geom))
                cell_id += 1
    grid = gpd.GeoDataFrame(rows, columns=["cell_id", "row", "col", "geometry"], geometry="geometry", crs=mask.crs)
    return grid, mask_union, (len(ys), len(xs))

def add_distance_feature(grid, source, name):
    source_union = unary_union(source.geometry)
    d = np.empty(len(grid), dtype=float)
    for i, geom in enumerate(grid.geometry.values):
        d[i] = 0.0 if geom.intersects(source_union) else geom.distance(source_union)
    grid[name] = d
    return grid

def distance_to_proximity(distance, transform="sqrt", q=0.75):
    d = np.asarray(distance, dtype=float)
    d = np.clip(d, 0, None)
    if transform == "sqrt":
        t = np.sqrt(d)
    elif transform == "cbrt":
        t = np.cbrt(d)
    else:
        t = d
    scale = float(np.nanquantile(t, q))
    if not np.isfinite(scale) or scale <= 0:
        scale = max(float(np.nanmean(t)), 1.0)
    return np.clip(np.exp(-t / scale), 0, 1)

def collect_points(mask_crs, aliases):
    base_names = {"svita_new", "fasii", "glub_raz_nw", "glub_r_nw", "gr_dol_vp_poly", "kory", "dayki_buf"}
    layers = []
    for name, shp_path in aliases.items():
        if name in base_names:
            continue
        gdf = to_crs_safe(load_layer(shp_path), mask_crs)
        types = {str(x) for x in gdf.geom_type.unique()}
        if "Point" in types or "MultiPoint" in types:
            layers.append(gdf)
    if not layers:
        return None
    pts = pd.concat(layers, ignore_index=True)
    return gpd.GeoDataFrame(pts, geometry="geometry", crs=mask_crs)

def make_display_classes(grid):
    disp = robust_normalize_01(grid["prospectivity"].to_numpy(), 0.02, 0.98)
    grid["display_score"] = disp
    bins = np.linspace(0, 1, N_DISPLAY_CLASSES + 1)
    grid["display_class"] = np.digitize(disp, bins[1:-1], right=False)
    return grid

def set_mask_extent(ax, mask):
    minx, miny, maxx, maxy = mask.total_bounds
    padx = (maxx - minx) * 0.02
    pady = (maxy - miny) * 0.02
    ax.set_xlim(minx - padx, maxx + padx)
    ax.set_ylim(miny - pady, maxy + pady)

def mark_gold_zones(grid, shape, mask_union):
    q_best = float(grid["prospectivity"].quantile(0.97))
    q_support = float(grid["prospectivity"].quantile(0.88))
    q_local = float(grid["local_bonus"].quantile(0.97))
    q_coinc = float(grid["coincidence_score"].quantile(0.94))
    q_magm = float(grid["prox_magm"].quantile(0.90))
    q_tmagm = float(grid["tect_magm_intersection"].quantile(0.84))

    local_peak = local_max_mask(grid, "prospectivity", shape)
    mask_boundary = mask_union.boundary
    grid["dist_to_boundary"] = np.array([geom.distance(mask_boundary) for geom in grid.geometry])

    support_zone = grid["prospectivity"] >= q_support

    core_gold = (
        support_zone &
        local_peak &
        (grid["prospectivity"] >= q_best) &
        (
            ((grid["local_bonus"] >= q_local) & (grid["tect_magm_intersection"] >= q_tmagm)) |
            ((grid["coincidence_score"] >= q_coinc) & (grid["prox_magm"] >= q_magm))
        )
    )

    linear_gold = (
        support_zone &
        (grid["prospectivity"] >= float(grid["prospectivity"].quantile(0.94))) &
        (grid["prox_magm"] >= q_magm) &
        (grid["tect_magm_intersection"] >= q_tmagm) &
        (grid["coincidence_score"] >= q_coinc)
    )

    edge_linear_gold = linear_gold & (grid["dist_to_boundary"] <= CELL_SIZE * 0.75)

    grid["gold_seed"] = (core_gold | edge_linear_gold).astype(int)
    grid["gold_zone"] = keep_large_components(grid, "gold_seed", shape, min_cells=4).astype(int)
    return grid

def plot_prox(grid, mask, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    grid.plot(column="prox_magm", ax=ax, cmap="RdYlBu_r", linewidth=0, legend=True)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    set_mask_extent(ax, mask)
    ax.set_title("prox_magm")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_final(grid, mask, points, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr_r.N)
    grid.plot(column="display_class", ax=ax, cmap="bwr_r", norm=norm, linewidth=0, legend=True)

    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=ax, color="#f2d200", linewidth=0)

    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)

    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=ax, color="yellow", markersize=8, edgecolor="black", linewidth=0.25)

    ax.legend(handles=[Patch(facecolor="#f2d200", edgecolor="black", label="Gold zones")], loc="lower right", frameon=True)
    set_mask_extent(ax, mask)
    ax.set_title("Итоговый прогноз")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_compare(grid, mask, points, out_png):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    grid.plot(column="prox_magm", ax=axes[0], cmap="RdYlBu_r", linewidth=0)
    mask.boundary.plot(ax=axes[0], color="black", linewidth=0.5)
    set_mask_extent(axes[0], mask)
    axes[0].set_title("prox_magm")
    axes[0].set_axis_off()

    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr_r.N)
    grid.plot(column="display_class", ax=axes[1], cmap="bwr_r", norm=norm, linewidth=0, legend=True)

    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=axes[1], color="#f2d200", linewidth=0)

    mask.boundary.plot(ax=axes[1], color="black", linewidth=0.5)

    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=axes[1], color="yellow", markersize=8, edgecolor="black", linewidth=0.25)

    axes[1].legend(handles=[Patch(facecolor="#f2d200", edgecolor="black", label="Gold zones")], loc="lower right", frameon=True)
    set_mask_extent(axes[1], mask)
    axes[1].set_title("Итоговый прогноз")
    axes[1].set_axis_off()

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

# =========================
# LOAD DATA
# =========================
aliases = prepare_ascii_aliases(SHP_DIR, SAFE_ALIAS_DIR)

mask = load_layer(aliases["svita_new"])
facies = to_crs_safe(load_layer(aliases["fasii"]), mask.crs)
tect1 = to_crs_safe(load_layer(aliases["glub_raz_nw"]), mask.crs)
tect2 = to_crs_safe(load_layer(aliases["glub_r_nw"]), mask.crs)
paleo = to_crs_safe(load_layer(aliases["gr_dol_vp_poly"]), mask.crs)
struct = to_crs_safe(load_layer(aliases["kory"]), mask.crs)
magm = to_crs_safe(load_layer(aliases["dayki_buf"]), mask.crs)
points = collect_points(mask.crs, aliases)

# =========================
# GRID + FEATURES
# =========================
grid, mask_union, grid_shape = build_grid(mask, CELL_SIZE)

for src, name in [
    (facies, "dist_facies"),
    (paleo, "dist_paleo"),
    (struct, "dist_struct"),
    (magm, "dist_magm"),
    (tect1, "dist_tect1"),
    (tect2, "dist_tect2"),
]:
    grid = add_distance_feature(grid, src, name)

grid["prox_facies"] = distance_to_proximity(grid["dist_facies"], "cbrt", Q_FACIES)
grid["prox_paleo"] = distance_to_proximity(grid["dist_paleo"], "cbrt", Q_PALEO)
grid["prox_struct"] = distance_to_proximity(grid["dist_struct"], "sqrt", Q_STRUCT)
grid["prox_magm"] = distance_to_proximity(grid["dist_magm"], "sqrt", Q_MAGM)
grid["prox_tect1"] = distance_to_proximity(grid["dist_tect1"], "cbrt", Q_TECT1)
grid["prox_tect2"] = distance_to_proximity(grid["dist_tect2"], "cbrt", Q_TECT2)

grid["tect_combo"] = 0.5 * (grid["prox_tect1"] + grid["prox_tect2"])
grid["tect_intersection"] = grid["prox_tect1"] * grid["prox_tect2"]
grid["tect_magm_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_magm"])
grid["tect_struct_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_struct"])
grid["paleo_struct_intersection"] = np.sqrt(grid["prox_paleo"] * grid["prox_struct"])

combo_core = (
    np.clip(grid["tect_combo"], 0, 1)
    * np.clip(0.55 * grid["prox_magm"] + 0.45 * grid["prox_struct"], 0, 1)
    * np.clip(0.60 * grid["prox_paleo"] + 0.40 * grid["prox_facies"], 0, 1)
)
grid["coincidence_score"] = robust_normalize_01(np.sqrt(np.clip(combo_core, 0, 1)), 0.02, 0.98)

tect_support = 0.40 * grid["prox_magm"] + 0.35 * grid["prox_struct"] + 0.25 * grid["prox_paleo"]
grid["tect_only_penalty"] = robust_normalize_01(np.clip(grid["tect_combo"] - tect_support, 0, 1), 0.02, 0.98)

# =========================
# GEO SCORE
# =========================
grid["geo_score_raw"] = (
    0.14 * grid["prox_tect1"] +
    0.14 * grid["prox_tect2"] +
    0.14 * grid["prox_paleo"] +
    0.11 * grid["prox_struct"] +
    0.08 * grid["prox_facies"] +
    0.08 * grid["prox_magm"] +
    0.08 * grid["tect_intersection"] +
    0.08 * grid["tect_magm_intersection"] +
    0.05 * grid["tect_struct_intersection"] +
    0.04 * grid["paleo_struct_intersection"] +
    0.10 * grid["coincidence_score"] -
    0.08 * grid["tect_only_penalty"]
)
grid["geo_score"] = robust_normalize_01(grid["geo_score_raw"], 0.02, 0.98)
grid["geo_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "geo_score", grid_shape, passes=2), 0.02, 0.98)

# =========================
# RANDOM FOREST
# =========================
grid["target"] = 0
grid["rf_score"] = grid["geo_score_sm"]
feature_importance = {}
rf_test_proxy = None
use_supervised = False

feature_cols = [
    "prox_facies", "prox_paleo", "prox_struct", "prox_magm",
    "prox_tect1", "prox_tect2", "tect_combo", "tect_intersection",
    "tect_magm_intersection", "tect_struct_intersection",
    "paleo_struct_intersection", "coincidence_score", "tect_only_penalty",
    "geo_score_sm"
]

if USE_SUPERVISED and points is not None and len(points) > 0:
    try:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", predicate="within")
    except Exception:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", op="within")

    positive_cells = joined["cell_id"].dropna().astype(int).unique().tolist()
    grid.loc[grid["cell_id"].isin(positive_cells), "target"] = 1

    pos = int(grid["target"].sum())
    neg = int((grid["target"] == 0).sum())

    if pos >= 20 and neg > pos:
        X = grid[feature_cols].fillna(0).to_numpy()
        y = grid["target"].to_numpy()

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
        )

        rf_eval = RandomForestClassifier(
            n_estimators=RF_N_ESTIMATORS,
            max_depth=RF_MAX_DEPTH,
            min_samples_leaf=RF_MIN_SAMPLES_LEAF,
            min_samples_split=RF_MIN_SAMPLES_SPLIT,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        rf_eval.fit(X_train, y_train)
        test_prob = rf_eval.predict_proba(X_test)[:, 1]
        pos_mean = float(np.mean(test_prob[y_test == 1])) if np.any(y_test == 1) else np.nan
        neg_mean = float(np.mean(test_prob[y_test == 0])) if np.any(y_test == 0) else np.nan
        rf_test_proxy = pos_mean - neg_mean

        rf = RandomForestClassifier(
            n_estimators=RF_N_ESTIMATORS,
            max_depth=RF_MAX_DEPTH,
            min_samples_leaf=RF_MIN_SAMPLES_LEAF,
            min_samples_split=RF_MIN_SAMPLES_SPLIT,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        rf.fit(X, y)
        grid["rf_score"] = robust_normalize_01(rf.predict_proba(X)[:, 1], 0.02, 0.98)
        feature_importance = dict(zip(feature_cols, rf.feature_importances_.tolist()))
        use_supervised = True

grid["rf_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "rf_score", grid_shape, passes=2), 0.02, 0.98)

# =========================
# FINAL SURFACE
# =========================
grid["local_bonus"] = robust_normalize_01(
    0.38 * grid["tect_intersection"] +
    0.37 * grid["tect_magm_intersection"] +
    0.25 * grid["tect_struct_intersection"],
    0.02, 0.98,
)

mask_boundary = mask_union.boundary
grid["dist_to_boundary"] = np.array([geom.distance(mask_boundary) for geom in grid.geometry])
edge_near = np.exp(-grid["dist_to_boundary"].to_numpy() / (CELL_SIZE * 2.2))
grid["edge_false_penalty"] = robust_normalize_01(edge_near * np.clip(grid["tect_only_penalty"], 0, 1), 0.02, 0.98)

grid["prospectivity_raw"] = (
    W_GEO * grid["geo_score_sm"] +
    W_RF * grid["rf_score_sm"] +
    W_COINCIDENCE * grid["coincidence_score"] +
    W_LOCAL * grid["local_bonus"]
)
grid["prospectivity_raw"] = grid["prospectivity_raw"] - 0.05 * grid["tect_only_penalty"] - 0.04 * grid["edge_false_penalty"]
grid["prospectivity_raw_sm"] = smooth_on_regular_grid(grid, "prospectivity_raw", grid_shape, passes=2)
grid["prospectivity"] = robust_normalize_01(grid["prospectivity_raw_sm"], 0.02, 0.98)
grid["prognoz"] = 1.0 - grid["prospectivity"]

grid = make_display_classes(grid)
grid = mark_gold_zones(grid, grid_shape, mask_union)

# =========================
# SAVE
# =========================
if OUT_GPKG.exists():
    OUT_GPKG.unlink()

grid.to_file(OUT_GPKG, layer="forecast_grid", driver="GPKG")
if points is not None and len(points) > 0:
    points.to_file(OUT_GPKG, layer="evidence_points", driver="GPKG")

grid.drop(columns="geometry").to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

plot_prox(grid, mask, OUT_PROX)
plot_final(grid, mask, points, OUT_PNG)
plot_compare(grid, mask, points, OUT_COMPARE)

metrics = {
    "base_dir": str(BASE_DIR),
    "grid_cells": int(len(grid)),
    "cell_size": CELL_SIZE,
    "use_supervised_requested": bool(USE_SUPERVISED),
    "use_supervised_applied": bool(use_supervised),
    "som_used": False,
    "kmeans_used": False,
    "positive_cells": int(grid["target"].sum()),
    "prospectivity_min": float(np.nanmin(grid["prospectivity"])),
    "prospectivity_p05": float(np.nanquantile(grid["prospectivity"], 0.05)),
    "prospectivity_p50": float(np.nanquantile(grid["prospectivity"], 0.50)),
    "prospectivity_p95": float(np.nanquantile(grid["prospectivity"], 0.95)),
    "prospectivity_max": float(np.nanmax(grid["prospectivity"])),
    "gold_zone_count": int(grid["gold_zone"].sum()),
    "gold_zone_share": float(grid["gold_zone"].mean()),
    "rf_test_proxy": None if rf_test_proxy is None else float(rf_test_proxy),
    "point_count": int(len(points)) if points is not None else 0,
    "rf_feature_importance": feature_importance,
}
OUT_JSON.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

print("Готово.")
print(f"PNG: {OUT_PNG}")
print(f"COMPARE: {OUT_COMPARE}")
print(f"GPKG: {OUT_GPKG}")
print(f"CSV: {OUT_CSV}")
print(f"JSON: {OUT_JSON}")
print(grid[["prospectivity", "prognoz", "rf_score_sm", "gold_zone"]].describe())


Готово.
PNG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\rf_clean_result\forecast_rf_clean.png
COMPARE: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\rf_clean_result\compare_rf_clean.png
GPKG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\rf_clean_result\forecast_rf_clean.gpkg
CSV: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\rf_clean_result\grid_rf_clean.csv
JSON: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\rf_clean_result\metrics_rf_clean.json
       prospectivity       prognoz   rf_score_sm     gold_zone
count   15684.000000  15684.000000  15684.000000  15684.000000
mean        0.415267      0.584733      0.190389      0.005483
std         0.250990      0.250990      0.237673      0.073848
min         0.000000      0.000000      0.000000      0.000000
25%         0.225935      0.428682      0.029985      0.000000
50%         0.395174      0.604826      0.096525      0.000000
75%         0.571318      0.774065      0.242892      0.000000
max         1.000000      1.000000  